# MobiHeal — Complete paper-aligned reproducibility notebook

**Paper:** *MobiHeal: One-Hop Compromise Recovery for Post-Quantum Vehicular Handover* (Nasir et al.).

This is **one self-contained Jupyter notebook** for GitHub / Google Colab. It embeds the executable Python research prototype, paper-aligned experiment drivers, an atomic application-session + report-persistence extension, the bounded Z3 model, and eight ProVerif models. Run all cells top to bottom. The notebook creates Python and `.pv` files **at runtime**; there are no extra repository files required to launch it.

### What it actually reproduces

| Manuscript evidence | Notebook section | Evidence provenance |
|---|---|---|
| Table VII — chronological same-root recovery | §3 | Live M0→M3 tests, 100 trials in FULL |
| Table VIII — PV1–PV8, Z1–Z9 | §10–11 | Actual checker output only; UNKNOWN / NOT_EXECUTED remain visible |
| Figure 3 — stage mean / p95 | §5, §13 | Native timed handovers |
| Figure 4 — vehicle, RSU and combined mean / p95 | §5, §13 | Native timed handovers |
| Table IX — five restart points | §4 | Process-level restart tests |
| Table X — RSU completion persistence | §9 | Additional **integrated atomic transaction** research variant with AES-GCM key wrapping; timing platform-dependent |
| Figure 5 — loss/delay p95 | §8, §13 | Independent packet-loss Monte Carlo calibrated with *current-run* native stage medians |
| Table XI — component-removal attacks | §12 | Six actual negative variants; two supplemental sensitivity variants |
| Table XII — offline Merkle preparation | §7 | Pool generation + SQLite + ML-DSA signing timings |
| Figure 6 — serialized on-air bytes, B=64–4096 | §6, §13 | Actual M0–M3 wire serialization compared against analytic lengths |
| Admission/retransmission/concurrency, storage, microbenchmarks | §3–8 | Live execution and separate accounting |
| One-hop PCS and multi-challenge exploratory games | §14 | **Specific-adversary empirical diagnostics only**, not cryptographic proofs |

**Scientific reproducibility rules:** manuscript numbers are stored in a dedicated **reference** table and never used as fresh measurements. Hardware, scheduler, RNG, SSD, SQLite, and library versions change timing; identical milliseconds are not guaranteed. `DEV` uses an intentionally insecure simulator for structural smoke tests and must **not** be cited as cryptographic or performance evidence. Cryptographic claims apply only to the assumptions and threat model stated in the paper. ProVerif symbolic checks are not computational ML-KEM proofs; bounded Z3 results assume atomic storage and authenticated guards. We explicitly do **not** claim a deployed certificate-chain validator, physical erasure, B0–B3 network recovery, or a computational proof of multi-challenge unlinkability.

**How to run:** open in Colab; choose `RUN_PROFILE = "FULL"` in §0 for paper-sized experiments. `QUICK` checks integration. Native `pqcrypto==1.0.0` / PyCryptodome are required for publication measurements; packages are installed when online. Set `MOBIHEAL_BACKEND=DEV` only for structure testing without native packages. In FULL, the notebook attempts the pinned ProVerif 2.05 source build if no binary is installed; if installation fails it reports NOT_EXECUTED rather than a false success.

## 0. Configuration and environment — choose `QUICK` or `FULL`

In [ ]:
from __future__ import annotations
import os, sys, subprocess, shutil, json, hashlib, platform, sqlite3, time, importlib.util
from pathlib import Path
from datetime import datetime, timezone

# FULL reproduces manuscript trial COUNTS; not necessarily the manuscript timing VALUES.
RUN_PROFILE = os.environ.get('MOBIHEAL_PROFILE', 'QUICK').upper()  # set 'FULL' for manuscript-sized study
CRYPTO_BACKEND = os.environ.get('MOBIHEAL_BACKEND', 'NATIVE').upper()  # 'NATIVE' or explicit 'DEV'
INSTALL_MISSING = os.environ.get('MOBIHEAL_INSTALL', '1') == '1'
RUN_FORMAL = os.environ.get('MOBIHEAL_SKIP_FORMAL', '0') != '1'
INSTALL_PROVERIF = os.environ.get('MOBIHEAL_INSTALL_PROVERIF', '1' if RUN_PROFILE=='FULL' else '0') == '1'  # FULL attempts pinned source build
assert RUN_PROFILE in ('QUICK', 'FULL')
assert CRYPTO_BACKEND in ('NATIVE', 'DEV')
SEEDS = list(range(2601, 2611)) if RUN_PROFILE == 'FULL' else [2601, 2602]
CONFIG = {
 'profile':RUN_PROFILE,'backend_requested':CRYPTO_BACKEND, 'native_required_for_paper':True,
 'seeds': SEEDS,'handover_trials_per_seed': 100 if RUN_PROFILE=='FULL' else 3,
 'warmups_per_seed':10 if RUN_PROFILE=='FULL' else 1,
 'handover_pool_B':1024 if RUN_PROFILE=='FULL' else 16,
 'same_root_trials_per_seed':10 if RUN_PROFILE=='FULL' else 2,
 'pool_prep_sizes':[64,256,1024] if RUN_PROFILE=='FULL' else [8,16],
 'primitive_reps_per_seed':50 if RUN_PROFILE=='FULL' else 2,
 'invalid_requests_per_case_seed':1000 if RUN_PROFILE=='FULL' else 5,
 'concurrent_sessions':128 if RUN_PROFILE=='FULL' else 4,
 'concurrent_workers':[1,8,32,64,128] if RUN_PROFILE=='FULL' else [1,4],
 'network_trials_per_cell_seed':100 if RUN_PROFILE=='FULL' else 2,
 'persistence_sessions_per_seed':10 if RUN_PROFILE=='FULL' else 2,
 'ablation_trials_per_seed':1,
 'signature_reps_per_seed':10 if RUN_PROFILE=='FULL' else 1,
 'pcs_trials':250 if RUN_PROFILE=='FULL' else 12,
 'proverif_timeout_s':600 if RUN_PROFILE=='FULL' else 90,
 'z3_timeout_s':900 if RUN_PROFILE=='FULL' else 180,
}
ROOT = Path.cwd() / 'MobiHeal_Complete_Results'
FIGS, LOGS, MODELS = ROOT/'figures', ROOT/'formal_logs', ROOT/'models'
for d in (ROOT,FIGS,LOGS,MODELS): d.mkdir(parents=True,exist_ok=True)
STARTED = datetime.now(timezone.utc).isoformat()
print('MobiHeal',RUN_PROFILE,CRYPTO_BACKEND,'seeds',SEEDS,'results',ROOT)

### 0.1 Install required libraries (native mode fails explicitly if cryptography is unavailable)

In [ ]:
dependencies = [
 ('pqcrypto','pqcrypto==1.0.0'), ('Crypto','pycryptodome>=3.20'),
 ('cryptography','cryptography>=42'),('numpy','numpy'),('pandas','pandas'),
 ('matplotlib','matplotlib'),('z3','z3-solver>=4.12')]
missing = [package for module,package in dependencies if importlib.util.find_spec(module) is None]
if missing and INSTALL_MISSING:
    result=subprocess.run([sys.executable,'-m','pip','install','-q',*missing],capture_output=True,text=True)
    (LOGS/'pip_install.txt').write_text(result.stdout+'\n'+result.stderr)
    if result.returncode != 0:
        print('Some dependencies could not be installed (offline/permissions). See formal_logs/pip_install.txt')
        print('To smoke-test offline set MOBIHEAL_BACKEND=DEV and MOBIHEAL_INSTALL=0 before running the notebook.')
if CRYPTO_BACKEND=='NATIVE':
    for lib in ('pqcrypto','Crypto'):
        if importlib.util.find_spec(lib) is None:
            raise RuntimeError('Missing '+lib+': NATIVE is required for paper evidence. Install it or explicitly select DEV for non-paper smoke tests.')
import numpy as np, pandas as pd
from IPython.display import display
import importlib.metadata
PROVENANCE={'started_utc':STARTED,'python':sys.version,'platform':platform.platform(),
  'cpu':platform.processor(),'sqlite_version':sqlite3.sqlite_version,
  'sqlite_journal':'WAL','sqlite_synchronous':'FULL','config':CONFIG,
  'packages':{pkg:(importlib.metadata.version(pkg) if importlib.util.find_spec(module) else 'MISSING')
             for module,pkg in [('pqcrypto','pqcrypto'),('Crypto','pycryptodome'),('cryptography','cryptography'),('z3','z3-solver'),('numpy','numpy'),('pandas','pandas')]}}
(ROOT/'environment.json').write_text(json.dumps(PROVENANCE,indent=2,default=str))
print('Dependencies/provenance recorded; native backend:',CRYPTO_BACKEND)

## 1. Complete four-message MobiHeal protocol core (paper Equations 2–6, Tables V–VI)
The following embedded module implements authenticated M0, request-bound β₁ in M1, α₂ before ML-KEM decapsulation, M3 verification, per-target derived credentials, a signed Merkle pool, exact duplicate caches, and durable monotone key-state transitions. **Certificate-chain and TA revocation are represented by local descriptor checks, not by full PKI validation.** The published full labels specify `dβ = MOBIHEAL-OFFER`; the imported prototype used `MOBIHEAL-OFFER-AUTH`, which is corrected here for the manuscript-aligned run. This changes the protocol-domain bytes without changing expected wire length or key-index logic.

In [ ]:
%%writefile mobiheal_core.py
"""MobiHeal current fast-path research harness.

DO NOT deploy: certificate-chain validation, filesystem encryption, process isolation,
physical erasure, and controller B0-B3 recovery are beyond this artifact.
Native paper-mode requires pqcrypto ML-KEM-768 / ML-DSA-65 and PyCryptodome KMAC256.
"""
from __future__ import annotations
import base64, hashlib, hmac, json, math, os, secrets, sqlite3, tempfile, threading, time
from dataclasses import dataclass, replace
from pathlib import Path
from typing import Optional

PK_N=1184; SK_N=2400; CT_N=1088; SS_N=32; SIGPK_N=1952; SIG_N=3309
LABEL={k:b'MOBIHEAL-'+v.encode() for k,v in {
 'AUTH':'AUTH','PID':'PID','ADMIT_CTX':'ADMIT-CTX','ADMIT':'ADMIT',
 'TH2':'TH2','PREAUTH':'PREAUTH','MIXA':'AUTH-MIX','MIXZ':'KEM-MIX',
 'CONFV':'CONF-V','FINV':'FIN-V','TH':'TH','CONFR':'CONF-R',
 'APP':'APP','FINR':'FIN-R','SID':'SID','KEM':'KEM',
 'NODE':'MERKLE-NODE','OFFER_CTX':'OFFER-CTX',
 'OFFER_AUTH':'OFFER','BUSY':'BUSY'}.items()}
# The three additional offer/BUSY domain-label bytes are explicit, nonstandard
# implementation choices; paper must publish exact values before interoperability.

def H(x):return hashlib.sha3_384(x).digest()
def u64(x):return int(x).to_bytes(8,'big')
def canon(*items):
 out=b''
 for x in items:
  if isinstance(x,(tuple,list)):
   b=canon(*x); t=b'L'
  elif isinstance(x,(bytes,bytearray)):b=bytes(x);t=b'B'
  elif isinstance(x,int):b=u64(x);t=b'I'
  elif isinstance(x,str):b=x.encode();t=b'S'
  elif hasattr(x,'canonical'):b=x.canonical();t=b'M'
  else:raise TypeError(type(x))
  out+=t+len(b).to_bytes(4,'big')+b
 return out

def xor(x,y):return bytes(a^b for a,b in zip(x,y))

class Crypto:
 def __init__(self,native=True):
  self.native=bool(native)
  if native:
   from pqcrypto.kem import ml_kem_768
   from pqcrypto.sign import ml_dsa_65
   from Crypto.Hash import KMAC256
   self.kem,self.sig,self.KMAC=ml_kem_768,ml_dsa_65,KMAC256
   self.label='NATIVE_ML-KEM-768_ML-DSA-65_KMAC256'
  else:
   self.label='DEV_EMULATOR_NOT_FOR_PAPER'
 def kmac(self,key,data,n,label):
  if self.native:return self.KMAC.new(key=key, data=data, mac_len=n, custom=label).digest()
  return hashlib.shake_256(b'UNSAFE_EMULATION'+len(label).to_bytes(2,'big')+label+key+data).digest(n)
 def kg(self):
  if self.native:
   p,s=self.kem.keygen();return bytes(p),bytes(s)
  s=secrets.token_bytes(SK_N);return hashlib.shake_256(b'PK'+s).digest(PK_N),s
 def encaps(self,pk):
  if self.native:
   c,z=self.kem.encaps(pk);return bytes(c),bytes(z)
  r=secrets.token_bytes(32); ct=r+hashlib.shake_256(b'CT'+pk+r).digest(CT_N-32)
  return ct,hashlib.sha3_256(b'Z'+pk+ct).digest()
 def decaps(self,sk,ct):
  if self.native:return bytes(self.kem.decaps(sk,ct))
  pk=hashlib.shake_256(b'PK'+sk).digest(PK_N)
  return hashlib.sha3_256(b'Z'+pk+ct).digest()
 def sigkg(self):
  if self.native:return tuple(map(bytes,self.sig.keygen()))
  s=secrets.token_bytes(4032);return hashlib.shake_256(b'SIGPK'+s).digest(SIGPK_N),s
 def sign(self,sk,data):
  if self.native:return bytes(self.sig.sign(sk,data))
  pk=hashlib.shake_256(b'SIGPK'+sk).digest(SIGPK_N)
  return hashlib.shake_256(b'SIG'+pk+data).digest(SIG_N)
 def verify(self,pk,data,sig):
  if self.native:
   try:return self.sig.verify(pk,data,sig) is None
   except Exception:return False
  return hmac.compare_digest(sig,hashlib.shake_256(b'SIG'+pk+data).digest(SIG_N))
 def self_test(self):
  pk,sk=self.kg();ct,z=self.encaps(pk);assert self.decaps(sk,ct)==z
  sigpk,sigsk=self.sigkg();t=self.sign(sigsk,b'TEST');assert self.verify(sigpk,b'TEST',t)
  assert (len(pk),len(sk),len(ct),len(z),len(sigpk),len(t))==(1184,2400,1088,32,1952,3309)


def derive(root,rid,w,c,a):
 C=canon(rid,w,c,a)
 a_secret=CR.kmac(root,C,32,LABEL['AUTH'])
 pid=CR.kmac(root,C,32,LABEL['PID'])[:16]
 return pid,a_secret

def leaf(rid,e,j,pk):return H(LABEL['KEM']+canon(rid,e,j,pk))
def tree(leaves):
 levels=[list(leaves)]
 while len(levels[-1])>1:
  last=levels[-1]
  padded=last+([last[-1]] if len(last)%2 else [])
  levels.append([H(LABEL['NODE']+canon(padded[i],padded[i+1])) for i in range(0,len(padded),2)])
 return levels

def proof(levels,pos):
 p=[]
 for layer in levels[:-1]:
  even=layer+([layer[-1]] if len(layer)%2 else [])
  p.append(even[pos^1]);pos//=2
 return tuple(p)
def verify_proof(rid,e,j,pk,path,root):
 v=leaf(rid,e,j,pk);i=j-1
 for sibling in path:
  v=H(LABEL['NODE']+canon(v,sibling)) if i%2==0 else H(LABEL['NODE']+canon(sibling,v));i//=2
 return hmac.compare_digest(v,root)

@dataclass(frozen=True)
class M0:
 pid:bytes;w:int;n0:bytes;a0:bytes
 def canonical(self):return canon(self.pid,self.w,self.n0,self.a0)
 def wire(self):return self.pid+u64(self.w)+self.n0+self.a0
@dataclass(frozen=True)
class M1:
 rid:bytes;e:int;w:int;j:int;nr:bytes;pk:bytes;path:tuple;beta:bytes
 def canonical(self):return canon(self.rid,self.e,self.w,self.j,self.nr,self.pk,self.path,self.beta)
 def wire(self):return self.rid+u64(self.e)+u64(self.w)+u64(self.j)+self.nr+self.pk+b''.join(self.path)+self.beta
@dataclass(frozen=True)
class M2:
 pid:bytes;nv:bytes;ct:bytes;a2:bytes;tv:bytes
 def canonical(self):return canon(self.pid,self.nv,self.ct,self.a2,self.tv)
 def wire(self):return self.pid+self.nv+self.ct+self.a2+self.tv
@dataclass(frozen=True)
class M3:
 sid:bytes;tr:bytes
 def canonical(self):return canon(self.sid,self.tr)
 def wire(self):return self.sid+self.tr

def parse_m0(b):
 assert len(b)==72
 return M0(b[:16],int.from_bytes(b[16:24],'big'),b[24:40],b[40:72])
def parse_m1(b,B):
 depth=math.ceil(math.log2(B));assert len(b)==8+24+16+PK_N+48*depth+16
 o=0;rid=b[o:o+8];o+=8;e=int.from_bytes(b[o:o+8],'big');o+=8
 w=int.from_bytes(b[o:o+8],'big');o+=8;j=int.from_bytes(b[o:o+8],'big');o+=8
 nr=b[o:o+16];o+=16;pk=b[o:o+PK_N];o+=PK_N
 path=tuple(b[o+i*48:o+(i+1)*48] for i in range(depth));o+=depth*48
 return M1(rid,e,w,j,nr,pk,path,b[o:o+16])
def parse_m2(b):
 assert len(b)==1184
 return M2(b[:16],b[16:32],b[32:1120],b[1120:1152],b[1152:1184])
def parse_m3(b):
 assert len(b)==80
 return M3(b[:48],b[48:80])

def alpha0(rid,m0,A):
 return CR.kmac(A,H(LABEL['ADMIT_CTX']+canon(rid,m0.w,m0.pid,m0.n0)),32,LABEL['ADMIT'])
def beta1(A,m0,e,j,nr,pk):
 h1=H(LABEL['OFFER_CTX']+canon(H(canon(m0)),e,j,nr,pk))
 return CR.kmac(A,h1,16,LABEL['OFFER_AUTH'])
def prehash(m0,m1,pid,nv,ct):return H(LABEL['TH2']+canon(m0,m1,pid,nv,ct))
def derivemix(A,z,h2):
 return xor(CR.kmac(A,h2,32,LABEL['MIXA']),CR.kmac(z,h2,32,LABEL['MIXZ']))
def h_full(m0,m1,m2):return H(LABEL['TH']+canon(m0,m1,m2))
def sid_for(m1,pid,h):return H(LABEL['SID']+canon(m1.rid,m1.e,m1.w,m1.j,pid,h))

# Explicit process-global cryptographic backend is set by the notebook.
CR=None

def JENC(obj):
 def convert(x):
  if isinstance(x,bytes):return {'__b64':base64.b64encode(x).decode()}
  if isinstance(x,dict):return {k:convert(v) for k,v in x.items()}
  if isinstance(x,(list,tuple)):return [convert(v) for v in x]
  return x
 return json.dumps(convert(obj),sort_keys=True)
def JDEC(s):
 def convert(x):
  if isinstance(x,dict) and '__b64' in x:return base64.b64decode(x['__b64'])
  if isinstance(x,dict):return {k:convert(v) for k,v in x.items()}
  if isinstance(x,list):return [convert(y) for y in x]
  return x
 return convert(json.loads(s))

class Vehicle:
 """Durable vehicle state. Assumes protected local storage, NOT at-rest encryption."""
 def __init__(self,db,root=None):
  self.db=str(db);self.conn=sqlite3.connect(self.db,check_same_thread=False)
  self.conn.execute('PRAGMA journal_mode=WAL');self.conn.execute('PRAGMA synchronous=FULL')
  self.conn.execute('CREATE TABLE IF NOT EXISTS vehicle(id INTEGER PRIMARY KEY CHECK(id=1),root BLOB,c INTEGER,pending TEXT,accepted_sid BLOB,accepted_key BLOB,accepted_m3 BLOB)')
  self.conn.execute('CREATE TABLE IF NOT EXISTS verified_epoch(rid BLOB PRIMARY KEY,epoch INTEGER NOT NULL)')
  if not self.conn.execute('SELECT 1 FROM vehicle').fetchone():
   self.conn.execute('INSERT INTO vehicle VALUES(1,?,0,NULL,NULL,NULL,NULL)',(root or secrets.token_bytes(32),));self.conn.commit()
  self.known_epochs={rid:int(e) for rid,e in self.conn.execute('SELECT rid,epoch FROM verified_epoch')}
 def row(self):return self.conn.execute('SELECT root,c,pending,accepted_sid,accepted_key,accepted_m3 FROM vehicle WHERE id=1').fetchone()
 @property
 def counter(self):return self.row()[1]
 @property
 def root(self):return self.row()[0]
 def verify_descriptor(self,rsu,expired=False,revoked=False):
  D=rsu.descriptor();rid,e,B,root,expiry,sig,spk=D
  if not rsu.CR.verify(spk,canon(rid,e,B,root,expiry),sig):raise ValueError('descriptor signature')
  if expired or revoked or expiry<=time.time():raise ValueError('descriptor expired or revoked')
  if self.known_epochs.get(rid,-1)>e:raise ValueError('epoch rollback')
  self.known_epochs[rid]=e
  with self.conn:self.conn.execute('INSERT OR REPLACE INTO verified_epoch(rid,epoch) VALUES(?,?)',(rid,int(e)))
  return D
 def start(self,rid,w,attempt):
  if self.row()[2] is not None:raise ValueError('pending completion: freeze counter')
  pid,A=derive(self.root,rid,int(w),self.counter,int(attempt))
  m0=M0(pid,int(w),secrets.token_bytes(16),b'')
  m0=replace(m0,a0=alpha0(rid,m0,A))
  return {'A':A,'pid':pid,'m0':m0,'rid':rid,'attempt':attempt,'counter':self.counter}
 def accept_busy(self,ctx,msg):
  if not msg or msg[0]!='BUSY':return False
  epoch,tag=msg[1],msg[2]
  return hmac.compare_digest(tag,CR.kmac(ctx['A'],canon(ctx['m0'],b'BUSY',epoch),16,LABEL['BUSY']))
 def send_m2(self,ctx,m1,desc):
  rid,e,B,root,expiry,signature,spk=desc
  if expiry<=time.time():raise ValueError('descriptor expired at encapsulation')
  if m1.rid!=rid or m1.e!=e or m1.w!=ctx['m0'].w:raise ValueError('offer context')
  # Essential: current-offer authenticator before Merkle/KEM work.
  if not hmac.compare_digest(m1.beta,beta1(ctx['A'],ctx['m0'],e,m1.j,m1.nr,m1.pk)):
   raise ValueError('invalid beta1')
  if not(1<=m1.j<=B) or not verify_proof(rid,e,m1.j,m1.pk,m1.path,root):raise ValueError('Merkle proof invalid')
  ct,z=CR.encaps(m1.pk);nv=secrets.token_bytes(16)
  h2=prehash(ctx['m0'],m1,ctx['pid'],nv,ct)
  a2=CR.kmac(ctx['A'],h2,32,LABEL['PREAUTH']);K=derivemix(ctx['A'],z,h2)
  cv=CR.kmac(K,h2,32,LABEL['CONFV']);tv=CR.kmac(cv,h2+a2,32,LABEL['FINV'])
  m2=M2(ctx['pid'],nv,ct,a2,tv)
  h=h_full(ctx['m0'],m1,m2); sid=sid_for(m1,ctx['pid'],h)
  kr=CR.kmac(K,h,32,LABEL['CONFR']);ka=CR.kmac(K,h,32,LABEL['APP'])
  # Persist exact pending M2 and acceptance state before emitting M2.
  data=JENC({'m2':m2.wire(),'sid':sid,'h':h,'kr':kr,'ka':ka})
  with self.conn:self.conn.execute('UPDATE vehicle SET pending=? WHERE id=1',(data,))
  return m2
 def complete(self,m3):
  root,c,pending,as_id,a_key,a_m3=self.row()
  if pending is None:
   if as_id==m3.sid and a_m3==m3.wire():return a_key,False
   raise ValueError('no matching pending session')
  d=JDEC(pending);exp=CR.kmac(d['kr'],d['h']+d['sid'],32,LABEL['FINR'])
  if not hmac.compare_digest(d['sid'],m3.sid) or not hmac.compare_digest(exp,m3.tr):
   raise ValueError('invalid M3')
  with self.conn:
   self.conn.execute('UPDATE vehicle SET c=?,pending=NULL,accepted_sid=?,accepted_key=?,accepted_m3=? WHERE id=1',
                     (c+1,d['sid'],d['ka'],m3.wire()))
  return d['ka'],True
 def retransmit_m2(self):
  p=self.row()[2]
  if p is None:raise ValueError('no pending M2')
  return parse_m2(JDEC(p)['m2'])
 def close(self):self.conn.close()

class RSU:
 def __init__(self,db,rid,B=16,e=1,ttl=3600,initialize=True,sigkeys=None,offer_ttl=30.0):
  self.db=str(db);self.rid=rid;self.B=B;self.e=e;self.ttl=ttl;self.offer_ttl=offer_ttl;self.lock=threading.RLock();self.CR=CR
  self.conn=sqlite3.connect(self.db,check_same_thread=False,timeout=30,isolation_level=None)
  self.conn.execute('PRAGMA journal_mode=WAL');self.conn.execute('PRAGMA synchronous=FULL')
  self.conn.executescript('''CREATE TABLE IF NOT EXISTS kem(j INTEGER PRIMARY KEY,pk BLOB,sk BLOB,state TEXT);
  CREATE TABLE IF NOT EXISTS grp(g BLOB PRIMARY KEY,state TEXT,next_attempt INTEGER,q INTEGER,active_pid BLOB);
  CREATE TABLE IF NOT EXISTS rec(pid BLOB PRIMARY KEY,g BLOB,w INTEGER,A BLOB,attempt INTEGER,
  state TEXT,j INTEGER,m0 BLOB,m1 BLOB,m2h BLOB,m3 BLOB);
  CREATE TABLE IF NOT EXISTS meta(k TEXT PRIMARY KEY,v BLOB);
  CREATE TABLE IF NOT EXISTS offer_lease(pid BLOB PRIMARY KEY, expires_at REAL NOT NULL);''')
  self.sigpk,self.sigsk=sigkeys or CR.sigkg()
  if initialize:
   self._initialize_pool()
  else:
   assert self.conn.execute('SELECT COUNT(*) FROM kem').fetchone()[0]==B
  self._levels=None
  self.counts={'alloc':0,'encaps':0,'decap':0,'counter':0,'busy':0}
  self.last={'commit_offer_ms':0.,'commit_completion_ms':0.,'decap_ms':0.,'total_ms':0.}
 def tx(self,func):
  with self.lock:
   self.conn.execute('BEGIN IMMEDIATE')
   try:
    r=func();t_commit=time.perf_counter_ns();self.conn.execute('COMMIT')
    self.last_db_commit_ms=(time.perf_counter_ns()-t_commit)/1e6
    return r
   except BaseException:self.conn.execute('ROLLBACK');raise
 def _initialize_pool(self):
  def f():
   for j in range(1,self.B+1):
    p,s=CR.kg();self.conn.execute('INSERT INTO kem VALUES(?,?,?,?)',(j,p,s,'free'))
  self.tx(f)
 def descriptor(self):
  if self._levels is None:
   leaves=[leaf(self.rid,self.e,j,pk) for j,pk in self.conn.execute('SELECT j,pk FROM kem ORDER BY j')]
   self._levels=tree(leaves)
  root=self._levels[-1][0];expiry=int(time.time())+self.ttl
  return (self.rid,self.e,self.B,root,expiry,CR.sign(self.sigsk,canon(self.rid,self.e,self.B,root,expiry)),self.sigpk)
 def provision(self,vehicle,w,q=2):
  # MSC emulation: only derived (PID,A), opaque g and w reach RSU.
  g=secrets.token_bytes(16);root=vehicle.root;c=vehicle.counter
  def f():
   self.conn.execute('INSERT INTO grp VALUES(?,?,?,?,?)',(g,'unused',0,q,None))
   for a in range(q):
    pid,A=derive(root,self.rid,int(w),c,a)
    self.conn.execute('INSERT INTO rec(pid,g,w,A,attempt,state,j,m0,m1,m2h,m3) VALUES(?,?,?,?,?,?,?,?,?,?,?)',
                      (pid,g,int(w),A,a,'unused',None,None,None,None,None))
  self.tx(f)
  return g
 def _group_rec(self,pid):
  r=self.conn.execute('SELECT g,w,A,attempt,state,j,m0,m1,m2h,m3 FROM rec WHERE pid=?',(pid,)).fetchone()
  if not r:return None,None
  group=self.conn.execute('SELECT state,next_attempt,q,active_pid FROM grp WHERE g=?',(r[0],)).fetchone()
  return r,group
 def recv_m0(self,m0):
  t0=time.perf_counter_ns();metrics={'commit_offer_ms':0.,'total_ms':0.}
  def f():
   rec,grp=self._group_rec(m0.pid)
   if not rec:return ('REJECT_UNKNOWN',None)
   g,w,A,a,state,j,cached_m0,cached_m1,m2h,m3=rec
   if int(w)!=m0.w:return('REJECT_WINDOW',None)
   if state=='offered':
    lease=self.conn.execute('SELECT expires_at FROM offer_lease WHERE pid=?',(m0.pid,)).fetchone()
    if lease and time.time()>lease[0]:return('REJECT_EXPIRED_OFFER',None)
    return ('DUPLICATE_M0',parse_m1(cached_m1,self.B)) if cached_m0==m0.wire() else ('REJECT_CONFLICT',None)
   if state!='unused' or grp[0]!='unused' or grp[1]!=a:return('REJECT_GROUP',None)
   if not hmac.compare_digest(alpha0(self.rid,m0,A),m0.a0):return('REJECT_ALPHA0',None)
   k=self.conn.execute("SELECT j,pk FROM kem WHERE state='free' ORDER BY j LIMIT 1").fetchone()
   if not k:
    self.counts['busy']+=1
    return ('BUSY',(self.e,CR.kmac(A,canon(m0,b'BUSY',self.e),16,LABEL['BUSY'])))
   j,pk=k;nr=secrets.token_bytes(16)
   if self._levels is None:self.descriptor()
   m1=M1(self.rid,self.e,m0.w,j,nr,pk,proof(self._levels,j-1),beta1(A,m0,self.e,j,nr,pk))
   self.conn.execute("UPDATE kem SET state='reserved' WHERE j=? AND state='free'",(j,))
   self.conn.execute("UPDATE rec SET state='offered',j=?,m0=?,m1=? WHERE pid=?",(j,m0.wire(),m1.wire(),m0.pid))
   self.conn.execute("UPDATE grp SET state='offered',active_pid=?,next_attempt=next_attempt+1 WHERE g=?",(m0.pid,g))
   self.conn.execute('INSERT OR REPLACE INTO offer_lease(pid,expires_at) VALUES(?,?)',(m0.pid,time.time()+self.offer_ttl))
   self.counts['alloc']+=1
   return ('OFFERED',m1)
  status,obj=self.tx(f)
  if status=='REJECT_EXPIRED_OFFER':self.abandon(m0.pid)
  self.last={**self.last,'commit_offer_ms':self.last_db_commit_ms if status=='OFFERED' else 0.,'total_ms':(time.perf_counter_ns()-t0)/1e6}
  return status,obj
 def recv_m2(self,m2,fault=None):
  t0=time.perf_counter_ns()
  with self.lock:
   rec,grp=self._group_rec(m2.pid)
   if rec is None:return 'REJECT_NO_OFFER',None,None
   g,w,A,a,state,j,m0raw,m1raw,m2h,m3raw=rec
   if state=='completed':
    if m2h==H(m2.wire()):return 'DUPLICATE_M2',parse_m3(m3raw),None
    return 'REJECT_CONFLICT',None,None
   if state!='offered':return 'REJECT_NO_OFFER',None,None
   lease=self.conn.execute('SELECT expires_at FROM offer_lease WHERE pid=?',(m2.pid,)).fetchone()
   if lease and time.time()>lease[0]:
    self.abandon(m2.pid)
    return 'REJECT_EXPIRED_OFFER',None,None
   m0=parse_m0(m0raw);m1=parse_m1(m1raw,self.B)
   h2=prehash(m0,m1,m2.pid,m2.nv,m2.ct)
   if not hmac.compare_digest(CR.kmac(A,h2,32,LABEL['PREAUTH']),m2.a2):
    return 'REJECT_ALPHA2',None,None
   sk=self.conn.execute('SELECT sk FROM kem WHERE j=?',(j,)).fetchone()[0]
   assert sk is not None
   td=time.perf_counter_ns();z=CR.decaps(sk,m2.ct);decap_ms=(time.perf_counter_ns()-td)/1e6
   self.counts['decap']+=1
   K=derivemix(A,z,h2);cv=CR.kmac(K,h2,32,LABEL['CONFV'])
   expected=CR.kmac(cv,h2+m2.a2,32,LABEL['FINV'])
   if fault=='before_completion_commit':
    raise RuntimeError('INJECTED_CRASH_BEFORE_COMPLETION_COMMIT')
   if not hmac.compare_digest(expected,m2.tv):
    self.abandon(m2.pid)
    return 'REJECT_TAUV',None,None
   h=h_full(m0,m1,m2);sid=sid_for(m1,m2.pid,h)
   kr=CR.kmac(K,h,32,LABEL['CONFR']);ka=CR.kmac(K,h,32,LABEL['APP'])
   m3=M3(sid,CR.kmac(kr,h+sid,32,LABEL['FINR']))
   tcommit=time.perf_counter_ns()
   def f():
    rr,gg=self._group_rec(m2.pid)
    if rr[4]!='offered':raise RuntimeError('offer state changed')
    self.conn.execute("UPDATE kem SET state='spent',sk=NULL WHERE j=? AND state='reserved'",(j,))
    self.conn.execute("UPDATE rec SET state='completed',m2h=?,m3=? WHERE pid=?",(H(m2.wire()),m3.wire(),m2.pid))
    self.conn.execute("UPDATE grp SET state='completed' WHERE g=?",(g,))
   self.tx(f)
   self.last={**self.last,'commit_completion_ms':self.last_db_commit_ms,'decap_ms':decap_ms,'total_ms':(time.perf_counter_ns()-t0)/1e6}
   if fault=='after_completion_commit':raise RuntimeError('INJECTED_CRASH_AFTER_COMPLETION_COMMIT')
   return 'COMPLETED',m3,ka
 def abandon(self,pid):
  def f():
   r,g=self._group_rec(pid)
   if not r or r[4]!='offered':return False
   g_id,w,A,a,state,j,m0raw,m1raw,m2h,m3=r
   self.conn.execute("UPDATE kem SET state='abandoned',sk=NULL WHERE j=? AND state='reserved'",(j,))
   self.conn.execute("UPDATE rec SET state='expired' WHERE pid=?",(pid,))
   self.conn.execute("UPDATE grp SET state=CASE WHEN next_attempt < q THEN 'unused' ELSE 'expired' END, active_pid=NULL WHERE g=?",(g_id,))
   return True
  return self.tx(f)
 def recover_process(self):
  # This artifact implements the checklist's conservative restart-abandon policy.
  offered=self.conn.execute("SELECT pid FROM rec WHERE state='offered'").fetchall()
  for (pid,) in offered:self.abandon(pid)
  return len(offered)
 def ledger(self):
  return [dict(j=j,state=state,has_sk=sk is not None) for j,state,sk in self.conn.execute('SELECT j,state,sk FROM kem ORDER BY j')]
 def close(self):self.conn.close()

In [ ]:
import mobiheal_core as mh
mh.CR = mh.Crypto(native=(CRYPTO_BACKEND=='NATIVE'))
mh.CR.self_test()
assert mh.LABEL['OFFER_AUTH']==b'MOBIHEAL-OFFER'
PROVENANCE['backend_actual'] = mh.CR.label
print('Protocol backend ready:',mh.CR.label)

## 2. Paper-aligned experiment drivers
All reported time distributions are derived from live processing in this notebook. The network model is a separate Monte Carlo experiment, not real wireless-packet measurements.

In [ ]:
%%writefile mobiheal_experiments.py
"""Paper-facing MobiHeal empirical experiments. Functions return raw dictionaries/DataFrames;
no previously published measurements are seeded or hardcoded as observed data.
"""
import concurrent.futures, hashlib, json, math, os, platform, random, statistics, tempfile, time
from pathlib import Path
from dataclasses import replace
import numpy as np
import pandas as pd
import mobiheal_core as mh


def pct95(x):return float(np.percentile(x,95)) if len(x) else float('nan')
def _new_pair(td,B=16,rid=b'RSU-0001',q=2,root=None):
 v=mh.Vehicle(Path(td)/'v.db',root=root)
 r=mh.RSU(Path(td)/'r.db',rid,B=B)
 d=v.verify_descriptor(r);r.provision(v,100,q=q)
 return v,r,d

def one(v,r,d,w=100,a=0):
 times={};t=time.perf_counter_ns();ctx=v.start(r.rid,w,a);times['vehicle_m0_ms']=(time.perf_counter_ns()-t)/1e6
 t=time.perf_counter_ns();st,m1=r.recv_m0(ctx['m0']);times['rsu_m0_to_m1_ms']=(time.perf_counter_ns()-t)/1e6
 assert st=='OFFERED',st
 t=time.perf_counter_ns();m2=v.send_m2(ctx,m1,d);times['vehicle_m1_to_m2_ms']=(time.perf_counter_ns()-t)/1e6
 t=time.perf_counter_ns();st,m3,rkey=r.recv_m2(m2);times['rsu_m2_to_m3_ms']=(time.perf_counter_ns()-t)/1e6
 assert st=='COMPLETED',st
 t=time.perf_counter_ns();key,accepted=v.complete(m3);times['vehicle_verify_m3_ms']=(time.perf_counter_ns()-t)/1e6
 assert key==rkey and accepted
 times.update({'commit_offer_ms':r.last['commit_offer_ms'],'commit_completion_ms':r.last['commit_completion_ms'],
               'decapsulation_ms_in_rsu':r.last['decap_ms'],
               'M0_bytes':len(ctx['m0'].wire()),'M1_bytes':len(m1.wire()),
               'M2_bytes':len(m2.wire()),'M3_bytes':len(m3.wire()),
               'total_bytes':sum(map(lambda x:len(x.wire()),(ctx['m0'],m1,m2,m3))),
               'sid_sha256':hashlib.sha256(m3.sid).hexdigest()})
 times['vehicle_total_ms']=sum(times[k] for k in ['vehicle_m0_ms','vehicle_m1_to_m2_ms','vehicle_verify_m3_ms'])
 times['rsu_total_ms']=times['rsu_m0_to_m1_ms']+times['rsu_m2_to_m3_ms']
 times['combined_local_ms']=times['vehicle_total_ms']+times['rsu_total_ms']
 return times

def run_current_handovers(seeds,trials=100,warmups=10,B=1024):
 rows=[]
 for seed in seeds:
  with tempfile.TemporaryDirectory() as td:
   v=mh.Vehicle(Path(td)/'vehicle.db',root=hashlib.sha3_256(f'vehicle-{seed}'.encode()).digest())
   r=mh.RSU(Path(td)/'rsu.db',b'RSU-MEAS',B=B)
   d=v.verify_descriptor(r)
   for k in range(trials+warmups):
    w=seed*10000+k;r.provision(v,w,q=2)
    x=one(v,r,d,w,0)
    if k>=warmups:rows.append({'seed':seed,'trial':k-warmups,'implementation':'current_beta1',**x})
   assert v.counter==trials+warmups
   r.close();v.close()
 return pd.DataFrame(rows)


def bootstrap_ci(df,col,seeds,reps=2000):
 # Resampling independent seed means, NOT treating 100 handovers as independent hosts.
 by=df.groupby('seed')[col].mean().values
 if len(by)<2:return (float('nan'),float('nan'))
 rng=np.random.default_rng(1429);samples=rng.choice(by,size=(reps,len(by)),replace=True).mean(axis=1)
 return map(float,np.percentile(samples,[2.5,97.5]))
def handover_stats(raw,seeds):
 cols=['vehicle_m0_ms','rsu_m0_to_m1_ms','vehicle_m1_to_m2_ms','rsu_m2_to_m3_ms',
       'vehicle_verify_m3_ms','vehicle_total_ms','rsu_total_ms','combined_local_ms',
       'commit_offer_ms','commit_completion_ms','decapsulation_ms_in_rsu']
 out=[]
 for c in cols:
  lo,hi=bootstrap_ci(raw,c,seeds)
  out.append(dict(metric=c,mean=float(raw[c].mean()),empirical_p95=pct95(raw[c]),CI95_low=lo,CI95_high=hi,
    unit='ms',n=len(raw),seed_count=len(seeds),evidence='native_measured' if mh.CR.native else 'DEV_ONLY'))
 return pd.DataFrame(out)


def primitive_bench(seeds,n=40,warmups=3):
 rows=[];pk,sk=mh.CR.kg();ct,z=mh.CR.encaps(pk);spk,ssk=mh.CR.sigkg();msg=b'MobiHeal-PQ-benchmark'
 leafs=[mh.leaf(b'RSU-TEST',1,j,pk) for j in range(1,9)];levels=mh.tree(leafs);path=mh.proof(levels,0)
 leafs1024=[mh.leaf(b'RSU-TEST',1,j,pk) for j in range(1,1025)]
 levels1024=mh.tree(leafs1024);path1024=mh.proof(levels1024,0)
 jobs={
 'ML-KEM-768 keygen':lambda:mh.CR.kg(),
 'ML-KEM-768 encaps':lambda:mh.CR.encaps(pk),
 'ML-KEM-768 decaps':lambda:mh.CR.decaps(sk,ct),
 'ML-DSA-65 signing':lambda:mh.CR.sign(ssk,msg),
 'ML-DSA-65 verify':lambda:mh.CR.verify(spk,msg,mh.CR.sign(ssk,msg)),
 'KMAC256-32B':lambda:mh.CR.kmac(b'a'*32,b'x'*48,32,b'MOBIHEAL-BENCH'),
 'SHA3-384':lambda:mh.H(msg),
 'Merkle proof verification B=8':lambda:mh.verify_proof(b'RSU-TEST',1,1,pk,path,levels[-1][0]),
 'Merkle proof verification B=1024':lambda:mh.verify_proof(b'RSU-TEST',1,1,pk,path1024,levels1024[-1][0])}
 for seed in seeds:
  for name,fn in jobs.items():
   for _ in range(warmups):fn()
   for j in range(n):
    start=time.perf_counter_ns();fn();dt=(time.perf_counter_ns()-start)/1e6
    rows.append(dict(seed=seed,operation=name,trial=j,ms=dt))
 return pd.DataFrame(rows)


def accounting(B):
 depth=math.ceil(math.log2(B)); m0=72;m1=8+24+16+mh.PK_N+48*depth+16;m2=1184;m3=80
 assert sum((m0,m1,m2,m3))=={64:2872,256:2968,1024:3064,4096:3160}.get(B,sum((m0,m1,m2,m3)))
 internal=B-1  # power of two
 storage=B*(mh.PK_N+mh.SK_N)+B+internal*48+48+mh.SIG_N
 return dict(B=B,depth=depth,M0=m0,M1=m1,M2=m2,M3=m3,total=m0+m1+m2+m3,
             legacy_total=m0+m1+m2+m3-16,storage_MiB=storage/2**20,source='formula_not_measured')

def comm_size_validation(Bs=(64,256,1024,4096)):
 rows=[]
 for B in Bs:
  with tempfile.TemporaryDirectory() as td:
   v,r,d=_new_pair(td,B=B,q=1)
   ctx=v.start(r.rid,100,0);status,m1=r.recv_m0(ctx['m0']);assert status=='OFFERED'
   m2=v.send_m2(ctx,m1,d);status,m3,key=r.recv_m2(m2);assert status=='COMPLETED'
   actual=[len(x.wire()) for x in (ctx['m0'],m1,m2,m3)]
   pred=accounting(B)
   assert actual==[pred[k] for k in ['M0','M1','M2','M3']],(B,actual,pred)
   rows.append({**pred,'M0_serialized':actual[0],'M1_serialized':actual[1],
                'M2_serialized':actual[2],'M3_serialized':actual[3],
                'matches_actual_serialization':True,'source':'validated_bytes_native' if mh.CR.native else 'DEV_ONLY'})
   v.close();r.close()
 return pd.DataFrame(rows)

def pool_bench(seeds,sizes=(64,256,1024)):
 rows=[]
 for B in sizes:
  for seed in seeds:
   with tempfile.TemporaryDirectory() as td:
    t=time.perf_counter_ns();r=mh.RSU(Path(td)/'pool.db',b'RSU-POOL',B=B)
    r.descriptor();ms=(time.perf_counter_ns()-t)/1e6
    size=Path(td,'pool.db').stat().st_size
    rows.append({'seed':seed,'B':B,'prep_ms':ms,'per_key_ms':ms/B,
                 'sqlite_db_file_bytes':size,'storage_accounted_MiB':accounting(B)['storage_MiB']})
    r.close()
 return pd.DataFrame(rows)

def functional_replay_tests(seeds):
 rows=[]
 def record(seed,name,ok,observed='',scope='functional'):
  rows.append(dict(seed=seed,test=name,passed=bool(ok),observation=str(observed),scope=scope))
 for seed in seeds:
  with tempfile.TemporaryDirectory() as td:
   v,r,d=_new_pair(td,B=8)
   ctx=v.start(r.rid,100,0)
   for tag,kw in [('descriptor_expired',{'expired':True}),('descriptor_revoked',{'revoked':True})]:
    try:v.verify_descriptor(r,**kw);rejected=False
    except ValueError:rejected=True
    record(seed,tag,rejected)
   prev_epoch=v.known_epochs[r.rid];v.known_epochs[r.rid]=r.e+1
   try:v.verify_descriptor(r);rollback_rejected=False
   except ValueError:rollback_rejected=True
   record(seed,'descriptor_epoch_rollback_rejected',rollback_rejected)
   v.known_epochs[r.rid]=prev_epoch
   before=r.counts['alloc'];st,obj=r.recv_m0(replace(ctx['m0'],a0=b'\x00'*32))
   record(seed,'invalid_alpha0_before_allocation',st=='REJECT_ALPHA0' and r.counts['alloc']==before,st)
   st,m1=r.recv_m0(ctx['m0']);record(seed,'first_offer',st=='OFFERED',st)
   st,m1copy=r.recv_m0(ctx['m0']);record(seed,'exact_duplicate_M0_returns_same_M1',st=='DUPLICATE_M0' and m1.wire()==m1copy.wire() and r.counts['alloc']==before+1,st)
   rec_tam=replace(m1,beta=bytes([m1.beta[0]^1])+m1.beta[1:]);
   try:v.send_m2(ctx,rec_tam,d);bad=False
   except ValueError:bad=True
   record(seed,'tampered_beta1_rejected',bad)
   m2=v.send_m2(ctx,m1,d)
   before=r.counts['decap'];st,_,_=r.recv_m2(replace(m2,a2=b'\x00'*32))
   recstate=r.conn.execute('SELECT state FROM rec WHERE pid=?',(m2.pid,)).fetchone()[0]
   record(seed,'invalid_alpha2_before_decapsulation',st=='REJECT_ALPHA2' and r.counts['decap']==before and recstate=='offered',st)
   st,m3,k=r.recv_m2(m2);record(seed,'legitimate_completion_after_invalid_traffic',st=='COMPLETED',st)
   st,m3b,_=r.recv_m2(m2);record(seed,'exact_duplicate_M2_cached_M3_no_decap',st=='DUPLICATE_M2' and m3b.wire()==m3.wire() and r.counts['decap']==before+1,st)
   before=v.counter;key,accepted=v.complete(m3);key2,accepted2=v.complete(m3)
   record(seed,'duplicate_M3_no_double_counter',accepted and not accepted2 and v.counter==before+1)
   states=[x['state'] for x in r.ledger()];record(seed,'used_index_spent_terminal',states.count('spent')==1)
   v.close();r.close()
  with tempfile.TemporaryDirectory() as td:
   v,r,d=_new_pair(td,B=8,q=2)
   ctx=v.start(r.rid,100,0);st,m1=r.recv_m0(ctx['m0']);first=m1.j
   # Force a previously committed offer lease to expire: the target must
   # durably abandon the key/attempt and permit the next prepared index.
   r.conn.execute('UPDATE offer_lease SET expires_at=? WHERE pid=?',(time.time()-1,ctx['pid']))
   expired_state,_=r.recv_m0(ctx['m0'])
   record(seed,'expired_offer_auto_abandoned',expired_state=='REJECT_EXPIRED_OFFER' and r.ledger()[first-1]['state']=='abandoned',expired_state)
   st,other=r.recv_m0(ctx['m0']);record(seed,'stale_retry_zero_rejected',st in ('REJECT_GROUP','REJECT_CONFLICT'),st)
   # Old offered M2 is rejected after timeout, with no decapsulation. The vehicle
   # would use authenticated B0-B3 to reconcile its pending counter in deployment.
   stale_m2=v.send_m2(ctx,m1,d)
   before_dec=r.counts['decap'];st_stale,_,_=r.recv_m2(stale_m2)
   record(seed,'expired_offer_M2_rejected_no_decap',st_stale=='REJECT_NO_OFFER' and r.counts['decap']==before_dec,st_stale)
   # Isolated test harness reset only; NOT evidence for B0-B3.
   with v.conn:v.conn.execute('UPDATE vehicle SET pending=NULL WHERE id=1')
   ctx2=v.start(r.rid,100,1);st,m1b=r.recv_m0(ctx2['m0']);
   record(seed,'new_attempt_after_timeout',st=='OFFERED' and m1b.j!=first,st)
   # Invalid traffic must not destroy legitimate offer.
   m2=v.send_m2(ctx2,m1b,d)
   st,m3,key=r.recv_m2(m2);v.complete(m3)
   record(seed,'retry_completed_once',st=='COMPLETED' and v.counter==1,st)
   v.close();r.close()
  with tempfile.TemporaryDirectory() as td:
   v,r,d=_new_pair(td,B=8,q=1)
   ctx=v.start(r.rid,100,0);st,m1=r.recv_m0(ctx['m0'])
   # Genuine parallel duplicate sends while lock protects target.
   with concurrent.futures.ThreadPoolExecutor(max_workers=8) as ex:
    outs=list(ex.map(r.recv_m0,[ctx['m0']]*8))
   record(seed,'simultaneous_duplicate_M0',all(s=='DUPLICATE_M0' and x.wire()==m1.wire() for s,x in outs) and r.counts['alloc']==1)
   m2=v.send_m2(ctx,m1,d)
   with concurrent.futures.ThreadPoolExecutor(max_workers=8) as ex:
    outs=list(ex.map(r.recv_m2,[m2]*8))
   record(seed,'simultaneous_duplicate_M2',r.counts['decap']==1 and len([s for s,_,_ in outs if s=='COMPLETED'])==1)
   v.close();r.close()
  with tempfile.TemporaryDirectory() as td:
   # Second vehicle gets authenticated BUSY when the sole pool index is reserved.
   r=mh.RSU(Path(td)/'r.db',b'RSU-BUSY',B=1)
   v1=mh.Vehicle(Path(td)/'v1.db');v2=mh.Vehicle(Path(td)/'v2.db')
   r.provision(v1,100,q=1);r.provision(v2,101,q=1)
   c1=v1.start(r.rid,100,0);st1,m1=r.recv_m0(c1['m0'])
   c2=v2.start(r.rid,101,0);prior=r.counts['alloc']
   st2,busy=r.recv_m0(c2['m0'])
   good=st2=='BUSY' and prior==r.counts['alloc'] and v2.accept_busy(c2,(st2,*busy))
   record(seed,'authenticated_BUSY_no_allocation',good,st2)
   record(seed,'spoofed_BUSY_rejected',not v2.accept_busy(c2,(st2,busy[0],b'\x00'*16)))
   v1.close();v2.close();r.close()
 return pd.DataFrame(rows)


def chronological_healing(seeds,trials=10):
    """Chronological same-root *functional* test, with enumerated actual RSU exposure.

    This does NOT provide an application-key indistinguishability theorem.
    The test harness exposes the older session application key as an additional
    diagnostic, but the RSU database does not store a resident application key.
    """
    rows=[]
    for seed in seeds:
        for k in range(trials):
            with tempfile.TemporaryDirectory() as td:
                root=hashlib.sha3_256(f'same-root-{seed}-{k}'.encode()).digest()
                v=mh.Vehicle(Path(td)/'vehicle.db',root=root)
                past=mh.RSU(Path(td)/'past.db',b'OLD-RSU1',B=4)
                d0=v.verify_descriptor(past)
                past.provision(v,100,2)
                pastctx=v.start(past.rid,100,0)
                st,m1=past.recv_m0(pastctx['m0'])
                m2=v.send_m2(pastctx,m1,d0)
                st,m3,old_key=past.recv_m2(m2)
                v.complete(m3)
                assert v.counter==1
                # Enumerate actual persistent state plus signing key. Never export secrets.
                rec_rows=past.conn.execute('SELECT pid,A,w,m0,m1,m3 FROM rec').fetchall()
                group_rows=past.conn.execute('SELECT g,state,next_attempt,q,active_pid FROM grp').fetchall()
                groups=len(group_rows)
                old_key_records=past.conn.execute('SELECT j,pk,sk,state FROM kem').fetchall()
                kem_keys=sum(1 for entry in old_key_records if entry[2] is not None)
                cache_m1=sum(1 for r in rec_rows if r[4] is not None)
                cache_m3=sum(1 for r in rec_rows if r[5] is not None)
                queued_reports_table=past.conn.execute("SELECT 1 FROM sqlite_master WHERE type='table' AND name='pending_reports'").fetchone() is not None
                stored_application_keys_table=past.conn.execute("SELECT 1 FROM sqlite_master WHERE type='table' AND name='appkeys'").fetchone() is not None
                leaked_signing_key=past.sigsk
                assert leaked_signing_key and rec_rows and groups and kem_keys>=0
                # CorruptRSU requested scope is wider than this RSU implementation.
                # The harness also holds old_key, enabling a direct old/new-key diagnostic.
                target=mh.RSU(Path(td)/'target.db',b'NEW-RSU1',B=4)
                d1=v.verify_descriptor(target)
                target.provision(v,101,2)
                alloc0=target.counts['alloc'];st,_=target.recv_m0(pastctx['m0'])
                old_reject=st in ('REJECT_UNKNOWN','REJECT_WINDOW') and alloc0==target.counts['alloc']
                newctx=v.start(target.rid,101,0)
                changed=replace(newctx['m0'],n0=b'\x00'*16)
                st_bad,_=target.recv_m0(changed)
                invalid_reject=st_bad=='REJECT_ALPHA0' and target.counts['alloc']==alloc0
                # Try a MAC calculated using the actually disclosed old-RSU credential.
                old_A=rec_rows[0][1]
                forged=replace(newctx['m0'],a0=mh.alpha0(target.rid,newctx['m0'],old_A))
                st_forged,_=target.recv_m0(forged)
                old_credential_forgery_rejected=(st_forged=='REJECT_ALPHA0' and target.counts['alloc']==alloc0)
                st,m1=target.recv_m0(newctx['m0']);m2=v.send_m2(newctx,m1,d1)
                st,m3,new_key=target.recv_m2(m2);vehicle_key,ok=v.complete(m3)
                completed=(st=='COMPLETED' and ok and new_key==vehicle_key and new_key!=old_key and v.counter==2)
                rows.append(dict(seed=seed,trial=k,old_RSU_compromised=True,root_retained=True,
                    old_replay_rejected=old_reject,altered_request_rejected=invalid_reject,
                    old_credential_forgery_rejected=old_credential_forgery_rejected,
                    first_eligible_handover_completed=completed,
                    unauthorized_target_allocations=target.counts['alloc']-1,
                    eligible_completed_transition_count=1 if completed else 0,
                    exposed_signing_key=bool(leaked_signing_key),
                    exposed_mobility_record_count=len(rec_rows),exposed_group_count=int(groups),
                    exposed_unerased_old_KEM_keys=int(kem_keys),
                    exposed_cached_M1_count=int(cache_m1),exposed_cached_M3_count=int(cache_m3),
                    queued_report_storage_implemented=bool(queued_reports_table),
                    persisted_application_key_storage_implemented=bool(stored_application_keys_table),
                    old_application_key_available_to_test_harness=bool(old_key),
                    new_application_key_differs_from_old=(new_key!=old_key),
                    exposure_scope='persisted local RSU records/signing key/remaining KEM secrets; queued reports and resident app-key store absent',
                    evidence='functional_native' if mh.CR.native else 'DEV_ONLY'))
                v.close();past.close();target.close()
    return pd.DataFrame(rows)

def crash_suite(seeds):
    """Five explicit PROCESS restart fault points; no physical power-loss claims."""
    rows=[]
    fault_points=['after_M1','during_M2_before_commit','after_completion_commit_before_M3',
                  'vehicle_before_accept_commit','vehicle_after_accept_commit']
    for seed in seeds:
        for stage in fault_points:
            with tempfile.TemporaryDirectory() as td:
                v,r,d=_new_pair(td,B=8,q=2)
                ctx=v.start(r.rid,100,0);st,m1=r.recv_m0(ctx['m0']);assert st=='OFFERED'
                m2=None;m3=None;expected_key=None;expected_m3_wire=None
                status='';expect='';cached_M3_exact=None;vehicle_restored_key_usable=None
                try:
                    if stage=='after_M1':
                        expect='abandoned after RSU process restart'
                    elif stage=='during_M2_before_commit':
                        m2=v.send_m2(ctx,m1,d)
                        try:r.recv_m2(m2,fault='before_completion_commit')
                        except RuntimeError:pass
                        expect='abandoned after RSU process restart'
                    elif stage=='after_completion_commit_before_M3':
                        m2=v.send_m2(ctx,m1,d)
                        expected_key=mh.JDEC(v.row()[2])['ka']
                        try:r.recv_m2(m2,fault='after_completion_commit')
                        except RuntimeError:pass
                        expected_m3_wire=r.conn.execute('SELECT m3 FROM rec WHERE pid=?',(ctx['pid'],)).fetchone()[0]
                        assert expected_m3_wire is not None
                        expect='spent state/cached M3 restored, vehicle session key usable'
                    elif stage=='vehicle_before_accept_commit':
                        m2=v.send_m2(ctx,m1,d);status,m3,k=r.recv_m2(m2);assert status=='COMPLETED'
                        expected_key=mh.JDEC(v.row()[2])['ka']
                        expect='pending retained, counter unchanged before M3'
                    elif stage=='vehicle_after_accept_commit':
                        m2=v.send_m2(ctx,m1,d);status,m3,k=r.recv_m2(m2);v.complete(m3)
                        expected_key=k
                        expect='advanced counter retained, duplicate M3 idempotent'
                    oldcounter=v.counter
                    old_sigpk,old_sigsk=r.sigpk,r.sigsk
                    r.close();v.close()
                    v=mh.Vehicle(Path(td)/'v.db')
                    r=mh.RSU(Path(td)/'r.db',b'RSU-0001',B=8,initialize=False,sigkeys=(old_sigpk,old_sigsk))
                    abandoned=r.recover_process()
                    if stage in ('after_M1','during_M2_before_commit'):
                        kem=r.conn.execute('SELECT state,sk FROM kem WHERE j=?',(m1.j,)).fetchone()
                        next_attempt=r.conn.execute('SELECT next_attempt FROM grp').fetchone()[0]
                        ok=(kem[0]=='abandoned' and kem[1] is None and next_attempt==1 and abandoned==1 and v.counter==oldcounter)
                    elif stage=='after_completion_commit_before_M3':
                        status,m3r,_=r.recv_m2(v.retransmit_m2())
                        cached_M3_exact=m3r is not None and m3r.wire()==expected_m3_wire
                        ka,inc=v.complete(m3r)
                        vehicle_restored_key_usable=(ka==expected_key and
                            mh.CR.kmac(ka,b'POST-RESTART-APPLICATION-CHECK',32,b'MOBIHEAL-APP-CHECK')==
                            mh.CR.kmac(expected_key,b'POST-RESTART-APPLICATION-CHECK',32,b'MOBIHEAL-APP-CHECK'))
                        ok=(status=='DUPLICATE_M2' and inc and cached_M3_exact and
                            vehicle_restored_key_usable and v.counter==oldcounter+1 and abandoned==0)
                    elif stage=='vehicle_before_accept_commit':
                        pending=v.row()[2] is not None
                        held=v.counter==oldcounter
                        ka,inc=v.complete(m3)
                        vehicle_restored_key_usable=(ka==expected_key)
                        ok=pending and held and inc and vehicle_restored_key_usable and v.counter==oldcounter+1
                    else:
                        ka,inc=v.complete(m3)
                        vehicle_restored_key_usable=(ka==expected_key)
                        ok=not inc and vehicle_restored_key_usable and v.counter==oldcounter and abandoned==0
                    allstate=r.ledger()
                    ok=ok and all(z['state']!='free' for z in allstate if z['j']==m1.j)
                    rows.append(dict(seed=seed,fault=stage,passed=bool(ok),observation=expect,
                        cached_M3_exact_match=cached_M3_exact,
                        vehicle_restored_application_key_usable=vehicle_restored_key_usable,
                        RSU_application_key_restore_tested=False,
                        process_restart_only=True,power_loss_tested=False,erasure_physical_tested=False))
                finally:
                    try:r.close();v.close()
                    except Exception:pass
    return pd.DataFrame(rows)

def rejection_bench(seeds,reps=1000):
 rows=[]
 for seed in seeds:
  with tempfile.TemporaryDirectory() as td:
   v,r,d=_new_pair(td,B=8,q=1);ctx=v.start(r.rid,100,0)
   status,m1=r.recv_m0(ctx['m0']);assert status=='OFFERED'
   m2=v.send_m2(ctx,m1,d)
   cases={'unknown_pid':replace(ctx['m0'],pid=b'\x99'*16),
          'known_pid_invalid_alpha0':replace(ctx['m0'],a0=b'\x00'*32),
          'known_pid_invalid_alpha2':replace(m2,a2=b'\x00'*32)}
   for label,packet in cases.items():
    before=dict(r.counts);t=time.perf_counter_ns()
    for _ in range(reps):
     st,_=r.recv_m0(packet) if label!='known_pid_invalid_alpha2' else r.recv_m2(packet)[:2]
    ns=time.perf_counter_ns()-t
    rows.append(dict(seed=seed,case=label,requests=reps,elapsed_s=ns/1e9,
      requests_per_s=reps*1e9/ns,mean_us=ns/1e3/reps,unauthorized_allocation_delta=r.counts['alloc']-before['alloc'],
      unauthorized_decapsulation_delta=r.counts['decap']-before['decap'],pending_offer_survived=bool(r.conn.execute("SELECT state FROM rec WHERE pid=?",(ctx['pid'],)).fetchone()[0]=='offered')))
   st,m3,key=r.recv_m2(m2);assert st=='COMPLETED';v.complete(m3)
   v.close();r.close()
 return pd.DataFrame(rows)

def concurrency_bench(seeds,levels=(1,8,32,64,128),sessions=128):
 rows=[]
 for seed in seeds:
  for workers in levels:
   with tempfile.TemporaryDirectory() as td:
    r=mh.RSU(Path(td)/'rsu.db',b'RSU-CONC',B=max(sessions,8))
    participants=[]
    for j in range(sessions):
     v=mh.Vehicle(Path(td)/f'v{j}.db')
     d=v.verify_descriptor(r);r.provision(v,100+j,q=1)
     participants.append((v,d,100+j))
    def worker(args):
     v,d,w=args
     t=time.perf_counter_ns();z=one(v,r,d,w,0);return (time.perf_counter_ns()-t)/1e6,z
    t=time.perf_counter_ns()
    with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as ex:values=list(ex.map(worker,participants))
    elapsed=(time.perf_counter_ns()-t)/1e9
    lat=[x for x,_ in values];assert len(lat)==sessions
    rows.append(dict(seed=seed,concurrent_workers=workers,definition='fixed_batch',sessions=sessions,
                     completed=sessions,wall_elapsed_s=elapsed,completed_per_s=sessions/elapsed,
                     mean_session_ms=float(np.mean(lat)),p95_session_ms=pct95(lat),
                     outstanding_population_not_maintained=True))
    for v,_,_ in participants:v.close()
    r.close()
 return pd.DataFrame(rows)


def network_simulation(raw_handover,seeds,trials=300,losses=(0,.01,.03,.05,.1,.2),
  delays=(1.,5.,10.,20.),speeds=(30,60,90,120),zones=(10.,25.,50.),attempts=4,rto=50.,rate_mbps=6.):
 # This simulation is an *independent packet erasure* abstraction with cached exact replies,
 # not an execution of ML-KEM per trial, and not physical V2X channel simulation.
 stage={k:float(raw_handover[k].median()) for k in ['vehicle_m0_ms','rsu_m0_to_m1_ms','vehicle_m1_to_m2_ms','rsu_m2_to_m3_ms','vehicle_verify_m3_ms']}
 sizes=[72,1728,1184,80]
 rows=[]
 for seed in seeds:
  rng=random.Random(seed+98989)
  for p in losses:
   for delay in delays:
    for speed in speeds:
     for zone in zones:
      dwell_ms=3600.*zone/speed
      for trial in range(trials):
       now=stage['vehicle_m0_ms'];fail_reason=None;stage1_sends=0;stage2_sends=0
       for stage_index in (0,1):
        success=False
        for n in range(attempts):
         if stage_index==0:stage1_sends+=1;req_n,resp_n=0,1;server_proc=stage['rsu_m0_to_m1_ms'] if n==0 else 0.
         else:stage2_sends+=1;req_n,resp_n=2,3;server_proc=stage['rsu_m2_to_m3_ms'] if n==0 else 0.
         # RTO timer starts when the stage request is sent; initial send counts.
         spent_req=8.*sizes[req_n]/(1000*rate_mbps)+delay
         req_arrives=(rng.random()>=p)
         spent_resp=8.*sizes[resp_n]/(1000*rate_mbps)+delay
         reply_arrives=req_arrives and rng.random()>=p
         if reply_arrives:
          now+=spent_req+server_proc+spent_resp;success=True;break
         now+=rto
         if now>dwell_ms:break
        if not success:
         fail_reason='dwell_deadline' if now>dwell_ms else 'retry_limit';break
        if stage_index==0:now+=stage['vehicle_m1_to_m2_ms']
        if now>dwell_ms:fail_reason='dwell_deadline';break
       if fail_reason is None:
        now+=stage['vehicle_verify_m3_ms']
        if now>dwell_ms:fail_reason='dwell_deadline'
       rows.append({'seed':seed,'trial':trial,'loss_prob':p,'one_way_ms':delay,
           'speed_kmh':speed,'zone_m':zone,'dwell_ms':dwell_ms,
           'successful':fail_reason is None,'failure_reason':fail_reason or 'none',
           'completion_ms':now if fail_reason is None else np.nan,
           'stage1_sends':stage1_sends,'stage2_sends':stage2_sends,
           'retry_limit_includes_initial':True,'RTO_ms':rto})
 return pd.DataFrame(rows)

def network_summary(network):
 rows=[]
 for key,sub in network.groupby(['loss_prob','one_way_ms','speed_kmh','zone_m']):
  success=sub[sub.successful];counts=sub.failure_reason.value_counts()
  rows.append(dict(zip(['loss_prob','one_way_ms','speed_kmh','zone_m'],key),n=len(sub),successes=len(success),
    success_rate_pct=100*len(success)/len(sub),mean_completion_ms=float(success.completion_ms.mean()) if len(success) else float('nan'),
    p95_completion_ms=pct95(success.completion_ms.dropna().values),
    retry_failures=int(counts.get('retry_limit',0)),deadline_failures=int(counts.get('dwell_deadline',0))))
 return pd.DataFrame(rows)


def plot_outputs(local,network_s,wire,fig_dir):
 import matplotlib;matplotlib.use('Agg')
 import matplotlib.pyplot as plt
 fig_dir=Path(fig_dir);fig_dir.mkdir(parents=True,exist_ok=True)
 plt.rcParams.update({'font.family':'serif','font.size':9,'axes.grid':True,'grid.linestyle':'--','grid.alpha':.30,
   'axes.spines.top':False,'axes.spines.right':False,'savefig.bbox':'tight'})
 files=[]
 def save(name):
  for ext in ('pdf','png'):
   target=fig_dir/(name+'.'+ext);plt.savefig(target,dpi=260);files.append(target)
  plt.close()
 # 1. stages (not sum)
 names=['vehicle_m0_ms','rsu_m0_to_m1_ms','vehicle_m1_to_m2_ms','rsu_m2_to_m3_ms','vehicle_verify_m3_ms']
 short=['Vehicle M0','RSU M0 to M1','Vehicle M1 to M2','RSU M2 to M3','Vehicle M3']
 data=local.set_index('metric').loc[names]
 fig,ax=plt.subplots(figsize=(3.65,2.45))
 y=np.arange(5);means=data['mean'].values;lo=means-data['CI95_low'].values;hi=data['CI95_high'].values-means
 ax.barh(y,means,color=['#a9cee5','#f5c99f','#b6d9bc','#efb1b2','#a9cee5'],edgecolor='#627080',lw=.6)
 ax.errorbar(means,y,xerr=np.vstack((np.maximum(lo,0),np.maximum(hi,0))),fmt='none',ecolor='#293a52',capsize=2)
 ax.set_yticks(y,short);ax.invert_yaxis();ax.set_xlabel('Local processing time (ms)');fig.tight_layout();save('fig_handover_stage_latency')
 # 2. overall loss, average all speed/zone/delay cells (unweighted equal trials)
 overall=network_s.groupby('loss_prob',as_index=False).agg(success=('successes','sum'),n=('n','sum'))
 overall['pct']=100*overall.success/overall.n
 fig,ax=plt.subplots(figsize=(3.65,2.4));ax.plot(overall.loss_prob*100,overall.pct,color='#2d79a7',marker='o',lw=1.65)
 ax.set(xlabel='Packet erasure probability (%)',ylabel='Successful handovers (%)');ax.set_ylim(0,101);fig.tight_layout();save('fig_loss_success_rate')
 # 3. loss/p95: mean of per-cell p95 values; never pooled p95
 f=network_s.groupby(['loss_prob','one_way_ms'],as_index=False)['p95_completion_ms'].mean()
 fig,ax=plt.subplots(figsize=(3.65,2.55))
 for delay,color in zip(sorted(f.one_way_ms.unique()),['#2d79a7','#e38a38','#37915c','#cb555c']):
  sl=f[f.one_way_ms==delay];ax.plot(sl.loss_prob*100,sl.p95_completion_ms,marker='o',ms=3.5,lw=1.3,color=color,label=f'{delay:g} ms')
 ax.set(xlabel='Packet erasure probability (%)',ylabel='Mean cell-wise p95 latency (ms)');ax.legend(title='One-way delay',fontsize=7);fig.tight_layout();save('fig_loss_p95_latency')
 # 4. accounted messages
 fig,ax=plt.subplots(figsize=(3.65,2.55));x=np.arange(len(wire));bot=np.zeros(len(wire))
 for k,col in [('M0','#a9cee5'),('M1','#f5c99f'),('M2','#b6d9bc'),('M3','#efb1b2')]:
  ax.bar(x,wire[k],bottom=bot,label=k,color=col,edgecolor='#ffffff',lw=.5);bot+=wire[k]
 ax.set_xticks(x,wire.B.astype(str));ax.set(xlabel='ML-KEM pool size B',ylabel='Warm-cache bytes');ax.legend(ncol=4,fontsize=7);fig.tight_layout();save('fig_message_size_breakdown')
 return files

In [ ]:
import mobiheal_experiments as xp
assert xp.accounting(1024)['total']==3064
print('Experiment drivers ready; B=1024 wire accounting =',xp.accounting(1024)['total'],'bytes')

## 3. Table VII — chronological same-root recovery, replay, credential-leakage negative tests
The exact same vehicle root survives the earlier compromised RSU and the next target. This checks sampled acceptance events, not universal computational AKE security.

In [ ]:
%%writefile recovery_table_experiment.py
"""Chronological same-root MobiHeal table experiment.

Records RSU and vehicle unauthorized ACCEPTANCE EVENTS explicitly rather than
inferring them only from rejection codes. Designed for the MobiHeal research
prototype, not a general computational security proof.
"""
from __future__ import annotations

import csv
import hashlib
import hmac
import json
import platform
import secrets
import tempfile
from dataclasses import replace
from pathlib import Path

import mobiheal_core as mh

EXPECTED_COLUMNS = [
    'seed','trial','pool_B','crypto_mode','setup_completed','error',
    'old_transcript_replay_tested','old_transcript_replay_rejected',
    'old_credential_authentication_tested','old_credential_authentication_rejected',
    'altered_context_tested','altered_context_rejected',
    'eligible_completion_tested','first_eligible_honest_target_completion',
    'unauthorized_target_key_allocations',
    'unauthorized_RSU_completion_events',
    'unauthorized_vehicle_acceptance_events',
    'malicious_M0_submissions','malicious_M2_submissions','malicious_M3_submissions',
    'matching_keys_tested','matching_application_keys',
    'prior_and_new_keys_distinct','root_retained',
    'old_RSU_signing_key_exposed','old_RSU_credential_exposed',
    'old_RSU_cached_M1_exposed','old_RSU_cached_M3_exposed',
    'old_RSU_resident_application_key_in_db',
    'old_RSU_pending_reports_implemented',
    'target_reserved_count_for_honest_request',
    'target_total_decaps','vehicle_counter_after',
    'old_M0_status','old_M2_status','old_credential_status',
    'altered_request_status','invalid_M2_status',
]


def one_chronological_trial(seed: int, trial: int, pool_size: int = 8) -> dict:
    """Two actual handovers with one retained root; malicious packets interleaved.

    One old R transfer completes before old-RSU credential disclosure.
    Old M0 and M2 are replayed at the new target. A leaked old-RSU A is
    used in a forged M0 for the new target. An authenticated new M0 is tampered
    and a forged M2 is submitted. Before M3, the vehicle sees old and forged
    confirmation messages. Only after all attacks does the new target handover
    complete honestly. All accept/allocate events in attack paths are counted.
    """
    row = {k: 0 for k in EXPECTED_COLUMNS}
    row.update(seed=seed, trial=trial, pool_B=pool_size, crypto_mode='native' if mh.CR.native else 'DEV_ONLY',
               setup_completed=False, error='', old_M0_status='', old_M2_status='',
               old_credential_status='', altered_request_status='', invalid_M2_status='')
    # RNG used for keys is secrets, never a deterministic Python test PRNG.
    with tempfile.TemporaryDirectory(prefix='mobiheal_recovery_') as temp:
        directory = Path(temp)
        old = new = vehicle = None
        try:
            root = secrets.token_bytes(32)
            vehicle = mh.Vehicle(directory / 'vehicle.db', root=root)
            old = mh.RSU(directory / 'old_rsu.db', b'OLD-RSU1', B=pool_size)
            descriptor_old = vehicle.verify_descriptor(old)
            old.provision(vehicle, w=100, q=2)

            previous = vehicle.start(old.rid, 100, 0)
            status, old_m1 = old.recv_m0(previous['m0'])
            if status != 'OFFERED': raise AssertionError(f'old M0: {status}')
            old_m2 = vehicle.send_m2(previous, old_m1, descriptor_old)
            status, old_m3, old_rsu_key = old.recv_m2(old_m2)
            if status != 'COMPLETED': raise AssertionError(f'old M2: {status}')
            old_vehicle_key, old_accepted = vehicle.complete(old_m3)
            if not (old_accepted and hmac.compare_digest(old_rsu_key, old_vehicle_key)
                    and vehicle.counter == 1):
                raise AssertionError('past-RSU honest completion failed')

            # Simulated CorruptRSU: explicitly inspect what this implementation stores.
            leaked_old_A = old.conn.execute('SELECT A FROM rec WHERE pid=?',
                                            (previous['pid'],)).fetchone()[0]
            row['old_RSU_signing_key_exposed'] = bool(old.sigsk)
            row['old_RSU_credential_exposed'] = bool(leaked_old_A)
            row['old_RSU_cached_M1_exposed'] = bool(old.conn.execute(
                'SELECT m1 FROM rec WHERE pid=?', (previous['pid'],)).fetchone()[0])
            row['old_RSU_cached_M3_exposed'] = bool(old.conn.execute(
                'SELECT m3 FROM rec WHERE pid=?', (previous['pid'],)).fetchone()[0])
            row['old_RSU_resident_application_key_in_db'] = bool(old.conn.execute(
                "SELECT name FROM sqlite_master WHERE type='table' AND name='appkeys'").fetchone())
            row['old_RSU_pending_reports_implemented'] = bool(old.conn.execute(
                "SELECT name FROM sqlite_master WHERE type='table' AND name='pending_reports'").fetchone())
            # old_rsu_key is present in the local test harness, not stored by the old RSU.

            new = mh.RSU(directory / 'new_rsu.db', b'NEW-RSU1', B=pool_size)
            descriptor_new = vehicle.verify_descriptor(new)
            new.provision(vehicle, w=101, q=2)
            row['root_retained'] = hmac.compare_digest(vehicle.root, root)
            fresh = vehicle.start(new.rid, 101, 0)

            def attack_m0(m0):
                row['malicious_M0_submissions'] += 1
                before = new.counts['alloc']
                outcome, obj = new.recv_m0(m0)
                row['unauthorized_target_key_allocations'] += max(0, new.counts['alloc'] - before)
                return outcome, obj

            def attack_m2(m2):
                row['malicious_M2_submissions'] += 1
                outcome, m3, key = new.recv_m2(m2)
                # This is an actual receiver completion event, NOT an inferred count.
                row['unauthorized_RSU_completion_events'] += int(outcome == 'COMPLETED')
                return outcome, m3, key

            def attack_m3(m3):
                row['malicious_M3_submissions'] += 1
                try:
                    _key, accepted = vehicle.complete(m3)
                    # This is an actual vehicle acceptance event, NOT a rejection inference.
                    row['unauthorized_vehicle_acceptance_events'] += int(accepted)
                    return 'ACCEPTED' if accepted else 'DUPLICATE_NON_ACCEPTING'
                except ValueError:
                    return 'REJECTED'

            row['old_transcript_replay_tested'] = True
            old_m0_status, _ = attack_m0(previous['m0'])
            old_m2_status, _, _ = attack_m2(old_m2)
            row['old_M0_status'] = old_m0_status
            row['old_M2_status'] = old_m2_status
            row['old_transcript_replay_rejected'] = (
                old_m0_status.startswith('REJECT') and old_m2_status.startswith('REJECT'))

            # Stronger input leak: even knowing target P and M0 context does not make
            # the old target A a valid MAC key for the independent new target.
            row['old_credential_authentication_tested'] = True
            forged_old_A = replace(fresh['m0'], a0=mh.alpha0(new.rid, fresh['m0'], leaked_old_A))
            status, _ = attack_m0(forged_old_A)
            row['old_credential_status'] = status
            row['old_credential_authentication_rejected'] = (status == 'REJECT_ALPHA0')

            row['altered_context_tested'] = True
            original_nonce = fresh['m0'].n0
            modified_nonce = bytes([original_nonce[0] ^ 0x80]) + original_nonce[1:]
            invalid_request = replace(fresh['m0'], n0=modified_nonce)
            status, _ = attack_m0(invalid_request)
            row['altered_request_status'] = status
            row['altered_context_rejected'] = (status == 'REJECT_ALPHA0')

            # Honest M0 should now reserve exactly one pool index.
            before = new.counts['alloc']
            status, current_m1 = new.recv_m0(fresh['m0'])
            if status != 'OFFERED': raise AssertionError(f'new honest M0: {status}')
            row['target_reserved_count_for_honest_request'] = new.counts['alloc'] - before
            current_m2 = vehicle.send_m2(fresh, current_m1, descriptor_new)

            # Unauthorized M2 submitted while there is a legitimate offer. The
            # invalid pre-decapsulation tag must not commit RSU completion.
            modified_a2 = bytes([current_m2.a2[0] ^ 0x80]) + current_m2.a2[1:]
            bad_m2 = replace(current_m2, a2=modified_a2)
            status, _, _ = attack_m2(bad_m2)
            row['invalid_M2_status'] = status

            status, current_m3, new_rsu_key = new.recv_m2(current_m2)
            if status != 'COMPLETED': raise AssertionError(f'new honest M2: {status}')

            # Attack the *actual* vehicle acceptance API while it has a pending
            # new-target session. Both old and forged M3 must be rejected.
            old_m3_result = attack_m3(old_m3)
            bad_tag = bytes([current_m3.tr[0] ^ 0x80]) + current_m3.tr[1:]
            tampered_m3_result = attack_m3(replace(current_m3, tr=bad_tag))

            row['eligible_completion_tested'] = True
            row['matching_keys_tested'] = True
            new_vehicle_key, new_accepted = vehicle.complete(current_m3)
            row['matching_application_keys'] = bool(hmac.compare_digest(new_rsu_key, new_vehicle_key))
            row['prior_and_new_keys_distinct'] = not hmac.compare_digest(new_vehicle_key, old_vehicle_key)
            row['first_eligible_honest_target_completion'] = bool(
                new_accepted and row['matching_application_keys']
                and row['prior_and_new_keys_distinct']
                and new.conn.execute('SELECT state FROM rec WHERE pid=?',
                                     (fresh['pid'],)).fetchone()[0] == 'completed'
                and vehicle.counter == 2
            )
            row['target_total_decaps'] = new.counts['decap']
            row['vehicle_counter_after'] = vehicle.counter
            # These assertions would catch a silent test-path omission.
            if row['malicious_M0_submissions'] != 3 or row['malicious_M2_submissions'] != 2 \
               or row['malicious_M3_submissions'] != 2:
                raise AssertionError('not all malicious packets were actually submitted')
            row['setup_completed'] = True
            row['error'] = ''
        except Exception as exc:
            row['error'] = f'{type(exc).__name__}: {exc}'
        finally:
            for entity in (new, old, vehicle):
                if entity is not None:
                    try: entity.close()
                    except Exception: pass
    return row


def run_trials(seeds=range(2601,2611), per_seed=10, pool_size=8):
    rows=[]
    for seed in seeds:
        for t in range(per_seed):
            row=one_chronological_trial(int(seed),int(t),int(pool_size))
            rows.append(row)
            if row['error']:
                print(f"FAIL seed={seed} trial={t}: {row['error']}")
        print(f'Finished seed {seed}: {per_seed} trials')
    return rows


def produce_counts(rows):
    """Compute final table from observed events, not asserted/expected values."""
    total=len(rows)
    success=lambda key:sum(int(bool(r[key])) for r in rows if r[key])
    tested=lambda key:sum(int(bool(r[key])) for r in rows)
    metrics={
      'total_trials':total,'completed_test_paths':tested('setup_completed'),
      'old_replay_rejections':success('old_transcript_replay_rejected'),
      'old_replay_trials':tested('old_transcript_replay_tested'),
      'old_credential_rejections':success('old_credential_authentication_rejected'),
      'old_credential_trials':tested('old_credential_authentication_tested'),
      'altered_context_rejections':success('altered_context_rejected'),
      'altered_context_trials':tested('altered_context_tested'),
      'honest_completions':success('first_eligible_honest_target_completion'),
      'honest_completion_trials':tested('eligible_completion_tested'),
      'matching_keys':success('matching_application_keys'),
      'matching_keys_trials':tested('matching_keys_tested'),
      'unauthorized_target_key_allocations':sum(int(r['unauthorized_target_key_allocations']) for r in rows),
      'unauthorized_RSU_completions':sum(int(r['unauthorized_RSU_completion_events']) for r in rows),
      'unauthorized_vehicle_acceptances':sum(int(r['unauthorized_vehicle_acceptance_events']) for r in rows),
      'malicious_M0_submissions':sum(int(r['malicious_M0_submissions']) for r in rows),
      'malicious_M2_submissions':sum(int(r['malicious_M2_submissions']) for r in rows),
      'malicious_M3_submissions':sum(int(r['malicious_M3_submissions']) for r in rows),
      'failed_or_incomplete_trials':sum(int(not r['setup_completed']) for r in rows),
      'cryptographic_mode': 'native' if mh.CR.native else 'DEV_ONLY',
      'pool_B': int(next(iter(rows))['pool_B']) if rows and 'pool_B' in rows[0] else None,
    }
    return metrics


def render_table(m):
    fmt=lambda a,b:f"{m[a]}/{m[b]}" if m[b] else 'Not tested'
    lines=[
      r'\begin{table}[!t]',
      r'\caption{Chronological Same-Root Recovery Tests}',
      r'\label{tab:healingtests}',
      r'\centering\footnotesize',
      r'\begin{tabular}{@{}p{0.69\linewidth}r@{}}',
      r'\toprule',
      r'Outcome & Result\\',
      r'\midrule',
      f"Old-transcript replay rejected & {fmt('old_replay_rejections','old_replay_trials')}" + r'\\',
      f"Old-credential authentication rejected & {fmt('old_credential_rejections','old_credential_trials')}" + r'\\',
      f"Altered target/request context rejected & {fmt('altered_context_rejections','altered_context_trials')}" + r'\\',
      f"First eligible honest-target completion & {fmt('honest_completions','honest_completion_trials')}" + r'\\',
      f"Unauthorized target key allocations & {m['unauthorized_target_key_allocations']}" + r'\\',
      f"Unauthorized RSU completions & {m['unauthorized_RSU_completions']}" + r'\\',
      f"Unauthorized vehicle acceptances & {m['unauthorized_vehicle_acceptances']}" + r'\\',
      f"Matching application keys at honest completion & {fmt('matching_keys','matching_keys_trials')}" + r'\\',
      r'\bottomrule',
      r'\end{tabular}',
      r'\end{table}',
    ]
    return '\n'.join(lines)+'\n'


def write_raw_csv(rows,path):
    with open(path,'w',encoding='utf-8',newline='') as f:
        w=csv.DictWriter(f,fieldnames=EXPECTED_COLUMNS,extrasaction='ignore')
        w.writeheader();w.writerows(rows)

In [ ]:
import recovery_table_experiment as recovery
recovery_raw = recovery.run_trials(seeds=SEEDS,per_seed=CONFIG['same_root_trials_per_seed'],pool_size=8)
recovery_df=pd.DataFrame(recovery_raw)
recovery_df.to_csv(ROOT/'table_VII_recovery_raw.csv',index=False)
recovery_counts=recovery.produce_counts(recovery_raw)
(ROOT/'table_VII_recovery_summary.json').write_text(json.dumps(recovery_counts,indent=2))
(ROOT/'table_VII_recovery.tex').write_text(recovery.render_table(recovery_counts))
assert recovery_df.error.eq('').all(), 'At least one recovery trial failed: see raw CSV'
display(pd.DataFrame([recovery_counts]).T.rename(columns={0:'observed'}))

## 4. Table IX — restart and cached-reply validation

In [ ]:
functional=xp.functional_replay_tests(SEEDS)
crash=xp.crash_suite(SEEDS)
functional.to_csv(ROOT/'functional_controls.csv',index=False)
crash.to_csv(ROOT/'table_IX_restart_raw.csv',index=False)
crash_summary=crash.groupby('fault',as_index=False).agg(tested=('passed','size'),passed=('passed','sum'))
crash_summary.to_csv(ROOT/'table_IX_restart_summary.csv',index=False)
display(crash_summary)
assert functional.passed.all() and crash.passed.all(), 'Failing control detected: do not report pass'
print('SQLite process restart safety only; not physical power loss or secure erasure.')

## 5. Figures 3–4 — native local processing, 10×100 measured sessions in FULL
Measurements include ML-KEM, serialization, SQLite WAL/FULL commits, caches, locks and vehicle counter acceptance; provisioning / signed descriptor check happen before the measured fast path. Bootstrap confidence intervals resample seed means; empirical p95 pools individual session observations.

In [ ]:
handover=xp.run_current_handovers(SEEDS,
  trials=CONFIG['handover_trials_per_seed'],warmups=CONFIG['warmups_per_seed'],
  B=CONFIG['handover_pool_B'])
local=xp.handover_stats(handover,SEEDS)
handover.to_csv(ROOT/'handover_raw.csv',index=False)
local.to_csv(ROOT/'table_local_processing.csv',index=False)
commit_share=100*(handover.commit_offer_ms.mean()+handover.commit_completion_ms.mean())/handover.rsu_total_ms.mean()
display(local)
print(f'OBSERVED mean combined local={handover.combined_local_ms.mean():.3f} ms | RSU durable commit share={commit_share:.2f}%')
print('Paper reference: 13.360 ms combined, 75.3% RSU processing from offer/completion commits; NOT automatically reproduced.')

## 6. Figure 6 — actual M0–M3 wire length and storage accounting
`B=4096` is validated by constructing a real pool; costs from this cell are not included in the measured online latency of §5.

In [ ]:
wire=xp.comm_size_validation((64,256,1024,4096))
wire.to_csv(ROOT/'figure_6_wire_bytes.csv',index=False)
assert wire.matches_actual_serialization.all()
assert int(wire.loc[wire.B.eq(1024),'total'].iloc[0])==3064
assert dict(zip(wire.B,wire.total))=={64:2872,256:2968,1024:3064,4096:3160}
display(wire[['B','M0','M1','M2','M3','total','storage_MiB','matches_actual_serialization']])

## 7. Table XII — offline Merkle pool preparation and native primitive costs

In [ ]:
primitive=xp.primitive_bench(SEEDS,n=CONFIG['primitive_reps_per_seed'],warmups=2)
primitive.to_csv(ROOT/'primitive_raw.csv',index=False)
primitive_summary=primitive.groupby('operation',as_index=False).agg(mean_ms=('ms','mean'),p95_ms=('ms',lambda x:float(np.percentile(x,95))))
primitive_summary.to_csv(ROOT/'primitive_summary.csv',index=False)
pool=xp.pool_bench(SEEDS,sizes=CONFIG['pool_prep_sizes'])
pool.to_csv(ROOT/'pool_preparation_raw.csv',index=False)
pool_summary=pool.groupby('B',as_index=False).agg(preparation_mean_ms=('prep_ms','mean'),ms_per_key=('per_key_ms','mean'),sqlite_file_bytes=('sqlite_db_file_bytes','mean'))
pool_summary.to_csv(ROOT/'table_XII_merkle_pool_prep.csv',index=False)
display(pool_summary)

## 8. Admission, concurrency, and Figure 5 packet-loss/retry simulation
Full settings: 6 erasure rates × 4 delays × 4 speeds × 3 zone lengths × 10 seeds × 100 trials = **288,000 independent Monte Carlo trials**. Four transmissions per request/reply stage **including the initial send**; 50 ms timeout and 6 Mb/s. Simulated processing stages are calibrated from the fresh local native stage *medians*, not manuscript fixtures.

In [ ]:
admission=xp.rejection_bench(SEEDS,reps=CONFIG['invalid_requests_per_case_seed'])
concurrency=xp.concurrency_bench(SEEDS,levels=CONFIG['concurrent_workers'],sessions=CONFIG['concurrent_sessions'])
admission.to_csv(ROOT/'admission_raw.csv',index=False)
concurrency.to_csv(ROOT/'concurrency_raw.csv',index=False)
admission_summary=admission.groupby('case',as_index=False).agg(requests_per_s=('requests_per_s','mean'),invalid_allocations=('unauthorized_allocation_delta','sum'),invalid_decaps=('unauthorized_decapsulation_delta','sum'))
concurrency_summary=concurrency.groupby('concurrent_workers',as_index=False).agg(completed_per_s=('completed_per_s','mean'),mean_session_ms=('mean_session_ms','mean'),p95_session_ms=('p95_session_ms','mean'))
admission_summary.to_csv(ROOT/'admission_summary.csv',index=False)
concurrency_summary.to_csv(ROOT/'concurrency_summary.csv',index=False)
assert admission_summary.invalid_allocations.eq(0).all() and admission_summary.invalid_decaps.eq(0).all()
display(admission_summary);display(concurrency_summary)

In [ ]:
network=xp.network_simulation(handover,SEEDS,trials=CONFIG['network_trials_per_cell_seed'])
network_s=xp.network_summary(network)
network.to_csv(ROOT/'network_raw.csv',index=False)
network_s.to_csv(ROOT/'figure_5_network_cells.csv',index=False)
network_by_loss=network_s.groupby('loss_prob',as_index=False).agg(trials=('n','sum'),successes=('successes','sum'),retry_failures=('retry_failures','sum'),dwell_failures=('deadline_failures','sum'))
network_by_loss['success_rate_pct']=100*network_by_loss.successes/network_by_loss.trials
network_by_loss.to_csv(ROOT/'network_by_loss.csv',index=False)
display(network_by_loss)
print('Reference paper reports 96.665% completion at 20% erasure; current run above is independently simulated.')

## 9. Table X — atomic RSU application-session persistence, AES-GCM wrapping, pending MSC report, idempotent delivery
The original uploaded main prototype stored `spent` and cached `M3` but **not** an RSU-wrapped application key or an actual report queue. This extension implements those missing records in **the same SQLite completion transaction**, restores wrapped keys after restart using an **externally supplied** AES-GCM key, and checks duplicate local MSC receipt before queue ACK. All results below are **new experimental measurements**, not imputed Table X publication values. Local MSC delivery is not a full networked B0–B3 implementation.

In [ ]:
%%writefile mobiheal_persistence.py
"""Paper Table X companion: application-session/report persistence extension.

A research-level RSU subclass using a single atomic SQLite completion transaction.
AES-GCM wrapping key MUST be injected from outside RSU database and retained safely.
Actual controller networking and secure hardware erasure are out of scope.
"""
from __future__ import annotations
import hashlib, os, secrets, sqlite3, tempfile, time
from pathlib import Path
import numpy as np
import pandas as pd
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from dataclasses import replace
import mobiheal_core as mh

class PersistentRSU(mh.RSU):
    def __init__(self,*args,wrapping_key=None,**kwargs):
        super().__init__(*args,**kwargs)
        if wrapping_key is None:
            raise ValueError('wrapping_key must be supplied by an external key manager')
        self.wrapping_key=wrapping_key
        self.conn.executescript('''
        CREATE TABLE IF NOT EXISTS app_sessions(sid BLOB PRIMARY KEY, g BLOB, nonce BLOB, wrapped BLOB);
        CREATE TABLE IF NOT EXISTS pending_reports(g BLOB PRIMARY KEY, sid BLOB, status TEXT NOT NULL, token BLOB);
        ''')
        self.last_persist={}

    def restore_key(self,sid):
        row=self.conn.execute('SELECT nonce,wrapped FROM app_sessions WHERE sid=?',(sid,)).fetchone()
        if row is None: raise KeyError('session key is not stored')
        return AESGCM(self.wrapping_key).decrypt(row[0],row[1],sid)

    def queued_reports(self):
        return self.conn.execute("SELECT g,sid,token FROM pending_reports WHERE status='pending'").fetchall()

    def ack_report(self,g):
        with self.conn:self.conn.execute("UPDATE pending_reports SET status='acked' WHERE g=?",(g,))

    def recv_m2(self,m2,fault=None):
        t0=time.perf_counter_ns()
        with self.lock:
            rec,grp=self._group_rec(m2.pid)
            if rec is None:return 'REJECT_NO_OFFER',None,None
            g,w,A,a,state,j,m0raw,m1raw,m2h,m3raw=rec
            if state=='completed':
                if m2h==mh.H(m2.wire()): return 'DUPLICATE_M2',mh.parse_m3(m3raw),None
                return 'REJECT_CONFLICT',None,None
            if state!='offered':return 'REJECT_NO_OFFER',None,None
            lease=self.conn.execute('SELECT expires_at FROM offer_lease WHERE pid=?',(m2.pid,)).fetchone()
            if lease and time.time()>lease[0]:
                self.abandon(m2.pid)
                return 'REJECT_EXPIRED_OFFER',None,None
            m0=mh.parse_m0(m0raw);m1=mh.parse_m1(m1raw,self.B)
            h2=mh.prehash(m0,m1,m2.pid,m2.nv,m2.ct)
            if not mh.hmac.compare_digest(mh.CR.kmac(A,h2,32,mh.LABEL['PREAUTH']),m2.a2):
                return 'REJECT_ALPHA2',None,None
            sk=self.conn.execute('SELECT sk FROM kem WHERE j=?',(j,)).fetchone()[0]
            if sk is None:return 'REJECT_ERASED_INDEX',None,None
            t=time.perf_counter_ns();z=mh.CR.decaps(sk,m2.ct)
            decap_ms=(time.perf_counter_ns()-t)/1e6
            self.counts['decap']+=1
            K=mh.derivemix(A,z,h2);cv=mh.CR.kmac(K,h2,32,mh.LABEL['CONFV'])
            exp=mh.CR.kmac(cv,h2+m2.a2,32,mh.LABEL['FINV'])
            if fault=='before_completion_commit':
                raise RuntimeError('INJECTED_CRASH_BEFORE_COMPLETION_COMMIT')
            if not mh.hmac.compare_digest(exp,m2.tv):
                self.abandon(m2.pid);return 'REJECT_TAUV',None,None
            h=mh.h_full(m0,m1,m2);sid=mh.sid_for(m1,m2.pid,h)
            kr=mh.CR.kmac(K,h,32,mh.LABEL['CONFR']);ka=mh.CR.kmac(K,h,32,mh.LABEL['APP'])
            m3=mh.M3(sid,mh.CR.kmac(kr,h+sid,32,mh.LABEL['FINR']))
            t=time.perf_counter_ns();nonce=secrets.token_bytes(12)
            wrapped=AESGCM(self.wrapping_key).encrypt(nonce,ka,sid)
            wrap_ms=(time.perf_counter_ns()-t)/1e6
            # 48 sid + 16 g + 12 nonce + 48 AESGCM ciphertext = 124.
            # Report = 16 handle + 48 sid + 16 random idempotency token = 80 bytes.
            token=secrets.token_bytes(16)
            measurements={}
            def commit_all():
                rr,gg=self._group_rec(m2.pid)
                if rr[4]!='offered':raise RuntimeError('offer changed during transaction')
                self.conn.execute("UPDATE kem SET state='spent',sk=NULL WHERE j=? AND state='reserved'",(j,))
                self.conn.execute('UPDATE rec SET state=\'completed\',m2h=?,m3=? WHERE pid=?',(mh.H(m2.wire()),m3.wire(),m2.pid))
                self.conn.execute("UPDATE grp SET state='completed' WHERE g=?",(g,))
                t=time.perf_counter_ns()
                self.conn.execute('INSERT INTO app_sessions(sid,g,nonce,wrapped) VALUES(?,?,?,?)',(sid,g,nonce,wrapped))
                measurements['app_insert_ms']=(time.perf_counter_ns()-t)/1e6
                t=time.perf_counter_ns()
                self.conn.execute("INSERT INTO pending_reports(g,sid,status,token) VALUES(?,?,'pending',?)",(g,sid,token))
                measurements['report_insert_ms']=(time.perf_counter_ns()-t)/1e6
            tx_start=time.perf_counter_ns();self.tx(commit_all)
            tx_ms=(time.perf_counter_ns()-tx_start)/1e6
            self.last={**self.last,'commit_completion_ms':self.last_db_commit_ms,
                       'decap_ms':decap_ms,'total_ms':(time.perf_counter_ns()-t0)/1e6}
            self.last_persist={'wrap_ms':wrap_ms,'app_insert_ms':measurements['app_insert_ms'],
                               'report_insert_ms':measurements['report_insert_ms'],
                               'completion_transaction_ms':tx_ms,'commit_call_ms':self.last_db_commit_ms,
                               'application_fields_logical_bytes':len(sid)+len(g)+len(nonce)+len(wrapped),
                               'report_logical_bytes':len(g)+len(sid)+len(token),
                               'M3_bytes':len(m3.wire())}
            if fault=='after_completion_commit':raise RuntimeError('INJECTED_CRASH_AFTER_COMPLETION_COMMIT')
            return 'COMPLETED',m3,ka

class LocalMSC:
    """SQLite idempotent *local* receiver; NOT B0-B3 authenticated network recovery."""
    def __init__(self,path):
        self.db=str(path)
        self.conn=sqlite3.connect(self.db)
        self.conn.execute('PRAGMA journal_mode=WAL')
        self.conn.execute('PRAGMA synchronous=FULL')
        self.conn.execute('CREATE TABLE IF NOT EXISTS reports(g BLOB PRIMARY KEY,sid BLOB NOT NULL,token BLOB NOT NULL)')
    def deliver(self,g,sid,token):
        with self.conn:
            self.conn.execute('INSERT OR IGNORE INTO reports(g,sid,token) VALUES(?,?,?)',(g,sid,token))
        return self.conn.execute('SELECT sid,token FROM reports WHERE g=?',(g,)).fetchone()==(sid,token)
    def count(self):return self.conn.execute('SELECT COUNT(*) FROM reports').fetchone()[0]
    def close(self):self.conn.close()

def _prep(td,rid,seed,B=8,persistent=False):
    wrap=hashlib.sha3_256(('WRAP'+str(seed)).encode()).digest()
    v=mh.Vehicle(Path(td)/('v'+rid.decode()+'.db'))
    klass=PersistentRSU if persistent else mh.RSU
    kwargs={'wrapping_key':wrap} if persistent else {}
    r=klass(Path(td)/('r'+rid.decode()+'.db'),rid,B=B,**kwargs)
    desc=v.verify_descriptor(r)
    return v,r,desc,wrap

def _one_session(v,r,d,w):
    r.provision(v,w,q=1)
    ctx=v.start(r.rid,w,0)
    st,m1=r.recv_m0(ctx['m0']);assert st=='OFFERED'
    m2=v.send_m2(ctx,m1,d)
    t=time.perf_counter_ns();status,m3,key=r.recv_m2(m2)
    cost=(time.perf_counter_ns()-t)/1e6
    assert status=='COMPLETED'
    restored,accepted=v.complete(m3)
    assert accepted and restored==key
    return ctx,m2,m3,key,cost

def bench_persistence(seeds,per_seed=10,B=32):
    """Native timings plus genuinely exercised restart/rollback/idempotence checks.

    M2-to-M3 storage-on includes atomic key+report inserts; storage-off is core RSU.
    Test each seed on separate databases and new keys. No result is fixed to manuscript numbers.
    """
    measurements=[];checks=[]
    for seed in seeds:
        with tempfile.TemporaryDirectory() as td:
            v0,r0,d0,_=_prep(td,b'RSU-OFF1',seed,B=max(per_seed+2,B),persistent=False)
            v1,r1,d1,wrap=_prep(td,b'RSU-ON01',seed,B=max(per_seed+2,B),persistent=True)
            msc_path=Path(td)/'msc.db';msc=LocalMSC(msc_path)
            for trial in range(per_seed):
                _,_,_,_,t_off=_one_session(v0,r0,d0,10000+trial)
                ctx,m2,m3,key,t_on=_one_session(v1,r1,d1,10000+trial)
                measurements.append(dict(seed=seed,trial=trial,storage_off_m2_m3_ms=t_off,
                     storage_on_m2_m3_ms=t_on,**r1.last_persist))
                # Reopen same database and same externally managed wrap key for every trial.
                r1_sigkeys=(r1.sigpk,r1.sigsk);r1.close()
                r1=PersistentRSU(Path(td)/'rRSU-ON01.db',b'RSU-ON01',B=max(per_seed+2,B),
                    initialize=False,sigkeys=r1_sigkeys,wrapping_key=wrap)
                status,cached,_=r1.recv_m2(m2)
                restored=r1.restore_key(m3.sid)
                checks.append(dict(seed=seed,trial=trial,category='post_commit_restart',
                    passed=(status=='DUPLICATE_M2' and cached.wire()==m3.wire() and restored==key
                            and len(r1.queued_reports())>0)))
                # A controller receipt committed before ACK must be detected as duplicate after restart.
                g,sid,token=r1.conn.execute('SELECT g,sid,token FROM pending_reports WHERE sid=?',(m3.sid,)).fetchone()
                delivery_started=time.perf_counter_ns()
                assert msc.deliver(g,sid,token)
                msc.close();msc=LocalMSC(msc_path)
                was=msc.count();duplicate=msc.deliver(g,sid,token)
                r1.ack_report(g)
                receipt_sequence_ms=(time.perf_counter_ns()-delivery_started)/1e6
                measurements[-1]['local_delivery_sequence_ms']=receipt_sequence_ms
                checks.append(dict(seed=seed,trial=trial,category='receipt_before_ACK_restart',
                                   passed=(duplicate and msc.count()==was and
                                           r1.conn.execute('SELECT status FROM pending_reports WHERE g=?',(g,)).fetchone()[0]=='acked')))
                # An injected fault before the completion commit has no effects on the next offer.
                w=50000+trial;r1.provision(v1,w,q=1)
                ctx2=v1.start(r1.rid,w,0);st,m1=r1.recv_m0(ctx2['m0']);assert st=='OFFERED'
                m2b=v1.send_m2(ctx2,m1,d1)
                pending0=r1.conn.execute('SELECT COUNT(*) FROM app_sessions').fetchone()[0]
                report0=r1.conn.execute('SELECT COUNT(*) FROM pending_reports').fetchone()[0]
                try:r1.recv_m2(m2b,fault='before_completion_commit')
                except RuntimeError:pass
                still_offered=r1.conn.execute('SELECT state FROM rec WHERE pid=?',(ctx2['pid'],)).fetchone()[0]=='offered'
                tx_clean=(pending0==r1.conn.execute('SELECT COUNT(*) FROM app_sessions').fetchone()[0] and
                          report0==r1.conn.execute('SELECT COUNT(*) FROM pending_reports').fetchone()[0])
                checks.append(dict(seed=seed,trial=trial,category='mid_transaction_rollback',passed=(still_offered and tx_clean)))
                st,m3b,keyb=r1.recv_m2(m2b);assert st=='COMPLETED';v1.complete(m3b)
            v0.close();v1.close();r0.close();r1.close();msc.close()
    return pd.DataFrame(measurements),pd.DataFrame(checks)

In [ ]:
import mobiheal_persistence as persistence
persist_raw,persist_checks=persistence.bench_persistence(SEEDS,
   per_seed=CONFIG['persistence_sessions_per_seed'],B=max(CONFIG['persistence_sessions_per_seed']*2+4,16))
persist_raw.to_csv(ROOT/'table_X_persistence_raw.csv',index=False)
persist_checks.to_csv(ROOT/'table_X_persistence_checks.csv',index=False)
assert persist_checks.passed.all(), 'Persistent-state research extension failed; inspect raw checks'
persist_summary=persist_raw.drop(columns=['seed','trial']).mean(numeric_only=True).to_frame('measured_mean').reset_index().rename(columns={'index':'metric'})
persist_summary.to_csv(ROOT/'table_X_persistence_metrics.csv',index=False)
checks_summary=persist_checks.groupby('category',as_index=False).agg(passed=('passed','sum'),trials=('passed','size'))
checks_summary.to_csv(ROOT/'table_X_persistence_check_summary.csv',index=False)
display(persist_summary);display(checks_summary)
assert (persist_raw.application_fields_logical_bytes==124).all()
assert (persist_raw.report_logical_bytes==80).all() and (persist_raw.M3_bytes==80).all()

## 10. Z3 — paper-bounded state safety and crash-rollback negative control
Model: 2 record groups, 3 ML-KEM indices, 2 retry pseudonyms per group, 10 transitions; 7 safety queries (expected `unsat` if proved within bound), 6 reachability queries (expected `sat`), plus 1 rollback counterexample (expected `sat`). Results are not assumed when `z3-solver` is absent or a timeout occurs. Only atomic event interleavings are modeled.

In [ ]:
%%writefile mobiheal_z3_current.py
"""MobiHeal Section 3.5: bounded *symbolic state-machine* safety verification.

Requires: pip install z3-solver (Google Colab). Cryptographic guards and atomic
persistence are assumptions, NOT proven by these constraints. Concurrent records
are represented by all interleavings of atomic actions, not overlapping writes.
"""
from z3 import *
FREE, RESERVED, SPENT, ABANDONED = range(4)
UNUSED, OFFERED, COMPLETED, EXPIRED = range(4)
NOOP, OFFER, FINISH, TIMEOUT, CRASH, BAD_M0, BAD_M2, RETX_M0, RETX_M2, CONFIRM_M3, SLOW_B0, AUTH_B3, BAD_B3 = range(13)
ACTIONS = ['noop','authenticated_M0_and_durable_M1','valid_M2_and_spent_commit','offer_timeout','crash_restart','invalid_M0','invalid_M2','retransmit_exact_M0','retransmit_exact_M2','verify_M3','slow_recovery_B0','authenticated_slow_B3','invalid_slow_B3']

def model(nrecords=2, nkeys=3, q=2, steps=10, rollback_mutant=False):
    s=Solver()
    ks=[[Int(f'ks_{t}_{j}') for j in range(nkeys)] for t in range(steps+1)]
    ko=[[Int(f'owner_{t}_{j}') for j in range(nkeys)] for t in range(steps+1)]
    offered=[[Int(f'numoffers_{t}_{j}') for j in range(nkeys)] for t in range(steps+1)]
    decap=[[Int(f'numdecaps_{t}_{j}') for j in range(nkeys)] for t in range(steps+1)]
    gs=[[Int(f'group_{t}_{r}') for r in range(nrecords)] for t in range(steps+1)]
    attempts=[[Int(f'attempts_{t}_{r}') for r in range(nrecords)] for t in range(steps+1)]
    active=[[Int(f'active_{t}_{r}') for r in range(nrecords)] for t in range(steps+1)]
    counter=[[Int(f'counter_{t}_{r}') for r in range(nrecords)] for t in range(steps+1)]
    m1cache=[[Bool(f'm1cache_{t}_{r}') for r in range(nrecords)] for t in range(steps+1)]
    m3cache=[[Bool(f'm3cache_{t}_{r}') for r in range(nrecords)] for t in range(steps+1)]
    confirmed=[[Bool(f'confirmed_{t}_{r}') for r in range(nrecords)] for t in range(steps+1)]
    slow_pending=[[Bool(f'slow_pending_{t}_{r}') for r in range(nrecords)] for t in range(steps+1)]
    act=[Int(f'act_{t}') for t in range(steps)]
    who=[Int(f'who_{t}') for t in range(steps)]
    idx=[Int(f'idx_{t}') for t in range(steps)]
    for j in range(nkeys): s.add(ks[0][j]==FREE,ko[0][j]==-1,offered[0][j]==0,decap[0][j]==0)
    for r in range(nrecords):
        s.add(gs[0][r]==UNUSED,attempts[0][r]==0,active[0][r]==-1,counter[0][r]==0)
        s.add(m1cache[0][r]==False,m3cache[0][r]==False,confirmed[0][r]==False,slow_pending[0][r]==False)
    expr={'offers':[],'finishes':[],'timeouts':[],'confirms':[],'slow_b0':[],'slow_b3':[]}
    for t in range(steps):
        s.add(act[t]>=0,act[t]<=BAD_B3,who[t]>=0,who[t]<nrecords,idx[t]>=0,idx[t]<nkeys)
        for j in range(nkeys): s.add(ks[t][j]>=FREE,ks[t][j]<=ABANDONED,ko[t][j]>=-1,ko[t][j]<nrecords)
        for r in range(nrecords): s.add(gs[t][r]>=UNUSED,gs[t][r]<=EXPIRED,attempts[t][r]>=0,attempts[t][r]<=q,active[t][r]>=-1,active[t][r]<nkeys)
        offer_r=[]; finish_r=[]; timeout_r=[]; confirm_r=[]; slow_r=[]; b3_r=[]
        for r in range(nrecords):
            offer=And(act[t]==OFFER,who[t]==r,gs[t][r]==UNUSED,attempts[t][r]<q,Not(slow_pending[t][r]),Or(*[And(idx[t]==j,ks[t][j]==FREE) for j in range(nkeys)]))
            finish=And(act[t]==FINISH,who[t]==r,gs[t][r]==OFFERED,Or(*[And(active[t][r]==j,ks[t][j]==RESERVED,ko[t][j]==r) for j in range(nkeys)]))
            timeout=And(act[t]==TIMEOUT,who[t]==r,gs[t][r]==OFFERED)
            confirm=And(act[t]==CONFIRM_M3,who[t]==r,gs[t][r]==COMPLETED,m3cache[t][r],Not(confirmed[t][r]))
            slow=And(act[t]==SLOW_B0,who[t]==r,Or(gs[t][r]==OFFERED,gs[t][r]==EXPIRED),Not(slow_pending[t][r]))
            b3=And(act[t]==AUTH_B3,who[t]==r,gs[t][r]==EXPIRED,slow_pending[t][r])
            offer_r.append(offer);finish_r.append(finish);timeout_r.append(timeout);confirm_r.append(confirm);slow_r.append(slow);b3_r.append(b3)
            s.add(gs[t+1][r] == If(offer,OFFERED,If(finish,COMPLETED,If(timeout,If(attempts[t][r]<q,UNUSED,EXPIRED),If(slow,EXPIRED,If(And(act[t]==CRASH,gs[t][r]==OFFERED), If(attempts[t][r]<q,UNUSED,EXPIRED),gs[t][r]))))))
            s.add(attempts[t+1][r] == attempts[t][r]+If(offer,1,0))
            s.add(active[t+1][r] == If(offer,idx[t],If(Or(timeout,slow,And(act[t]==CRASH,gs[t][r]==OFFERED)),-1,active[t][r])))
            s.add(m1cache[t+1][r] == Or(m1cache[t][r],offer))
            s.add(m3cache[t+1][r] == Or(m3cache[t][r],finish))
            s.add(confirmed[t+1][r] == Or(confirmed[t][r],confirm))
            s.add(slow_pending[t+1][r] == If(b3,False,Or(slow_pending[t][r],slow)))
            s.add(counter[t+1][r] == counter[t][r] + If(Or(confirm,b3),1,0))
        for j in range(nkeys):
            alloc=Or(*[And(offer_r[r],idx[t]==j) for r in range(nrecords)])
            finish=Or(*[And(finish_r[r],active[t][r]==j) for r in range(nrecords)])
            abandon=Or(*[And(Or(timeout_r[r],slow_r[r]),active[t][r]==j,ks[t][j]==RESERVED,ko[t][j]==r) for r in range(nrecords)])
            crash_abandon=And(act[t]==CRASH,ks[t][j]==RESERVED)
            rollback=And(rollback_mutant,crash_abandon)
            s.add(ks[t+1][j]==If(rollback,FREE,If(alloc,RESERVED,If(finish,SPENT,If(Or(abandon,crash_abandon),ABANDONED,ks[t][j])))))
            s.add(ko[t+1][j]==If(rollback,-1,If(alloc,who[t],ko[t][j])))
            s.add(offered[t+1][j]==offered[t][j]+If(alloc,1,0))
            s.add(decap[t+1][j]==decap[t][j]+If(finish,1,0))
        expr['offers'].append(offer_r); expr['finishes'].append(finish_r);expr['timeouts'].append(timeout_r);expr['confirms'].append(confirm_r);expr['slow_b0'].append(slow_r);expr['slow_b3'].append(b3_r)
    return s,locals()

def violation(m,kind):
    steps=len(m['act']); nr=len(m['gs'][0]); nk=len(m['ks'][0]);
    if kind=='no_reuse': return Or(*[m['offered'][-1][j]>1 for j in range(nk)])
    if kind=='no_rollback': return Or(*[And(m['ks'][t][j]!=FREE,m['ks'][t+1][j]==FREE) for t in range(steps) for j in range(nk)])
    if kind=='no_multiple_decap': return Or(*[m['decap'][-1][j]>1 for j in range(nk)])
    if kind=='gate_invalid':
        return Or(*[And(Or(m['act'][t]==BAD_M0,m['act'][t]==BAD_M2,m['act'][t]==BAD_B3), Or(*[Or(m['offered'][t+1][j]!=m['offered'][t][j],m['decap'][t+1][j]!=m['decap'][t][j]) for j in range(nk)])) for t in range(steps)])
    if kind=='counter_guard':
        return Or(*[And(m['counter'][t+1][r]>m['counter'][t][r], Not(Or(m['expr']['confirms'][t][r],m['expr']['slow_b3'][t][r]))) for t in range(steps) for r in range(nr)])
    if kind=='no_double_allocation':
        return Or(*[And(m['gs'][t][r]==OFFERED,m['gs'][t][r2]==OFFERED,m['active'][t][r]==m['active'][t][r2]) for t in range(steps+1) for r in range(nr) for r2 in range(r+1,nr)])
    if kind=='offered_then_spent_or_abandoned':
        return Or(*[And(m['ks'][t][j]==SPENT, m['ks'][t+1][j]!=SPENT) for t in range(steps) for j in range(nk)]+[And(m['ks'][t][j]==ABANDONED,m['ks'][t+1][j]!=ABANDONED) for t in range(steps) for j in range(nk)])
    raise ValueError(kind)

def show_trace(sol,m):
    mo=sol.model()
    def v(x):return mo.eval(x).as_long()
    for t in range(len(m['act'])):
        print(f'   t={t} action={ACTIONS[v(m["act"][t])]} record={v(m["who"][t])} key={v(m["idx"][t])} states={[v(x) for x in m["ks"][t]]} groups={[v(x) for x in m["gs"][t]]} attempts={[v(x) for x in m["attempts"][t]]}')
    print('   final key-states:',[v(x) for x in m['ks'][-1]])
    print('   offers:',[v(x) for x in m['offered'][-1]])
    print('   decaps:',[v(x) for x in m['decap'][-1]])

def run():
    cfg=dict(nrecords=2,nkeys=3,q=2,steps=10)
    import json
    results=[]
    for prop in ['no_reuse','no_rollback','no_multiple_decap','gate_invalid','counter_guard','no_double_allocation','offered_then_spent_or_abandoned']:
        s,m=model(**cfg); s.add(violation(m,prop)); res=str(s.check()); print('SAFETY',prop,res,flush=True)
        results.append(dict(kind='safety',property=prop,result=res))
        if res=='sat': show_trace(s,m)
    # Positive witnesses test protocol is not all dead states.
    for label, sequence in [
        ('complete_and_confirm',[OFFER,FINISH,CONFIRM_M3]),
        ('two_records_interleaved',[OFFER,OFFER,FINISH,FINISH]),
        ('retry_after_timeout',[OFFER,TIMEOUT,OFFER,FINISH,CONFIRM_M3]),
        ('crash_abandons_then_retry',[OFFER,CRASH,OFFER,FINISH,CONFIRM_M3]),
        ('crash_preserves_completed_M3',[OFFER,FINISH,CRASH,RETX_M2,CONFIRM_M3]),
        ('slow_recovery_counter',[OFFER,SLOW_B0,AUTH_B3,BAD_B3])
    ]:
        s,m=model(**cfg)
        for t,a in enumerate(sequence): s.add(m['act'][t]==a)
        if label=='two_records_interleaved':
            s.add(m['who'][0]==0,m['who'][1]==1,m['who'][2]==0,m['who'][3]==1, m['idx'][0]!=m['idx'][1])
        else:
            for t in range(len(sequence)): s.add(m['who'][t]==0)
        if label in ('retry_after_timeout','crash_abandons_then_retry'): s.add(m['idx'][0]!=m['idx'][2])
        result=str(s.check()); print('REACHABILITY',label,result,flush=True)
        results.append(dict(kind='reachability',property=label,result=result))
        if result=='sat': show_trace(s,m)
    s,m=model(**cfg,rollback_mutant=True); s.add(violation(m,'no_rollback')); res=str(s.check());print('NEGATIVE_CONTROL','broken_crash_rollback',res,flush=True)
    results.append(dict(kind='negative-control',property='broken_crash_rollback',result=res))
    if res=='sat': show_trace(s,m)
    with open('MobiHeal_Current_Z3_results.json','w') as f:json.dump({'configuration':cfg,'results':results},f,indent=2)
if __name__=='__main__':run()

In [ ]:
Z3_LOG=LOGS/'z3_stdout.txt'
if RUN_FORMAL and importlib.util.find_spec('z3'):
    try:
        zrun=subprocess.run([sys.executable,'mobiheal_z3_current.py'],capture_output=True,text=True,
                             timeout=CONFIG['z3_timeout_s'])
        Z3_LOG.write_text(zrun.stdout+'\nSTDERR:\n'+zrun.stderr)
        fp=Path('MobiHeal_Current_Z3_results.json')
        z3_results=json.loads(fp.read_text())['results'] if zrun.returncode==0 and fp.exists() else []
        z3_status='EXECUTED' if z3_results else 'FAILED_OR_NO_RESULTS'
    except subprocess.TimeoutExpired as e:
        Z3_LOG.write_text(str(e));z3_results=[];z3_status='TIMEOUT'
else:
    z3_results=[];z3_status='NOT_EXECUTED_MISSING_Z3_OR_DISABLED'
z3_df=pd.DataFrame(z3_results,columns=['kind','property','result'])
z3_df.to_csv(ROOT/'table_VIII_Z3_raw.csv',index=False)
if z3_status=='EXECUTED':
    verified=(len(z3_results)==14 and all(r['result']=='unsat' for r in z3_results if r['kind']=='safety')
              and all(r['result']=='sat' for r in z3_results if r['kind']!='safety'))
    print('Bounded Z3 paper-query profile matched:',verified)
else:print('Z3 unavailable or not run:',z3_status)
display(z3_df)

## 11. ProVerif 2.05 — embedded models and actual verifier results
The models below are taken from the uploaded verification notebooks. The explicit-transcript model is the paper’s `aa712aac...` source, the single-challenge privacy model `ac5e73f8...`, and the two-hop same-root model `bc2c3354...`. **PV8 two-challenge identity-swap and linkage are not established in the manuscript**; report actual verifier verdicts (including timeout/no verdict), never mark these as true by expectation. `ProVerif` installation from pinned official source is opt-in. Native protocol tests run regardless of the formal-tool availability.

In [ ]:
%%writefile MobiHeal_Explicit_Transcript.pv
(* EXPERIMENTAL strengthened model: explicit authorization context and full M0,M1,M2
   in authenticated event tuples. Must be parsed and proved by ProVerif before citing.
   Retains original idealized KEM/Merkle constructs and distinct-root replication. *)
(* MobiHeal M0-M3 single-target symbolic AKE abstraction. UNPROVED until ProVerif runs.
   Source: MobiHeal Sections 3.3-3.5; equations (2)-(6).
   Old RSU's credential, derived from the same vehicle root using a DIFFERENT
   target ID, is deliberately leaked to the Dolev-Yao attacker before M0.
   The target and vehicle root are not compromised before the challenge.
   The vehicle root is disclosed after the target completes one handover.
   Public radio channel is fully attacker-controlled.
   One ephemeral target key and one prepared target record are modeled.

   IDEALIZATIONS: ML-KEM is abstract public-key encryption with perfect secrecy;
   KMAC is a free (unbreakable) constructor; XOR/PRF combiner an opaque function;
   certified ML-DSA/Merkle membership represented by a trusted verification
   key and signed leaf. Crash persistence, B0-B3 recovery, multiple attempts,
   revocation/epoch state, and transcript unlinkability are NOT modeled here.
   This is neither a computational PQ proof nor a full-paper proof.
*)

free c: channel.

type skey.
type pkey.
fun pk(skey): pkey.
fun kemenc(bitstring, pkey): bitstring.
reduc forall m:bitstring, sk:skey; kemdec(kemenc(m,pk(sk)),sk) = m.

type signsk.
type signpk.
fun spk(signsk): signpk.
fun sign(bitstring,signsk): bitstring.
reduc forall m:bitstring, sk:signsk; checksign(sign(m,sk),spk(sk)) = m.

fun KMAC(bitstring,bitstring):bitstring.
fun auth(bitstring,bitstring,bitstring,bitstring):bitstring.
fun pseud(bitstring,bitstring,bitstring,bitstring):bitstring.
fun mix(bitstring,bitstring):bitstring.
fun leaf(bitstring,bitstring,bitstring,pkey):bitstring.

fun c_adm(bitstring,bitstring,bitstring,bitstring):bitstring.
fun c_offer(bitstring,bitstring,bitstring,bitstring,pkey):bitstring.
fun c_th2(bitstring,bitstring,bitstring,bitstring,bitstring):bitstring.
fun c_th(bitstring,bitstring,bitstring):bitstring.
fun c_amix(bitstring):bitstring.
fun c_zmix(bitstring):bitstring.
fun c_pre(bitstring):bitstring.
fun c_cv(bitstring):bitstring.
fun c_fv(bitstring,bitstring):bitstring.
fun c_cr(bitstring):bitstring.
fun c_fr(bitstring,bitstring):bitstring.
fun c_app(bitstring):bitstring.
fun c_sid(bitstring,bitstring,bitstring,bitstring,bitstring,bitstring):bitstring.

fun m0(bitstring,bitstring,bitstring,bitstring):bitstring [data].
fun m1(bitstring,bitstring,bitstring,bitstring,bitstring,pkey,bitstring,bitstring,bitstring):bitstring [data].
fun m2(bitstring,bitstring,bitstring,bitstring,bitstring):bitstring [data].
fun m3(bitstring,bitstring):bitstring [data].

fun senc(bitstring,bitstring):bitstring.
reduc forall m:bitstring,k:bitstring; sdec(senc(m,k),k) = m.

free previous_target:bitstring.
free honest_target:bitstring.
free window_id:bitstring.
free epoch_id:bitstring.
free pool_index:bitstring.
free attempt_id:bitstring.

(* A private challenge payload is released only under the derived app key.
   query attacker(payload) tests application-key confidentiality indirectly. *)
free challenge_payload:bitstring [private].

(* These are noninjective correspondences; repeat/parallel session uniqueness
   is not asserted by the single-target model. *)
event VehicleSentM2(bitstring,bitstring,bitstring,bitstring,bitstring,bitstring,bitstring,bitstring).
event RSUCompleted(bitstring,bitstring,bitstring,bitstring,bitstring,bitstring,bitstring,bitstring).
event VehicleAccepted(bitstring,bitstring,bitstring,bitstring,bitstring,bitstring,bitstring,bitstring).
event OldRSULeaked.
event VehicleRootLeaked.

(* Unbounded sessions from distinct, freshly enrolled vehicles. No vehicle root
   leaks before its OWN target completion. Non-injective baseline queries
   are strengthened to INJECTIVE correspondence. This file MUST BE RUN.
   It does not model two handovers under one retained vehicle root. *)
event TargetAttemptStarted(bitstring,bitstring).
query sid:bitstring,k:bitstring,p:bitstring,r:bitstring,w:bitstring,m0v:bitstring,m1v:bitstring,m2v:bitstring;
  inj-event(RSUCompleted(sid,k,p,r,w,m0v,m1v,m2v)) ==> inj-event(VehicleSentM2(sid,k,p,r,w,m0v,m1v,m2v)).
query sid:bitstring,k:bitstring,p:bitstring,r:bitstring,w:bitstring,m0v:bitstring,m1v:bitstring,m2v:bitstring;
  inj-event(VehicleAccepted(sid,k,p,r,w,m0v,m1v,m2v)) ==> inj-event(RSUCompleted(sid,k,p,r,w,m0v,m1v,m2v)).
query attacker(challenge_payload).
query sid:bitstring,k:bitstring,p:bitstring,r:bitstring,w:bitstring,m0v:bitstring,m1v:bitstring,m2v:bitstring; event(RSUCompleted(sid,k,p,r,w,m0v,m1v,m2v)).
query sid:bitstring,k:bitstring,p:bitstring,r:bitstring,w:bitstring,m0v:bitstring,m1v:bitstring,m2v:bitstring; event(VehicleAccepted(sid,k,p,r,w,m0v,m1v,m2v)).

let Vehicle(A:bitstring, P:bitstring, R:bitstring, W:bitstring, trusted_sign_pk:signpk) =
  new n0:bitstring;
  let alpha0 = KMAC(A,c_adm(R,W,P,n0)) in
  let M0 = m0(P,W,n0,alpha0) in
  out(c,M0);
  in(c,received1:bitstring);
  let m1(=R, e:bitstring, =W, j:bitstring, nr:bitstring,
         offered_pk:pkey, offered_leaf:bitstring,
         offered_sig:bitstring, offered_tag:bitstring) = received1 in
  if offered_leaf = leaf(R,e,j,offered_pk) then
    let signed_leaf = checksign(offered_sig,trusted_sign_pk) in
    if signed_leaf = offered_leaf then
      if offered_tag = KMAC(A,c_offer(M0,e,j,nr,offered_pk)) then
        new nv:bitstring;
        new z:bitstring;
        let ct = kemenc(z,offered_pk) in
        let h2 = c_th2(M0,received1,P,nv,ct) in
        let Y_A = KMAC(A,c_amix(h2)) in
        let Y_Z = KMAC(z,c_zmix(h2)) in
        let Kstar = mix(Y_A,Y_Z) in
        let alpha2 = KMAC(A,c_pre(h2)) in
        let tauV = KMAC(KMAC(Kstar,c_cv(h2)),c_fv(h2,alpha2)) in
        let M2 = m2(P,nv,ct,alpha2,tauV) in
        let h = c_th(M0,received1,M2) in
        let sid = c_sid(R,e,W,j,P,h) in
        let Kapp = KMAC(Kstar,c_app(h)) in
        let expected_tauR = KMAC(KMAC(Kstar,c_cr(h)),c_fr(h,sid)) in
        event VehicleSentM2(sid,Kapp,P,R,W,M0,received1,M2);
        out(c,M2);
        in(c,received3:bitstring);
        let m3(=sid,actual_tauR:bitstring) = received3 in
        if actual_tauR = expected_tauR then
          event VehicleAccepted(sid,Kapp,P,R,W,M0,received1,M2);
          0.

let Target(A:bitstring,P:bitstring,R:bitstring,W:bitstring,
           e:bitstring,j:bitstring,kem_sk:skey,sign_sk:signsk,done:channel) =
  in(c,received0:bitstring);
  let m0(=P,=W,n0:bitstring,actual_alpha0:bitstring) = received0 in
  if actual_alpha0 = KMAC(A,c_adm(R,W,P,n0)) then
    new nr:bitstring;
    let public_key = pk(kem_sk) in
    let certified_leaf = leaf(R,e,j,public_key) in
    let signed_leaf = sign(certified_leaf,sign_sk) in
    let beta = KMAC(A,c_offer(received0,e,j,nr,public_key)) in
    let M1 = m1(R,e,W,j,nr,public_key,certified_leaf,signed_leaf,beta) in
    out(c,M1);
    in(c,received2:bitstring);
    let m2(=P,nv:bitstring,ct:bitstring,actual_alpha2:bitstring,
           actual_tauV:bitstring) = received2 in
    let h2 = c_th2(received0,M1,P,nv,ct) in
    if actual_alpha2 = KMAC(A,c_pre(h2)) then
      let z = kemdec(ct,kem_sk) in
      let Y_A = KMAC(A,c_amix(h2)) in
      let Y_Z = KMAC(z,c_zmix(h2)) in
      let Kstar = mix(Y_A,Y_Z) in
      let expected_tauV = KMAC(KMAC(Kstar,c_cv(h2)),c_fv(h2,actual_alpha2)) in
      if actual_tauV = expected_tauV then
        let h = c_th(received0,M1,received2) in
        let sid = c_sid(R,e,W,j,P,h) in
        let Kapp = KMAC(Kstar,c_app(h)) in
        let tauR = KMAC(KMAC(Kstar,c_cr(h)),c_fr(h,sid)) in
        event RSUCompleted(sid,Kapp,P,R,W,received0,M1,received2);
        out(c,senc(challenge_payload,Kapp));
        out(c,m3(sid,tauR));
        out(done,sid);
        0.

(* Replicated VEHICLE registrations: sessions may interleave arbitrarily, but
   root and prepared target credential are independent per registration. *)
process
  !(
    new root:bitstring;
    new counter:bitstring;
    new kem_sk:skey;
    new sign_sk:signsk;
    new done:channel;
    let P = pseud(root,honest_target,counter,attempt_id) in
    let A = auth(root,honest_target,counter,attempt_id) in
    let Aold = auth(root,previous_target,counter,attempt_id) in
    event OldRSULeaked;
    out(c,Aold);
    out(c,spk(sign_sk));
    out(c,pk(kem_sk));
    (Vehicle(A,P,honest_target,window_id,spk(sign_sk))
     | Target(A,P,honest_target,window_id,epoch_id,pool_index,kem_sk,sign_sk,done)
     | (in(done,ack:bitstring); event VehicleRootLeaked; out(c,root)) )
   )

In [ ]:
%%writefile MobiHeal_Single_Challenge_Privacy.pv
(* ONE matched-context left/right privacy experiment for MobiHeal M0-M3.
   Both eligible candidate roots remain confidential; both old-RSU credentials
   are exposed, but their MSC mappings and TARGET records are not.
   All public timing, target, epoch, window, schedule are identical.
   Symbolic constructors, one challenge only, no metadata/location.
   ProVerif 'Observational equivalence is true' is required to claim proof.
   File is UNRUN and MAY yield cannot be proved (not necessarily an attack).
*)
(* MobiHeal M0-M3 single-target symbolic AKE abstraction. UNPROVED until ProVerif runs.
   Source: MobiHeal Sections 3.3-3.5; equations (2)-(6).
   Old RSU's credential, derived from the same vehicle root using a DIFFERENT
   target ID, is deliberately leaked to the Dolev-Yao attacker before M0.
   The target and vehicle root are not compromised before the challenge.
   The vehicle root is disclosed after the target completes one handover.
   Public radio channel is fully attacker-controlled.
   One ephemeral target key and one prepared target record are modeled.

   IDEALIZATIONS: ML-KEM is abstract public-key encryption with perfect secrecy;
   KMAC is a free (unbreakable) constructor; XOR/PRF combiner an opaque function;
   certified ML-DSA/Merkle membership represented by a trusted verification
   key and signed leaf. Crash persistence, B0-B3 recovery, multiple attempts,
   revocation/epoch state, and transcript unlinkability are NOT modeled here.
   This is neither a computational PQ proof nor a full-paper proof.
*)

free c: channel.

type skey.
type pkey.
fun pk(skey): pkey.
fun kemenc(bitstring, pkey): bitstring.
reduc forall m:bitstring, sk:skey; kemdec(kemenc(m,pk(sk)),sk) = m.

type signsk.
type signpk.
fun spk(signsk): signpk.
fun sign(bitstring,signsk): bitstring.
reduc forall m:bitstring, sk:signsk; checksign(sign(m,sk),spk(sk)) = m.

fun KMAC(bitstring,bitstring):bitstring.
fun auth(bitstring,bitstring,bitstring,bitstring):bitstring.
fun pseud(bitstring,bitstring,bitstring,bitstring):bitstring.
fun mix(bitstring,bitstring):bitstring.
fun leaf(bitstring,bitstring,bitstring,pkey):bitstring.

fun c_adm(bitstring,bitstring,bitstring,bitstring):bitstring.
fun c_offer(bitstring,bitstring,bitstring,bitstring,pkey):bitstring.
fun c_th2(bitstring,bitstring,bitstring,bitstring,bitstring):bitstring.
fun c_th(bitstring,bitstring,bitstring):bitstring.
fun c_amix(bitstring):bitstring.
fun c_zmix(bitstring):bitstring.
fun c_pre(bitstring):bitstring.
fun c_cv(bitstring):bitstring.
fun c_fv(bitstring,bitstring):bitstring.
fun c_cr(bitstring):bitstring.
fun c_fr(bitstring,bitstring):bitstring.
fun c_app(bitstring):bitstring.
fun c_sid(bitstring,bitstring,bitstring,bitstring,bitstring,bitstring):bitstring.

fun m0(bitstring,bitstring,bitstring,bitstring):bitstring [data].
fun m1(bitstring,bitstring,bitstring,bitstring,bitstring,pkey,bitstring,bitstring,bitstring):bitstring [data].
fun m2(bitstring,bitstring,bitstring,bitstring,bitstring):bitstring [data].
fun m3(bitstring,bitstring):bitstring [data].

fun senc(bitstring,bitstring):bitstring.
reduc forall m:bitstring,k:bitstring; sdec(senc(m,k),k) = m.

free previous_target:bitstring.
free honest_target:bitstring.
free window_id:bitstring.
free epoch_id:bitstring.
free pool_index:bitstring.
free attempt_id:bitstring.

(* A private challenge payload is released only under the derived app key.
   query attacker(payload) tests application-key confidentiality indirectly. *)
free challenge_payload:bitstring [private].

(* These are noninjective correspondences; repeat/parallel session uniqueness
   is not asserted by the single-target model. *)
event VehicleSentM2(bitstring,bitstring).
event RSUCompleted(bitstring,bitstring).
event VehicleAccepted(bitstring,bitstring).
event OldRSULeaked.
event VehicleRootLeaked.

(* No reachability queries: ProVerif analyzes the choice biprocess. *)

let Vehicle(A:bitstring, P:bitstring, R:bitstring, W:bitstring, trusted_sign_pk:signpk) =
  new n0:bitstring;
  let alpha0 = KMAC(A,c_adm(R,W,P,n0)) in
  let M0 = m0(P,W,n0,alpha0) in
  out(c,M0);
  in(c,received1:bitstring);
  let m1(=R, e:bitstring, =W, j:bitstring, nr:bitstring,
         offered_pk:pkey, offered_leaf:bitstring,
         offered_sig:bitstring, offered_tag:bitstring) = received1 in
  if offered_leaf = leaf(R,e,j,offered_pk) then
    let signed_leaf = checksign(offered_sig,trusted_sign_pk) in
    if signed_leaf = offered_leaf then
      if offered_tag = KMAC(A,c_offer(M0,e,j,nr,offered_pk)) then
        new nv:bitstring;
        new z:bitstring;
        let ct = kemenc(z,offered_pk) in
        let h2 = c_th2(M0,received1,P,nv,ct) in
        let Y_A = KMAC(A,c_amix(h2)) in
        let Y_Z = KMAC(z,c_zmix(h2)) in
        let Kstar = mix(Y_A,Y_Z) in
        let alpha2 = KMAC(A,c_pre(h2)) in
        let tauV = KMAC(KMAC(Kstar,c_cv(h2)),c_fv(h2,alpha2)) in
        let M2 = m2(P,nv,ct,alpha2,tauV) in
        let h = c_th(M0,received1,M2) in
        let sid = c_sid(R,e,W,j,P,h) in
        let Kapp = KMAC(Kstar,c_app(h)) in
        let expected_tauR = KMAC(KMAC(Kstar,c_cr(h)),c_fr(h,sid)) in
        event VehicleSentM2(sid,Kapp);
        out(c,M2);
        in(c,received3:bitstring);
        let m3(=sid,actual_tauR:bitstring) = received3 in
        if actual_tauR = expected_tauR then
          event VehicleAccepted(sid,Kapp);
          0.

let Target(A:bitstring,P:bitstring,R:bitstring,W:bitstring,
           e:bitstring,j:bitstring,kem_sk:skey,sign_sk:signsk,done:channel) =
  in(c,received0:bitstring);
  let m0(=P,=W,n0:bitstring,actual_alpha0:bitstring) = received0 in
  if actual_alpha0 = KMAC(A,c_adm(R,W,P,n0)) then
    new nr:bitstring;
    let public_key = pk(kem_sk) in
    let certified_leaf = leaf(R,e,j,public_key) in
    let signed_leaf = sign(certified_leaf,sign_sk) in
    let beta = KMAC(A,c_offer(received0,e,j,nr,public_key)) in
    let M1 = m1(R,e,W,j,nr,public_key,certified_leaf,signed_leaf,beta) in
    out(c,M1);
    in(c,received2:bitstring);
    let m2(=P,nv:bitstring,ct:bitstring,actual_alpha2:bitstring,
           actual_tauV:bitstring) = received2 in
    let h2 = c_th2(received0,M1,P,nv,ct) in
    if actual_alpha2 = KMAC(A,c_pre(h2)) then
      let z = kemdec(ct,kem_sk) in
      let Y_A = KMAC(A,c_amix(h2)) in
      let Y_Z = KMAC(z,c_zmix(h2)) in
      let Kstar = mix(Y_A,Y_Z) in
      let expected_tauV = KMAC(KMAC(Kstar,c_cv(h2)),c_fv(h2,actual_alpha2)) in
      if actual_tauV = expected_tauV then
        let h = c_th(received0,M1,received2) in
        let sid = c_sid(R,e,W,j,P,h) in
        let Kapp = KMAC(Kstar,c_app(h)) in
        let tauR = KMAC(KMAC(Kstar,c_cr(h)),c_fr(h,sid)) in
        event RSUCompleted(sid,Kapp);
        out(c,senc(challenge_payload,Kapp));
        out(c,m3(sid,tauR));
        out(done,sid);
        0.

process
  new root0:bitstring;
  new root1:bitstring;
  new counter:bitstring;
  new kem_sk:skey;
  new sign_sk:signsk;
  new done:channel;
  (* The adversary obtains history credentials for BOTH eligible candidates. *)
  let old0 = auth(root0,previous_target,counter,attempt_id) in
  let old1 = auth(root1,previous_target,counter,attempt_id) in
  out(c,old0);
  out(c,old1);
  let root = choice[root0,root1] in
  let P = pseud(root,honest_target,counter,attempt_id) in
  let A = auth(root,honest_target,counter,attempt_id) in
  out(c,spk(sign_sk));
  out(c,pk(kem_sk));
  (Vehicle(A,P,honest_target,window_id,spk(sign_sk))
   | Target(A,P,honest_target,window_id,epoch_id,pool_index,kem_sk,sign_sk,done))

In [ ]:
%%writefile MobiHeal_Target_Credential_Leak_Mutant.pv
(* MobiHeal M0-M3 single-target symbolic AKE abstraction. UNPROVED until ProVerif runs.
   Source: MobiHeal Sections 3.3-3.5; equations (2)-(6).
   Old RSU's credential, derived from the same vehicle root using a DIFFERENT
   target ID, is deliberately leaked to the Dolev-Yao attacker before M0.
   The target and vehicle root are not compromised before the challenge.
   The vehicle root is disclosed after the target completes one handover.
   Public radio channel is fully attacker-controlled.
   One ephemeral target key and one prepared target record are modeled.

   IDEALIZATIONS: ML-KEM is abstract public-key encryption with perfect secrecy;
   KMAC is a free (unbreakable) constructor; XOR/PRF combiner an opaque function;
   certified ML-DSA/Merkle membership represented by a trusted verification
   key and signed leaf. Crash persistence, B0-B3 recovery, multiple attempts,
   revocation/epoch state, and transcript unlinkability are NOT modeled here.
   This is neither a computational PQ proof nor a full-paper proof.
*)

free c: channel.

type skey.
type pkey.
fun pk(skey): pkey.
fun kemenc(bitstring, pkey): bitstring.
reduc forall m:bitstring, sk:skey; kemdec(kemenc(m,pk(sk)),sk) = m.

type signsk.
type signpk.
fun spk(signsk): signpk.
fun sign(bitstring,signsk): bitstring.
reduc forall m:bitstring, sk:signsk; checksign(sign(m,sk),spk(sk)) = m.

fun KMAC(bitstring,bitstring):bitstring.
fun auth(bitstring,bitstring,bitstring,bitstring):bitstring.
fun pseud(bitstring,bitstring,bitstring,bitstring):bitstring.
fun mix(bitstring,bitstring):bitstring.
fun leaf(bitstring,bitstring,bitstring,pkey):bitstring.

fun c_adm(bitstring,bitstring,bitstring,bitstring):bitstring.
fun c_offer(bitstring,bitstring,bitstring,bitstring,pkey):bitstring.
fun c_th2(bitstring,bitstring,bitstring,bitstring,bitstring):bitstring.
fun c_th(bitstring,bitstring,bitstring):bitstring.
fun c_amix(bitstring):bitstring.
fun c_zmix(bitstring):bitstring.
fun c_pre(bitstring):bitstring.
fun c_cv(bitstring):bitstring.
fun c_fv(bitstring,bitstring):bitstring.
fun c_cr(bitstring):bitstring.
fun c_fr(bitstring,bitstring):bitstring.
fun c_app(bitstring):bitstring.
fun c_sid(bitstring,bitstring,bitstring,bitstring,bitstring,bitstring):bitstring.

fun m0(bitstring,bitstring,bitstring,bitstring):bitstring [data].
fun m1(bitstring,bitstring,bitstring,bitstring,bitstring,pkey,bitstring,bitstring,bitstring):bitstring [data].
fun m2(bitstring,bitstring,bitstring,bitstring,bitstring):bitstring [data].
fun m3(bitstring,bitstring):bitstring [data].

fun senc(bitstring,bitstring):bitstring.
reduc forall m:bitstring,k:bitstring; sdec(senc(m,k),k) = m.

free previous_target:bitstring.
free honest_target:bitstring.
free window_id:bitstring.
free epoch_id:bitstring.
free pool_index:bitstring.
free attempt_id:bitstring.

(* A private challenge payload is released only under the derived app key.
   query attacker(payload) tests application-key confidentiality indirectly. *)
free challenge_payload:bitstring [private].

(* These are noninjective correspondences; repeat/parallel session uniqueness
   is not asserted by the single-target model. *)
event VehicleSentM2(bitstring,bitstring).
event RSUCompleted(bitstring,bitstring).
event VehicleAccepted(bitstring,bitstring).
event OldRSULeaked.
event VehicleRootLeaked.

(* Sanity checks: ProVerif prints "not event(...) is false" when
   a matching event IS reachable (and "true" if dead/vacuous). *)
query sid:bitstring,k:bitstring; event(RSUCompleted(sid,k)).
query sid:bitstring,k:bitstring; event(VehicleAccepted(sid,k)).
query attacker(challenge_payload).
query sid:bitstring,k:bitstring;
  event(RSUCompleted(sid,k)) ==> event(VehicleSentM2(sid,k)).
query sid:bitstring,k:bitstring;
  event(VehicleAccepted(sid,k)) ==> event(RSUCompleted(sid,k)).

let Vehicle(A:bitstring, P:bitstring, R:bitstring, W:bitstring, trusted_sign_pk:signpk) =
  new n0:bitstring;
  let alpha0 = KMAC(A,c_adm(R,W,P,n0)) in
  let M0 = m0(P,W,n0,alpha0) in
  out(c,M0);
  in(c,received1:bitstring);
  let m1(=R, e:bitstring, =W, j:bitstring, nr:bitstring,
         offered_pk:pkey, offered_leaf:bitstring,
         offered_sig:bitstring, offered_tag:bitstring) = received1 in
  if offered_leaf = leaf(R,e,j,offered_pk) then
    let signed_leaf = checksign(offered_sig,trusted_sign_pk) in
    if signed_leaf = offered_leaf then
      if offered_tag = KMAC(A,c_offer(M0,e,j,nr,offered_pk)) then
        new nv:bitstring;
        new z:bitstring;
        let ct = kemenc(z,offered_pk) in
        let h2 = c_th2(M0,received1,P,nv,ct) in
        let Y_A = KMAC(A,c_amix(h2)) in
        let Y_Z = KMAC(z,c_zmix(h2)) in
        let Kstar = mix(Y_A,Y_Z) in
        let alpha2 = KMAC(A,c_pre(h2)) in
        let tauV = KMAC(KMAC(Kstar,c_cv(h2)),c_fv(h2,alpha2)) in
        let M2 = m2(P,nv,ct,alpha2,tauV) in
        let h = c_th(M0,received1,M2) in
        let sid = c_sid(R,e,W,j,P,h) in
        let Kapp = KMAC(Kstar,c_app(h)) in
        let expected_tauR = KMAC(KMAC(Kstar,c_cr(h)),c_fr(h,sid)) in
        event VehicleSentM2(sid,Kapp);
        out(c,M2);
        in(c,received3:bitstring);
        let m3(=sid,actual_tauR:bitstring) = received3 in
        if actual_tauR = expected_tauR then
          event VehicleAccepted(sid,Kapp);
          0.

let Target(A:bitstring,P:bitstring,R:bitstring,W:bitstring,
           e:bitstring,j:bitstring,kem_sk:skey,sign_sk:signsk,done:channel) =
  in(c,received0:bitstring);
  let m0(=P,=W,n0:bitstring,actual_alpha0:bitstring) = received0 in
  if actual_alpha0 = KMAC(A,c_adm(R,W,P,n0)) then
    new nr:bitstring;
    let public_key = pk(kem_sk) in
    let certified_leaf = leaf(R,e,j,public_key) in
    let signed_leaf = sign(certified_leaf,sign_sk) in
    let beta = KMAC(A,c_offer(received0,e,j,nr,public_key)) in
    let M1 = m1(R,e,W,j,nr,public_key,certified_leaf,signed_leaf,beta) in
    out(c,M1);
    in(c,received2:bitstring);
    let m2(=P,nv:bitstring,ct:bitstring,actual_alpha2:bitstring,
           actual_tauV:bitstring) = received2 in
    let h2 = c_th2(received0,M1,P,nv,ct) in
    if actual_alpha2 = KMAC(A,c_pre(h2)) then
      let z = kemdec(ct,kem_sk) in
      let Y_A = KMAC(A,c_amix(h2)) in
      let Y_Z = KMAC(z,c_zmix(h2)) in
      let Kstar = mix(Y_A,Y_Z) in
      let expected_tauV = KMAC(KMAC(Kstar,c_cv(h2)),c_fv(h2,actual_alpha2)) in
      if actual_tauV = expected_tauV then
        let h = c_th(received0,M1,received2) in
        let sid = c_sid(R,e,W,j,P,h) in
        let Kapp = KMAC(Kstar,c_app(h)) in
        let tauR = KMAC(KMAC(Kstar,c_cr(h)),c_fr(h,sid)) in
        event RSUCompleted(sid,Kapp);
        out(c,senc(challenge_payload,Kapp));
        out(c,m3(sid,tauR));
        out(done,sid);
        0.

process
  new root:bitstring;
  new counter:bitstring;
  new kem_sk:skey;
  new sign_sk:signsk;
  new done:channel;
  let P = pseud(root,honest_target,counter,attempt_id) in
  let A = auth(root,honest_target,counter,attempt_id) in
  let Aold = A in  (* INTENTIONALLY BROKEN: old-RSU compromise reveals target A *)
  event OldRSULeaked;
  out(c,Aold);
  out(c,spk(sign_sk));
  out(c,pk(kem_sk));
  (Vehicle(A,P,honest_target,window_id,spk(sign_sk))
   | Target(A,P,honest_target,window_id,epoch_id,pool_index,kem_sk,sign_sk,done)
   | (in(done,ack:bitstring); event VehicleRootLeaked; out(c,root)) )

In [ ]:
%%writefile MobiHeal_Replicated_AKE.pv
(* MobiHeal M0-M3 single-target symbolic AKE abstraction. UNPROVED until ProVerif runs.
   Source: MobiHeal Sections 3.3-3.5; equations (2)-(6).
   Old RSU's credential, derived from the same vehicle root using a DIFFERENT
   target ID, is deliberately leaked to the Dolev-Yao attacker before M0.
   The target and vehicle root are not compromised before the challenge.
   The vehicle root is disclosed after the target completes one handover.
   Public radio channel is fully attacker-controlled.
   One ephemeral target key and one prepared target record are modeled.

   IDEALIZATIONS: ML-KEM is abstract public-key encryption with perfect secrecy;
   KMAC is a free (unbreakable) constructor; XOR/PRF combiner an opaque function;
   certified ML-DSA/Merkle membership represented by a trusted verification
   key and signed leaf. Crash persistence, B0-B3 recovery, multiple attempts,
   revocation/epoch state, and transcript unlinkability are NOT modeled here.
   This is neither a computational PQ proof nor a full-paper proof.
*)

free c: channel.

type skey.
type pkey.
fun pk(skey): pkey.
fun kemenc(bitstring, pkey): bitstring.
reduc forall m:bitstring, sk:skey; kemdec(kemenc(m,pk(sk)),sk) = m.

type signsk.
type signpk.
fun spk(signsk): signpk.
fun sign(bitstring,signsk): bitstring.
reduc forall m:bitstring, sk:signsk; checksign(sign(m,sk),spk(sk)) = m.

fun KMAC(bitstring,bitstring):bitstring.
fun auth(bitstring,bitstring,bitstring,bitstring):bitstring.
fun pseud(bitstring,bitstring,bitstring,bitstring):bitstring.
fun mix(bitstring,bitstring):bitstring.
fun leaf(bitstring,bitstring,bitstring,pkey):bitstring.

fun c_adm(bitstring,bitstring,bitstring,bitstring):bitstring.
fun c_offer(bitstring,bitstring,bitstring,bitstring,pkey):bitstring.
fun c_th2(bitstring,bitstring,bitstring,bitstring,bitstring):bitstring.
fun c_th(bitstring,bitstring,bitstring):bitstring.
fun c_amix(bitstring):bitstring.
fun c_zmix(bitstring):bitstring.
fun c_pre(bitstring):bitstring.
fun c_cv(bitstring):bitstring.
fun c_fv(bitstring,bitstring):bitstring.
fun c_cr(bitstring):bitstring.
fun c_fr(bitstring,bitstring):bitstring.
fun c_app(bitstring):bitstring.
fun c_sid(bitstring,bitstring,bitstring,bitstring,bitstring,bitstring):bitstring.

fun m0(bitstring,bitstring,bitstring,bitstring):bitstring [data].
fun m1(bitstring,bitstring,bitstring,bitstring,bitstring,pkey,bitstring,bitstring,bitstring):bitstring [data].
fun m2(bitstring,bitstring,bitstring,bitstring,bitstring):bitstring [data].
fun m3(bitstring,bitstring):bitstring [data].

fun senc(bitstring,bitstring):bitstring.
reduc forall m:bitstring,k:bitstring; sdec(senc(m,k),k) = m.

free previous_target:bitstring.
free honest_target:bitstring.
free window_id:bitstring.
free epoch_id:bitstring.
free pool_index:bitstring.
free attempt_id:bitstring.

(* A private challenge payload is released only under the derived app key.
   query attacker(payload) tests application-key confidentiality indirectly. *)
free challenge_payload:bitstring [private].

(* These are noninjective correspondences; repeat/parallel session uniqueness
   is not asserted by the single-target model. *)
event VehicleSentM2(bitstring,bitstring).
event RSUCompleted(bitstring,bitstring).
event VehicleAccepted(bitstring,bitstring).
event OldRSULeaked.
event VehicleRootLeaked.

(* Unbounded sessions from distinct, freshly enrolled vehicles. No vehicle root
   leaks before its OWN target completion. Non-injective baseline queries
   are strengthened to INJECTIVE correspondence. This file MUST BE RUN.
   It does not model two handovers under one retained vehicle root. *)
event TargetAttemptStarted(bitstring,bitstring).
query sid:bitstring,k:bitstring;
  inj-event(RSUCompleted(sid,k)) ==> inj-event(VehicleSentM2(sid,k)).
query sid:bitstring,k:bitstring;
  inj-event(VehicleAccepted(sid,k)) ==> inj-event(RSUCompleted(sid,k)).
query attacker(challenge_payload).
query sid:bitstring,k:bitstring; event(RSUCompleted(sid,k)).
query sid:bitstring,k:bitstring; event(VehicleAccepted(sid,k)).

let Vehicle(A:bitstring, P:bitstring, R:bitstring, W:bitstring, trusted_sign_pk:signpk) =
  new n0:bitstring;
  let alpha0 = KMAC(A,c_adm(R,W,P,n0)) in
  let M0 = m0(P,W,n0,alpha0) in
  out(c,M0);
  in(c,received1:bitstring);
  let m1(=R, e:bitstring, =W, j:bitstring, nr:bitstring,
         offered_pk:pkey, offered_leaf:bitstring,
         offered_sig:bitstring, offered_tag:bitstring) = received1 in
  if offered_leaf = leaf(R,e,j,offered_pk) then
    let signed_leaf = checksign(offered_sig,trusted_sign_pk) in
    if signed_leaf = offered_leaf then
      if offered_tag = KMAC(A,c_offer(M0,e,j,nr,offered_pk)) then
        new nv:bitstring;
        new z:bitstring;
        let ct = kemenc(z,offered_pk) in
        let h2 = c_th2(M0,received1,P,nv,ct) in
        let Y_A = KMAC(A,c_amix(h2)) in
        let Y_Z = KMAC(z,c_zmix(h2)) in
        let Kstar = mix(Y_A,Y_Z) in
        let alpha2 = KMAC(A,c_pre(h2)) in
        let tauV = KMAC(KMAC(Kstar,c_cv(h2)),c_fv(h2,alpha2)) in
        let M2 = m2(P,nv,ct,alpha2,tauV) in
        let h = c_th(M0,received1,M2) in
        let sid = c_sid(R,e,W,j,P,h) in
        let Kapp = KMAC(Kstar,c_app(h)) in
        let expected_tauR = KMAC(KMAC(Kstar,c_cr(h)),c_fr(h,sid)) in
        event VehicleSentM2(sid,Kapp);
        out(c,M2);
        in(c,received3:bitstring);
        let m3(=sid,actual_tauR:bitstring) = received3 in
        if actual_tauR = expected_tauR then
          event VehicleAccepted(sid,Kapp);
          0.

let Target(A:bitstring,P:bitstring,R:bitstring,W:bitstring,
           e:bitstring,j:bitstring,kem_sk:skey,sign_sk:signsk,done:channel) =
  in(c,received0:bitstring);
  let m0(=P,=W,n0:bitstring,actual_alpha0:bitstring) = received0 in
  if actual_alpha0 = KMAC(A,c_adm(R,W,P,n0)) then
    new nr:bitstring;
    let public_key = pk(kem_sk) in
    let certified_leaf = leaf(R,e,j,public_key) in
    let signed_leaf = sign(certified_leaf,sign_sk) in
    let beta = KMAC(A,c_offer(received0,e,j,nr,public_key)) in
    let M1 = m1(R,e,W,j,nr,public_key,certified_leaf,signed_leaf,beta) in
    out(c,M1);
    in(c,received2:bitstring);
    let m2(=P,nv:bitstring,ct:bitstring,actual_alpha2:bitstring,
           actual_tauV:bitstring) = received2 in
    let h2 = c_th2(received0,M1,P,nv,ct) in
    if actual_alpha2 = KMAC(A,c_pre(h2)) then
      let z = kemdec(ct,kem_sk) in
      let Y_A = KMAC(A,c_amix(h2)) in
      let Y_Z = KMAC(z,c_zmix(h2)) in
      let Kstar = mix(Y_A,Y_Z) in
      let expected_tauV = KMAC(KMAC(Kstar,c_cv(h2)),c_fv(h2,actual_alpha2)) in
      if actual_tauV = expected_tauV then
        let h = c_th(received0,M1,received2) in
        let sid = c_sid(R,e,W,j,P,h) in
        let Kapp = KMAC(Kstar,c_app(h)) in
        let tauR = KMAC(KMAC(Kstar,c_cr(h)),c_fr(h,sid)) in
        event RSUCompleted(sid,Kapp);
        out(c,senc(challenge_payload,Kapp));
        out(c,m3(sid,tauR));
        out(done,sid);
        0.

(* Replicated VEHICLE registrations: sessions may interleave arbitrarily, but
   root and prepared target credential are independent per registration. *)
process
  !(
    new root:bitstring;
    new counter:bitstring;
    new kem_sk:skey;
    new sign_sk:signsk;
    new done:channel;
    let P = pseud(root,honest_target,counter,attempt_id) in
    let A = auth(root,honest_target,counter,attempt_id) in
    let Aold = auth(root,previous_target,counter,attempt_id) in
    event OldRSULeaked;
    out(c,Aold);
    out(c,spk(sign_sk));
    out(c,pk(kem_sk));
    (Vehicle(A,P,honest_target,window_id,spk(sign_sk))
     | Target(A,P,honest_target,window_id,epoch_id,pool_index,kem_sk,sign_sk,done)
     | (in(done,ack:bitstring); event VehicleRootLeaked; out(c,root)) )
   )

In [ ]:
%%writefile MobiHeal_Two_Hops_Same_Root.pv
(* Research extension: exactly two TARGET handovers with ONE retained vehicle root.
  Prior RSU target credentials for both counters are revealed; new-target
  credentials and KEM decapsulation keys remain uncompromised. The first
  VEHICLE acceptance is the sequencing boundary before the second handover.
  KEM, KMAC, ML-DSA/Merkle and state durability are IDEALIZED. This is
  symbolic q=2 evidence, NOT a computational or unbounded-q PCS proof. *)
free c: channel.

type skey.
type pkey.
fun pk(skey): pkey.
fun kemenc(bitstring, pkey): bitstring.
reduc forall m:bitstring, sk:skey; kemdec(kemenc(m,pk(sk)),sk) = m.

type signsk.
type signpk.
fun spk(signsk): signpk.
fun sign(bitstring,signsk): bitstring.
reduc forall m:bitstring, sk:signsk; checksign(sign(m,sk),spk(sk)) = m.

fun KMAC(bitstring,bitstring):bitstring.
fun auth(bitstring,bitstring,bitstring,bitstring):bitstring.
fun pseud(bitstring,bitstring,bitstring,bitstring):bitstring.
fun mix(bitstring,bitstring):bitstring.
fun leaf(bitstring,bitstring,bitstring,pkey):bitstring.

fun c_adm(bitstring,bitstring,bitstring,bitstring):bitstring.
fun c_offer(bitstring,bitstring,bitstring,bitstring,pkey):bitstring.
fun c_th2(bitstring,bitstring,bitstring,bitstring,bitstring):bitstring.
fun c_th(bitstring,bitstring,bitstring):bitstring.
fun c_amix(bitstring):bitstring.
fun c_zmix(bitstring):bitstring.
fun c_pre(bitstring):bitstring.
fun c_cv(bitstring):bitstring.
fun c_fv(bitstring,bitstring):bitstring.
fun c_cr(bitstring):bitstring.
fun c_fr(bitstring,bitstring):bitstring.
fun c_app(bitstring):bitstring.
fun c_sid(bitstring,bitstring,bitstring,bitstring,bitstring,bitstring):bitstring.

fun m0(bitstring,bitstring,bitstring,bitstring):bitstring [data].
fun m1(bitstring,bitstring,bitstring,bitstring,bitstring,pkey,bitstring,bitstring,bitstring):bitstring [data].
fun m2(bitstring,bitstring,bitstring,bitstring,bitstring):bitstring [data].
fun m3(bitstring,bitstring):bitstring [data].

fun senc(bitstring,bitstring):bitstring.
reduc forall m:bitstring,k:bitstring; sdec(senc(m,k),k) = m.

free previous_target:bitstring.
free honest_target:bitstring.
free window_id:bitstring.
free epoch_id:bitstring.
free pool_index:bitstring.
free attempt_id:bitstring.

(* A private challenge payload is released only under the derived app key.
   query attacker(payload) tests application-key confidentiality indirectly. *)
free challenge_payload:bitstring [private].

(* These are noninjective correspondences; repeat/parallel session uniqueness
   is not asserted by the single-target model. *)
event VehicleSentM2(bitstring,bitstring,bitstring,bitstring,bitstring,bitstring,bitstring,bitstring).
event RSUCompleted(bitstring,bitstring,bitstring,bitstring,bitstring,bitstring,bitstring,bitstring).
event VehicleAccepted(bitstring,bitstring,bitstring,bitstring,bitstring,bitstring,bitstring,bitstring).
event TwoVehicleAcceptances.
event OldRSULeaked.
event VehicleRootLeaked.

(* Unbounded sessions from distinct, freshly enrolled vehicles. No vehicle root
   leaks before its OWN target completion. Non-injective baseline queries
   are strengthened to INJECTIVE correspondence. This file MUST BE RUN.
   It does not model two handovers under one retained vehicle root. *)
event TargetAttemptStarted(bitstring,bitstring).
query sid:bitstring,k:bitstring,p:bitstring,r:bitstring,w:bitstring,m0v:bitstring,m1v:bitstring,m2v:bitstring;
  inj-event(RSUCompleted(sid,k,p,r,w,m0v,m1v,m2v)) ==> inj-event(VehicleSentM2(sid,k,p,r,w,m0v,m1v,m2v)).
query sid:bitstring,k:bitstring,p:bitstring,r:bitstring,w:bitstring,m0v:bitstring,m1v:bitstring,m2v:bitstring;
  inj-event(VehicleAccepted(sid,k,p,r,w,m0v,m1v,m2v)) ==> inj-event(RSUCompleted(sid,k,p,r,w,m0v,m1v,m2v)).
query attacker(challenge_payload).
query sid:bitstring,k:bitstring,p:bitstring,r:bitstring,w:bitstring,m0v:bitstring,m1v:bitstring,m2v:bitstring; event(RSUCompleted(sid,k,p,r,w,m0v,m1v,m2v)).
query sid:bitstring,k:bitstring,p:bitstring,r:bitstring,w:bitstring,m0v:bitstring,m1v:bitstring,m2v:bitstring; event(VehicleAccepted(sid,k,p,r,w,m0v,m1v,m2v)).

let Vehicle(A:bitstring, P:bitstring, R:bitstring, W:bitstring, trusted_sign_pk:signpk,vehicle_done:channel) =
  new n0:bitstring;
  let alpha0 = KMAC(A,c_adm(R,W,P,n0)) in
  let M0 = m0(P,W,n0,alpha0) in
  out(c,M0);
  in(c,received1:bitstring);
  let m1(=R, e:bitstring, =W, j:bitstring, nr:bitstring,
         offered_pk:pkey, offered_leaf:bitstring,
         offered_sig:bitstring, offered_tag:bitstring) = received1 in
  if offered_leaf = leaf(R,e,j,offered_pk) then
    let signed_leaf = checksign(offered_sig,trusted_sign_pk) in
    if signed_leaf = offered_leaf then
      if offered_tag = KMAC(A,c_offer(M0,e,j,nr,offered_pk)) then
        new nv:bitstring;
        new z:bitstring;
        let ct = kemenc(z,offered_pk) in
        let h2 = c_th2(M0,received1,P,nv,ct) in
        let Y_A = KMAC(A,c_amix(h2)) in
        let Y_Z = KMAC(z,c_zmix(h2)) in
        let Kstar = mix(Y_A,Y_Z) in
        let alpha2 = KMAC(A,c_pre(h2)) in
        let tauV = KMAC(KMAC(Kstar,c_cv(h2)),c_fv(h2,alpha2)) in
        let M2 = m2(P,nv,ct,alpha2,tauV) in
        let h = c_th(M0,received1,M2) in
        let sid = c_sid(R,e,W,j,P,h) in
        let Kapp = KMAC(Kstar,c_app(h)) in
        let expected_tauR = KMAC(KMAC(Kstar,c_cr(h)),c_fr(h,sid)) in
        event VehicleSentM2(sid,Kapp,P,R,W,M0,received1,M2);
        out(c,M2);
        in(c,received3:bitstring);
        let m3(=sid,actual_tauR:bitstring) = received3 in
        if actual_tauR = expected_tauR then
          event VehicleAccepted(sid,Kapp,P,R,W,M0,received1,M2);
          out(vehicle_done,sid);
          0.

let Target(A:bitstring,P:bitstring,R:bitstring,W:bitstring,
           e:bitstring,j:bitstring,kem_sk:skey,sign_sk:signsk,done:channel) =
  in(c,received0:bitstring);
  let m0(=P,=W,n0:bitstring,actual_alpha0:bitstring) = received0 in
  if actual_alpha0 = KMAC(A,c_adm(R,W,P,n0)) then
    new nr:bitstring;
    let public_key = pk(kem_sk) in
    let certified_leaf = leaf(R,e,j,public_key) in
    let signed_leaf = sign(certified_leaf,sign_sk) in
    let beta = KMAC(A,c_offer(received0,e,j,nr,public_key)) in
    let M1 = m1(R,e,W,j,nr,public_key,certified_leaf,signed_leaf,beta) in
    out(c,M1);
    in(c,received2:bitstring);
    let m2(=P,nv:bitstring,ct:bitstring,actual_alpha2:bitstring,
           actual_tauV:bitstring) = received2 in
    let h2 = c_th2(received0,M1,P,nv,ct) in
    if actual_alpha2 = KMAC(A,c_pre(h2)) then
      let z = kemdec(ct,kem_sk) in
      let Y_A = KMAC(A,c_amix(h2)) in
      let Y_Z = KMAC(z,c_zmix(h2)) in
      let Kstar = mix(Y_A,Y_Z) in
      let expected_tauV = KMAC(KMAC(Kstar,c_cv(h2)),c_fv(h2,actual_alpha2)) in
      if actual_tauV = expected_tauV then
        let h = c_th(received0,M1,received2) in
        let sid = c_sid(R,e,W,j,P,h) in
        let Kapp = KMAC(Kstar,c_app(h)) in
        let tauR = KMAC(KMAC(Kstar,c_cr(h)),c_fr(h,sid)) in
        event RSUCompleted(sid,Kapp,P,R,W,received0,M1,received2);
        out(c,senc(challenge_payload,Kapp));
        out(c,m3(sid,tauR));
        out(done,sid);
        0.

(* Replicated VEHICLE registrations: sessions may interleave arbitrarily, but
   root and prepared target credential are independent per registration. *)
query event(TwoVehicleAcceptances).
process
  new root:bitstring;
  new counter0:bitstring;
  new counter1:bitstring;
  new kem_sk0:skey;
  new kem_sk1:skey;
  new sign_sk:signsk;
  new j0:bitstring;
  new j1:bitstring;
  new veh_done0:channel;
  new veh_done1:channel;
  new done0:channel;
  new done1:channel;
  let A0 = auth(root,honest_target,counter0,attempt_id) in
  let P0 = pseud(root,honest_target,counter0,attempt_id) in
  let A1 = auth(root,honest_target,counter1,attempt_id) in
  let P1 = pseud(root,honest_target,counter1,attempt_id) in
  let OldA0 = auth(root,previous_target,counter0,attempt_id) in
  let OldA1 = auth(root,previous_target,counter1,attempt_id) in
  event OldRSULeaked;
  out(c,OldA0);
  out(c,OldA1);
  out(c,spk(sign_sk));
  out(c,pk(kem_sk0));
  out(c,pk(kem_sk1));
  ( Vehicle(A0,P0,honest_target,window_id,spk(sign_sk),veh_done0)
  | Target(A0,P0,honest_target,window_id,epoch_id,j0,kem_sk0,sign_sk,done0)
  | (in(veh_done0,accepted0:bitstring);
     (Vehicle(A1,P1,honest_target,window_id,spk(sign_sk),veh_done1)
     | Target(A1,P1,honest_target,window_id,epoch_id,j1,kem_sk1,sign_sk,done1)
     | (in(veh_done1,accepted1:bitstring); event TwoVehicleAcceptances; 0))) )

In [ ]:
%%writefile MobiHeal_Two_Challenge_Identity_Swap.pv
(* Research extension: two consecutive matched public-context sessions
   from the SAME HIDDEN ROOT in each world: left world uses root0 for both;
   right world uses root1 for both. Both old-RSU root-specific credential
   histories are disclosed without identity mappings. Symbolic q=2 only.
   Completion sequencing is RSU M3 emission, not vehicle acceptance. *)
free c: channel.

type skey.
type pkey.
fun pk(skey): pkey.
fun kemenc(bitstring, pkey): bitstring.
reduc forall m:bitstring, sk:skey; kemdec(kemenc(m,pk(sk)),sk) = m.

type signsk.
type signpk.
fun spk(signsk): signpk.
fun sign(bitstring,signsk): bitstring.
reduc forall m:bitstring, sk:signsk; checksign(sign(m,sk),spk(sk)) = m.

fun KMAC(bitstring,bitstring):bitstring.
fun auth(bitstring,bitstring,bitstring,bitstring):bitstring.
fun pseud(bitstring,bitstring,bitstring,bitstring):bitstring.
fun mix(bitstring,bitstring):bitstring.
fun leaf(bitstring,bitstring,bitstring,pkey):bitstring.

fun c_adm(bitstring,bitstring,bitstring,bitstring):bitstring.
fun c_offer(bitstring,bitstring,bitstring,bitstring,pkey):bitstring.
fun c_th2(bitstring,bitstring,bitstring,bitstring,bitstring):bitstring.
fun c_th(bitstring,bitstring,bitstring):bitstring.
fun c_amix(bitstring):bitstring.
fun c_zmix(bitstring):bitstring.
fun c_pre(bitstring):bitstring.
fun c_cv(bitstring):bitstring.
fun c_fv(bitstring,bitstring):bitstring.
fun c_cr(bitstring):bitstring.
fun c_fr(bitstring,bitstring):bitstring.
fun c_app(bitstring):bitstring.
fun c_sid(bitstring,bitstring,bitstring,bitstring,bitstring,bitstring):bitstring.

fun m0(bitstring,bitstring,bitstring,bitstring):bitstring [data].
fun m1(bitstring,bitstring,bitstring,bitstring,bitstring,pkey,bitstring,bitstring,bitstring):bitstring [data].
fun m2(bitstring,bitstring,bitstring,bitstring,bitstring):bitstring [data].
fun m3(bitstring,bitstring):bitstring [data].

fun senc(bitstring,bitstring):bitstring.
reduc forall m:bitstring,k:bitstring; sdec(senc(m,k),k) = m.

free previous_target:bitstring.
free honest_target:bitstring.
free window_id:bitstring.
free epoch_id:bitstring.
free pool_index:bitstring.
free attempt_id:bitstring.

(* A private challenge payload is released only under the derived app key.
   query attacker(payload) tests application-key confidentiality indirectly. *)
free challenge_payload:bitstring [private].

(* These are noninjective correspondences; repeat/parallel session uniqueness
   is not asserted by the single-target model. *)
event VehicleSentM2(bitstring,bitstring).
event RSUCompleted(bitstring,bitstring).
event VehicleAccepted(bitstring,bitstring).
event OldRSULeaked.
event VehicleRootLeaked.

(* No reachability queries: ProVerif analyzes the choice biprocess. *)

let Vehicle(A:bitstring, P:bitstring, R:bitstring, W:bitstring, trusted_sign_pk:signpk) =
  new n0:bitstring;
  let alpha0 = KMAC(A,c_adm(R,W,P,n0)) in
  let M0 = m0(P,W,n0,alpha0) in
  out(c,M0);
  in(c,received1:bitstring);
  let m1(=R, e:bitstring, =W, j:bitstring, nr:bitstring,
         offered_pk:pkey, offered_leaf:bitstring,
         offered_sig:bitstring, offered_tag:bitstring) = received1 in
  if offered_leaf = leaf(R,e,j,offered_pk) then
    let signed_leaf = checksign(offered_sig,trusted_sign_pk) in
    if signed_leaf = offered_leaf then
      if offered_tag = KMAC(A,c_offer(M0,e,j,nr,offered_pk)) then
        new nv:bitstring;
        new z:bitstring;
        let ct = kemenc(z,offered_pk) in
        let h2 = c_th2(M0,received1,P,nv,ct) in
        let Y_A = KMAC(A,c_amix(h2)) in
        let Y_Z = KMAC(z,c_zmix(h2)) in
        let Kstar = mix(Y_A,Y_Z) in
        let alpha2 = KMAC(A,c_pre(h2)) in
        let tauV = KMAC(KMAC(Kstar,c_cv(h2)),c_fv(h2,alpha2)) in
        let M2 = m2(P,nv,ct,alpha2,tauV) in
        let h = c_th(M0,received1,M2) in
        let sid = c_sid(R,e,W,j,P,h) in
        let Kapp = KMAC(Kstar,c_app(h)) in
        let expected_tauR = KMAC(KMAC(Kstar,c_cr(h)),c_fr(h,sid)) in
        event VehicleSentM2(sid,Kapp);
        out(c,M2);
        in(c,received3:bitstring);
        let m3(=sid,actual_tauR:bitstring) = received3 in
        if actual_tauR = expected_tauR then
          event VehicleAccepted(sid,Kapp);
          0.

let Target(A:bitstring,P:bitstring,R:bitstring,W:bitstring,
           e:bitstring,j:bitstring,kem_sk:skey,sign_sk:signsk,done:channel) =
  in(c,received0:bitstring);
  let m0(=P,=W,n0:bitstring,actual_alpha0:bitstring) = received0 in
  if actual_alpha0 = KMAC(A,c_adm(R,W,P,n0)) then
    new nr:bitstring;
    let public_key = pk(kem_sk) in
    let certified_leaf = leaf(R,e,j,public_key) in
    let signed_leaf = sign(certified_leaf,sign_sk) in
    let beta = KMAC(A,c_offer(received0,e,j,nr,public_key)) in
    let M1 = m1(R,e,W,j,nr,public_key,certified_leaf,signed_leaf,beta) in
    out(c,M1);
    in(c,received2:bitstring);
    let m2(=P,nv:bitstring,ct:bitstring,actual_alpha2:bitstring,
           actual_tauV:bitstring) = received2 in
    let h2 = c_th2(received0,M1,P,nv,ct) in
    if actual_alpha2 = KMAC(A,c_pre(h2)) then
      let z = kemdec(ct,kem_sk) in
      let Y_A = KMAC(A,c_amix(h2)) in
      let Y_Z = KMAC(z,c_zmix(h2)) in
      let Kstar = mix(Y_A,Y_Z) in
      let expected_tauV = KMAC(KMAC(Kstar,c_cv(h2)),c_fv(h2,actual_alpha2)) in
      if actual_tauV = expected_tauV then
        let h = c_th(received0,M1,received2) in
        let sid = c_sid(R,e,W,j,P,h) in
        let Kapp = KMAC(Kstar,c_app(h)) in
        let tauR = KMAC(KMAC(Kstar,c_cr(h)),c_fr(h,sid)) in
        event RSUCompleted(sid,Kapp);
        out(c,senc(challenge_payload,Kapp));
        out(c,m3(sid,tauR));
        out(done,sid);
        0.

process
  new root0:bitstring;
  new root1:bitstring;
  new counter0:bitstring;
  new counter1:bitstring;
  new kem_sk0:skey;
  new kem_sk1:skey;
  new sign_sk:signsk;
  new j0:bitstring;
  new j1:bitstring;
  new done0:channel;
  new done1:channel;
  let old00 = auth(root0,previous_target,counter0,attempt_id) in
  let old01 = auth(root0,previous_target,counter1,attempt_id) in
  let old10 = auth(root1,previous_target,counter0,attempt_id) in
  let old11 = auth(root1,previous_target,counter1,attempt_id) in
  out(c,old00);out(c,old01);out(c,old10);out(c,old11);
  let selected_root = choice[root0,root1] in
  let A0 = auth(selected_root,honest_target,counter0,attempt_id) in
  let P0 = pseud(selected_root,honest_target,counter0,attempt_id) in
  let A1 = auth(selected_root,honest_target,counter1,attempt_id) in
  let P1 = pseud(selected_root,honest_target,counter1,attempt_id) in
  out(c,spk(sign_sk));out(c,pk(kem_sk0));out(c,pk(kem_sk1));
  (Vehicle(A0,P0,honest_target,window_id,spk(sign_sk))
   | Target(A0,P0,honest_target,window_id,epoch_id,j0,kem_sk0,sign_sk,done0)
   | (in(done0,completed0:bitstring);
       (Vehicle(A1,P1,honest_target,window_id,spk(sign_sk))
       |Target(A1,P1,honest_target,window_id,epoch_id,j1,kem_sk1,sign_sk,done1))))

In [ ]:
%%writefile MobiHeal_Two_Challenge_Linkability.pv
(* Research extension: matched-context TWO-SESSION LINKABILITY game.
   Left world: session0 root0, session1 root0 (same vehicle).
   Right world: session0 root0, session1 root1 (different vehicles).
   Two distinct counters and single-use KEM keys. Target/epoch/window and
   on-air schedule fixed; both candidates' historical old-RSU credentials
   are exposed without target identity mappings. No physical tracking.
   Symbolic finite-q observational equivalence is NOT computational PCS. *)
free c: channel.

type skey.
type pkey.
fun pk(skey): pkey.
fun kemenc(bitstring, pkey): bitstring.
reduc forall m:bitstring, sk:skey; kemdec(kemenc(m,pk(sk)),sk) = m.

type signsk.
type signpk.
fun spk(signsk): signpk.
fun sign(bitstring,signsk): bitstring.
reduc forall m:bitstring, sk:signsk; checksign(sign(m,sk),spk(sk)) = m.

fun KMAC(bitstring,bitstring):bitstring.
fun auth(bitstring,bitstring,bitstring,bitstring):bitstring.
fun pseud(bitstring,bitstring,bitstring,bitstring):bitstring.
fun mix(bitstring,bitstring):bitstring.
fun leaf(bitstring,bitstring,bitstring,pkey):bitstring.

fun c_adm(bitstring,bitstring,bitstring,bitstring):bitstring.
fun c_offer(bitstring,bitstring,bitstring,bitstring,pkey):bitstring.
fun c_th2(bitstring,bitstring,bitstring,bitstring,bitstring):bitstring.
fun c_th(bitstring,bitstring,bitstring):bitstring.
fun c_amix(bitstring):bitstring.
fun c_zmix(bitstring):bitstring.
fun c_pre(bitstring):bitstring.
fun c_cv(bitstring):bitstring.
fun c_fv(bitstring,bitstring):bitstring.
fun c_cr(bitstring):bitstring.
fun c_fr(bitstring,bitstring):bitstring.
fun c_app(bitstring):bitstring.
fun c_sid(bitstring,bitstring,bitstring,bitstring,bitstring,bitstring):bitstring.

fun m0(bitstring,bitstring,bitstring,bitstring):bitstring [data].
fun m1(bitstring,bitstring,bitstring,bitstring,bitstring,pkey,bitstring,bitstring,bitstring):bitstring [data].
fun m2(bitstring,bitstring,bitstring,bitstring,bitstring):bitstring [data].
fun m3(bitstring,bitstring):bitstring [data].

fun senc(bitstring,bitstring):bitstring.
reduc forall m:bitstring,k:bitstring; sdec(senc(m,k),k) = m.

free previous_target:bitstring.
free honest_target:bitstring.
free window_id:bitstring.
free epoch_id:bitstring.
free pool_index:bitstring.
free attempt_id:bitstring.

(* A private challenge payload is released only under the derived app key.
   query attacker(payload) tests application-key confidentiality indirectly. *)
free challenge_payload:bitstring [private].

(* These are noninjective correspondences; repeat/parallel session uniqueness
   is not asserted by the single-target model. *)
event VehicleSentM2(bitstring,bitstring).
event RSUCompleted(bitstring,bitstring).
event VehicleAccepted(bitstring,bitstring).
event OldRSULeaked.
event VehicleRootLeaked.

(* No reachability queries: ProVerif analyzes the choice biprocess. *)

let Vehicle(A:bitstring, P:bitstring, R:bitstring, W:bitstring, trusted_sign_pk:signpk) =
  new n0:bitstring;
  let alpha0 = KMAC(A,c_adm(R,W,P,n0)) in
  let M0 = m0(P,W,n0,alpha0) in
  out(c,M0);
  in(c,received1:bitstring);
  let m1(=R, e:bitstring, =W, j:bitstring, nr:bitstring,
         offered_pk:pkey, offered_leaf:bitstring,
         offered_sig:bitstring, offered_tag:bitstring) = received1 in
  if offered_leaf = leaf(R,e,j,offered_pk) then
    let signed_leaf = checksign(offered_sig,trusted_sign_pk) in
    if signed_leaf = offered_leaf then
      if offered_tag = KMAC(A,c_offer(M0,e,j,nr,offered_pk)) then
        new nv:bitstring;
        new z:bitstring;
        let ct = kemenc(z,offered_pk) in
        let h2 = c_th2(M0,received1,P,nv,ct) in
        let Y_A = KMAC(A,c_amix(h2)) in
        let Y_Z = KMAC(z,c_zmix(h2)) in
        let Kstar = mix(Y_A,Y_Z) in
        let alpha2 = KMAC(A,c_pre(h2)) in
        let tauV = KMAC(KMAC(Kstar,c_cv(h2)),c_fv(h2,alpha2)) in
        let M2 = m2(P,nv,ct,alpha2,tauV) in
        let h = c_th(M0,received1,M2) in
        let sid = c_sid(R,e,W,j,P,h) in
        let Kapp = KMAC(Kstar,c_app(h)) in
        let expected_tauR = KMAC(KMAC(Kstar,c_cr(h)),c_fr(h,sid)) in
        event VehicleSentM2(sid,Kapp);
        out(c,M2);
        in(c,received3:bitstring);
        let m3(=sid,actual_tauR:bitstring) = received3 in
        if actual_tauR = expected_tauR then
          event VehicleAccepted(sid,Kapp);
          0.

let Target(A:bitstring,P:bitstring,R:bitstring,W:bitstring,
           e:bitstring,j:bitstring,kem_sk:skey,sign_sk:signsk,done:channel) =
  in(c,received0:bitstring);
  let m0(=P,=W,n0:bitstring,actual_alpha0:bitstring) = received0 in
  if actual_alpha0 = KMAC(A,c_adm(R,W,P,n0)) then
    new nr:bitstring;
    let public_key = pk(kem_sk) in
    let certified_leaf = leaf(R,e,j,public_key) in
    let signed_leaf = sign(certified_leaf,sign_sk) in
    let beta = KMAC(A,c_offer(received0,e,j,nr,public_key)) in
    let M1 = m1(R,e,W,j,nr,public_key,certified_leaf,signed_leaf,beta) in
    out(c,M1);
    in(c,received2:bitstring);
    let m2(=P,nv:bitstring,ct:bitstring,actual_alpha2:bitstring,
           actual_tauV:bitstring) = received2 in
    let h2 = c_th2(received0,M1,P,nv,ct) in
    if actual_alpha2 = KMAC(A,c_pre(h2)) then
      let z = kemdec(ct,kem_sk) in
      let Y_A = KMAC(A,c_amix(h2)) in
      let Y_Z = KMAC(z,c_zmix(h2)) in
      let Kstar = mix(Y_A,Y_Z) in
      let expected_tauV = KMAC(KMAC(Kstar,c_cv(h2)),c_fv(h2,actual_alpha2)) in
      if actual_tauV = expected_tauV then
        let h = c_th(received0,M1,received2) in
        let sid = c_sid(R,e,W,j,P,h) in
        let Kapp = KMAC(Kstar,c_app(h)) in
        let tauR = KMAC(KMAC(Kstar,c_cr(h)),c_fr(h,sid)) in
        event RSUCompleted(sid,Kapp);
        out(c,senc(challenge_payload,Kapp));
        out(c,m3(sid,tauR));
        out(done,sid);
        0.

process
  new root0:bitstring;
  new root1:bitstring;
  new counter0:bitstring;
  new counter1:bitstring;
  new kem_sk0:skey;
  new kem_sk1:skey;
  new sign_sk:signsk;
  new j0:bitstring;
  new j1:bitstring;
  new done0:channel;
  new done1:channel;
  let old00 = auth(root0,previous_target,counter0,attempt_id) in
  let old01 = auth(root0,previous_target,counter1,attempt_id) in
  let old10 = auth(root1,previous_target,counter0,attempt_id) in
  let old11 = auth(root1,previous_target,counter1,attempt_id) in
  out(c,old00);out(c,old01);out(c,old10);out(c,old11);
  let A0 = auth(root0,honest_target,counter0,attempt_id) in
  let P0 = pseud(root0,honest_target,counter0,attempt_id) in
  let root_for_second = choice[root0,root1] in
  let A1 = auth(root_for_second,honest_target,counter1,attempt_id) in
  let P1 = pseud(root_for_second,honest_target,counter1,attempt_id) in
  out(c,spk(sign_sk));out(c,pk(kem_sk0));out(c,pk(kem_sk1));
  (Vehicle(A0,P0,honest_target,window_id,spk(sign_sk))
   | Target(A0,P0,honest_target,window_id,epoch_id,j0,kem_sk0,sign_sk,done0)
   | (in(done0,completed0:bitstring);
       (Vehicle(A1,P1,honest_target,window_id,spk(sign_sk))
       |Target(A1,P1,honest_target,window_id,epoch_id,j1,kem_sk1,sign_sk,done1))))

In [ ]:
%%writefile MobiHeal_AKE_Baseline.pv
(* MobiHeal M0-M3 single-target symbolic AKE abstraction. UNPROVED until ProVerif runs.
   Source: MobiHeal Sections 3.3-3.5; equations (2)-(6).
   Old RSU's credential, derived from the same vehicle root using a DIFFERENT
   target ID, is deliberately leaked to the Dolev-Yao attacker before M0.
   The target and vehicle root are not compromised before the challenge.
   The vehicle root is disclosed after the target completes one handover.
   Public radio channel is fully attacker-controlled.
   One ephemeral target key and one prepared target record are modeled.

   IDEALIZATIONS: ML-KEM is abstract public-key encryption with perfect secrecy;
   KMAC is a free (unbreakable) constructor; XOR/PRF combiner an opaque function;
   certified ML-DSA/Merkle membership represented by a trusted verification
   key and signed leaf. Crash persistence, B0-B3 recovery, multiple attempts,
   revocation/epoch state, and transcript unlinkability are NOT modeled here.
   This is neither a computational PQ proof nor a full-paper proof.
*)

free c: channel.

type skey.
type pkey.
fun pk(skey): pkey.
fun kemenc(bitstring, pkey): bitstring.
reduc forall m:bitstring, sk:skey; kemdec(kemenc(m,pk(sk)),sk) = m.

type signsk.
type signpk.
fun spk(signsk): signpk.
fun sign(bitstring,signsk): bitstring.
reduc forall m:bitstring, sk:signsk; checksign(sign(m,sk),spk(sk)) = m.

fun KMAC(bitstring,bitstring):bitstring.
fun auth(bitstring,bitstring,bitstring,bitstring):bitstring.
fun pseud(bitstring,bitstring,bitstring,bitstring):bitstring.
fun mix(bitstring,bitstring):bitstring.
fun leaf(bitstring,bitstring,bitstring,pkey):bitstring.

fun c_adm(bitstring,bitstring,bitstring,bitstring):bitstring.
fun c_offer(bitstring,bitstring,bitstring,bitstring,pkey):bitstring.
fun c_th2(bitstring,bitstring,bitstring,bitstring,bitstring):bitstring.
fun c_th(bitstring,bitstring,bitstring):bitstring.
fun c_amix(bitstring):bitstring.
fun c_zmix(bitstring):bitstring.
fun c_pre(bitstring):bitstring.
fun c_cv(bitstring):bitstring.
fun c_fv(bitstring,bitstring):bitstring.
fun c_cr(bitstring):bitstring.
fun c_fr(bitstring,bitstring):bitstring.
fun c_app(bitstring):bitstring.
fun c_sid(bitstring,bitstring,bitstring,bitstring,bitstring,bitstring):bitstring.

fun m0(bitstring,bitstring,bitstring,bitstring):bitstring [data].
fun m1(bitstring,bitstring,bitstring,bitstring,bitstring,pkey,bitstring,bitstring,bitstring):bitstring [data].
fun m2(bitstring,bitstring,bitstring,bitstring,bitstring):bitstring [data].
fun m3(bitstring,bitstring):bitstring [data].

fun senc(bitstring,bitstring):bitstring.
reduc forall m:bitstring,k:bitstring; sdec(senc(m,k),k) = m.

free previous_target:bitstring.
free honest_target:bitstring.
free window_id:bitstring.
free epoch_id:bitstring.
free pool_index:bitstring.
free attempt_id:bitstring.

(* A private challenge payload is released only under the derived app key.
   query attacker(payload) tests application-key confidentiality indirectly. *)
free challenge_payload:bitstring [private].

(* These are noninjective correspondences; repeat/parallel session uniqueness
   is not asserted by the single-target model. *)
event VehicleSentM2(bitstring,bitstring).
event RSUCompleted(bitstring,bitstring).
event VehicleAccepted(bitstring,bitstring).
event OldRSULeaked.
event VehicleRootLeaked.

(* Sanity checks: ProVerif prints "not event(...) is false" when
   a matching event IS reachable (and "true" if dead/vacuous). *)
query sid:bitstring,k:bitstring; event(RSUCompleted(sid,k)).
query sid:bitstring,k:bitstring; event(VehicleAccepted(sid,k)).
query attacker(challenge_payload).
query sid:bitstring,k:bitstring;
  event(RSUCompleted(sid,k)) ==> event(VehicleSentM2(sid,k)).
query sid:bitstring,k:bitstring;
  event(VehicleAccepted(sid,k)) ==> event(RSUCompleted(sid,k)).

let Vehicle(A:bitstring, P:bitstring, R:bitstring, W:bitstring, trusted_sign_pk:signpk) =
  new n0:bitstring;
  let alpha0 = KMAC(A,c_adm(R,W,P,n0)) in
  let M0 = m0(P,W,n0,alpha0) in
  out(c,M0);
  in(c,received1:bitstring);
  let m1(=R, e:bitstring, =W, j:bitstring, nr:bitstring,
         offered_pk:pkey, offered_leaf:bitstring,
         offered_sig:bitstring, offered_tag:bitstring) = received1 in
  if offered_leaf = leaf(R,e,j,offered_pk) then
    let signed_leaf = checksign(offered_sig,trusted_sign_pk) in
    if signed_leaf = offered_leaf then
      if offered_tag = KMAC(A,c_offer(M0,e,j,nr,offered_pk)) then
        new nv:bitstring;
        new z:bitstring;
        let ct = kemenc(z,offered_pk) in
        let h2 = c_th2(M0,received1,P,nv,ct) in
        let Y_A = KMAC(A,c_amix(h2)) in
        let Y_Z = KMAC(z,c_zmix(h2)) in
        let Kstar = mix(Y_A,Y_Z) in
        let alpha2 = KMAC(A,c_pre(h2)) in
        let tauV = KMAC(KMAC(Kstar,c_cv(h2)),c_fv(h2,alpha2)) in
        let M2 = m2(P,nv,ct,alpha2,tauV) in
        let h = c_th(M0,received1,M2) in
        let sid = c_sid(R,e,W,j,P,h) in
        let Kapp = KMAC(Kstar,c_app(h)) in
        let expected_tauR = KMAC(KMAC(Kstar,c_cr(h)),c_fr(h,sid)) in
        event VehicleSentM2(sid,Kapp);
        out(c,M2);
        in(c,received3:bitstring);
        let m3(=sid,actual_tauR:bitstring) = received3 in
        if actual_tauR = expected_tauR then
          event VehicleAccepted(sid,Kapp);
          0.

let Target(A:bitstring,P:bitstring,R:bitstring,W:bitstring,
           e:bitstring,j:bitstring,kem_sk:skey,sign_sk:signsk,done:channel) =
  in(c,received0:bitstring);
  let m0(=P,=W,n0:bitstring,actual_alpha0:bitstring) = received0 in
  if actual_alpha0 = KMAC(A,c_adm(R,W,P,n0)) then
    new nr:bitstring;
    let public_key = pk(kem_sk) in
    let certified_leaf = leaf(R,e,j,public_key) in
    let signed_leaf = sign(certified_leaf,sign_sk) in
    let beta = KMAC(A,c_offer(received0,e,j,nr,public_key)) in
    let M1 = m1(R,e,W,j,nr,public_key,certified_leaf,signed_leaf,beta) in
    out(c,M1);
    in(c,received2:bitstring);
    let m2(=P,nv:bitstring,ct:bitstring,actual_alpha2:bitstring,
           actual_tauV:bitstring) = received2 in
    let h2 = c_th2(received0,M1,P,nv,ct) in
    if actual_alpha2 = KMAC(A,c_pre(h2)) then
      let z = kemdec(ct,kem_sk) in
      let Y_A = KMAC(A,c_amix(h2)) in
      let Y_Z = KMAC(z,c_zmix(h2)) in
      let Kstar = mix(Y_A,Y_Z) in
      let expected_tauV = KMAC(KMAC(Kstar,c_cv(h2)),c_fv(h2,actual_alpha2)) in
      if actual_tauV = expected_tauV then
        let h = c_th(received0,M1,received2) in
        let sid = c_sid(R,e,W,j,P,h) in
        let Kapp = KMAC(Kstar,c_app(h)) in
        let tauR = KMAC(KMAC(Kstar,c_cr(h)),c_fr(h,sid)) in
        event RSUCompleted(sid,Kapp);
        out(c,senc(challenge_payload,Kapp));
        out(c,m3(sid,tauR));
        out(done,sid);
        0.

process
  new root:bitstring;
  new counter:bitstring;
  new kem_sk:skey;
  new sign_sk:signsk;
  new done:channel;
  let P = pseud(root,honest_target,counter,attempt_id) in
  let A = auth(root,honest_target,counter,attempt_id) in
  let Aold = auth(root,previous_target,counter,attempt_id) in
  event OldRSULeaked;
  out(c,Aold);
  out(c,spk(sign_sk));
  out(c,pk(kem_sk));
  (Vehicle(A,P,honest_target,window_id,spk(sign_sk))
   | Target(A,P,honest_target,window_id,epoch_id,pool_index,kem_sk,sign_sk,done)
   | (in(done,ack:bitstring); event VehicleRootLeaked; out(c,root)) )

### 11.1 Verify ProVerif binary / optionally compile pinned official release (SHA-256 checked)

In [ ]:
import re, tarfile, urllib.request
PV_SHA256='4871f53c32ab4a04669a060c4886ba5d9080496963fb980a9a62d2c429ceabc4'
PV_URLS=['https://bblanche.gitlabpages.inria.fr/proverif/proverif2.05.tar.gz',
         'https://proverif.inria.fr/proverif2.05.tar.gz']

def command(args,timeout=60,cwd=None):
    try:
        p=subprocess.run([str(a) for a in args],cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.PIPE,
                          text=True,timeout=timeout)
        return p.returncode,p.stdout,p.stderr,False
    except subprocess.TimeoutExpired as e:
        decode=lambda x:x.decode(errors='replace') if isinstance(x,bytes) else x or ''
        return None,decode(e.stdout),decode(e.stderr),True

def proverif_binary():
    found=shutil.which('proverif')
    if found:
        rc,so,se,_=command([found,'-help'],20)
        if 'Proverif 2.05.' in so+se: return Path(found)
        print('Installed ProVerif version is not 2.05; symbolic outputs will not be labelled 2.05.')
        return None
    if not INSTALL_PROVERIF:return None
    if not hasattr(os,'geteuid') or os.geteuid()!=0:
        print('Root access needed for optional OCaml toolchain install');return None
    rc,so,se,_=command(['bash','-lc','export DEBIAN_FRONTEND=noninteractive; apt-get update -qq && apt-get install -y -qq ocaml ocaml-findlib libocamlgraph-ocaml-dev make wget pkg-config'],420)
    (LOGS/'proverif_toolchain.txt').write_text(so+'\n'+se)
    if rc!=0:return None
    archive=ROOT/'proverif2.05.tar.gz'
    if not archive.exists() or hashlib.sha256(archive.read_bytes()).hexdigest()!=PV_SHA256:
        for url in PV_URLS:
            try:
                urllib.request.urlretrieve(url,archive)
                if hashlib.sha256(archive.read_bytes()).hexdigest()==PV_SHA256:break
            except Exception as exc:print('ProVerif mirror failure:',type(exc).__name__)
    if not archive.exists() or hashlib.sha256(archive.read_bytes()).hexdigest()!=PV_SHA256:
        print('Unverified / unavailable ProVerif source archive — refusing to build');return None
    with tarfile.open(archive,'r:gz') as tf:tf.extractall(ROOT,filter='data')
    directory=ROOT/'proverif2.05'
    rc,so,se,_=command(['bash','-lc','./build'],420,cwd=directory)
    (LOGS/'proverif_build.txt').write_text(so+'\n'+se)
    if rc!=0:return None
    executable=directory/'proverif'
    rc,so,se,_=command([executable,'-help'],20)
    return executable if rc==0 and 'Proverif 2.05.' in so+se else None

pv_executable=proverif_binary() if RUN_FORMAL else None
print('ProVerif 2.05:',str(pv_executable) if pv_executable else 'NOT_EXECUTED')

### 11.2 Parse then run all models and store unedited query logs

In [ ]:
PV_TASKS={
 'MobiHeal_Explicit_Transcript.pv':'PV1_PV2_PV3_explicit_transcript',
 'MobiHeal_Single_Challenge_Privacy.pv':'PV4_single_challenge_privacy',
 'MobiHeal_Target_Credential_Leak_Mutant.pv':'PV5_target_credential_leak_mutant',
 'MobiHeal_Two_Hops_Same_Root.pv':'PV6_PV7_same_root_two_hops',
 'MobiHeal_Two_Challenge_Identity_Swap.pv':'PV8_identity_swap_UNESTABLISHED',
 'MobiHeal_Two_Challenge_Linkability.pv':'PV8_linkability_UNESTABLISHED',
 'MobiHeal_Replicated_AKE.pv':'supplement_replicated',
 'MobiHeal_AKE_Baseline.pv':'supplement_baseline',
}
query_pattern=re.compile(r'^\s*RESULT\s+(.+?)\s+is\s+(true|false|cannot be proved)\s*\.\s*$',re.I|re.M)
pv_rows=[]
for filename,objective in PV_TASKS.items():
    p=Path(filename)
    target=MODELS/filename
    target.write_bytes(p.read_bytes())
    record={'model':filename,'objective':objective,'sha256':hashlib.sha256(target.read_bytes()).hexdigest(),
            'parse':'NOT_EXECUTED','status':'NOT_EXECUTED','query_count':0,'results':[]}
    if pv_executable:
        rc,so,se,timeout=command([pv_executable,'-parse-only',target],120)
        (LOGS/(target.stem+'_parse.log')).write_text(so+'\n'+se)
        record['parse']='PASS' if rc==0 else 'TIMEOUT' if timeout else 'FAIL'
        if rc==0:
            rc,so,se,timeout=command([pv_executable,target],CONFIG['proverif_timeout_s'])
            (LOGS/(target.stem+'_execute.log')).write_text(so+'\n'+se)
            record['results']=[{'query':q,'result':v.lower()} for q,v in query_pattern.findall(so)]
            record['query_count']=len(record['results'])
            record['status']='EXECUTED' if rc==0 and record['query_count'] else 'TIMEOUT' if timeout else 'FAILED_OR_NO_VERDICT'
    pv_rows.append(record)
    print(filename,record['parse'],record['status'],record['query_count'],record['sha256'][:12])
(ROOT/'proverif_full_results.json').write_text(json.dumps(pv_rows,indent=2))
flat=[{k:r[k] for k in ('model','objective','sha256','parse','status','query_count')} | q
      for r in pv_rows for q in (r['results'] or [{'query':'','result':'NOT_EXECUTED_OR_NO_VERDICT'}])]
pv_df=pd.DataFrame(flat)
pv_df.to_csv(ROOT/'table_VIII_ProVerif_results.csv',index=False)
display(pv_df[['objective','status','query','result']])

## 12. Table XI — six adversarial component ablations + two supplemental variants
Variants deliberately break one design property at a time. Results reflect an explicit mutant and the tested attack strategy; they are not universal cryptographic breaks or comparable end-to-end latency measurements.

In [ ]:
%%writefile mobiheal_ablations.py
"""MobiHeal eight targeted component ablations.

Runs against the supplied current-beta1 MobiHeal research harness, NOT a deployed
or formally validated production implementation. Deliberate gate bypasses and
rollback faults occur only in isolated temporary databases.
"""
from __future__ import annotations
import contextlib, hashlib, hmac, json, os, platform, secrets, sqlite3, statistics, tempfile, time
from dataclasses import replace
from pathlib import Path

import mobiheal_core as mh

VARIANTS = [
 'Reusable source-issued credential',
 'No ephemeral ML-KEM contribution',
 'No request-bound beta1',
 'No alpha0 gate',
 'No alpha2 gate',
 'Volatile allocation state',
 'Per-offer ML-DSA authentication',
 'Cold-cache operation',
]

def elapsed(fn):
 t=time.perf_counter_ns();out=fn();return (time.perf_counter_ns()-t)/1e6,out

def m0_with(pid,A,rid,w):
 m=mh.M0(pid,int(w),secrets.token_bytes(16),b'')
 return replace(m,a0=mh.alpha0(rid,m,A))

def attack_complete(r,m0,pid,A,m1):
 """Attack with only a compromised RSU's pseudonym and authentication secret."""
 ct,z=mh.CR.encaps(m1.pk);nv=secrets.token_bytes(16)
 h2=mh.prehash(m0,m1,pid,nv,ct)
 a2=mh.CR.kmac(A,h2,32,mh.LABEL['PREAUTH'])
 k=mh.derivemix(A,z,h2)
 cv=mh.CR.kmac(k,h2,32,mh.LABEL['CONFV'])
 tv=mh.CR.kmac(cv,h2+a2,32,mh.LABEL['FINV'])
 st,m3,rsu_key=r.recv_m2(mh.M2(pid,nv,ct,a2,tv))
 return st,m3,rsu_key

def insert_reused_record(r,old_pid,old_A,w):
 """Explicit insecure variant: the target accepts the exposed source credential."""
 g=secrets.token_bytes(16)
 with r.conn:
  r.conn.execute('INSERT INTO grp(g,state,next_attempt,q,active_pid) VALUES(?,?,?,?,?)',(g,'unused',0,1,None))
  r.conn.execute('INSERT INTO rec(pid,g,w,A,attempt,state,j,m0,m1,m2h,m3) VALUES(?,?,?,?,?,?,?,?,?,?,?)',
                 (old_pid,g,int(w),old_A,0,'unused',None,None,None,None,None))

def test_source_credential(seed):
 with tempfile.TemporaryDirectory() as td:
  p=Path(td);v=mh.Vehicle(p/'v.db',root=hashlib.sha3_256(f'VEH:{seed}'.encode()).digest())
  old=mh.RSU(p/'old.db',b'OLD-RSU1',B=4);new=mh.RSU(p/'new.db',b'NEW-RSU1',B=4)
  w0=1000+seed;old.provision(v,w0,q=1)
  oldctx=v.start(old.rid,w0,0)
  old_desc=v.verify_descriptor(old)
  oldst,old_offer=old.recv_m0(oldctx['m0'])
  assert oldst=='OFFERED'
  old_m2=v.send_m2(oldctx,old_offer,old_desc)
  old_completed,old_m3,old_key=old.recv_m2(old_m2)
  assert old_completed=='COMPLETED' and v.complete(old_m3)[0]==old_key
  leaked=old.conn.execute('SELECT pid,A,w FROM rec LIMIT 1').fetchone()
  pid,A,w=leaked
  assert A==oldctx['A'] and pid==oldctx['pid']
  # Genuine new-target provisioning uses a distinct r_k domain and is independent.
  new.provision(v,w0,q=1)
  forged=m0_with(pid,A,new.rid,w)
  st0,_=new.recv_m0(forged)
  before=new.counts['alloc']
  assert st0 in ('REJECT_UNKNOWN','REJECT_GROUP') and before==0,st0
  insert_reused_record(new,pid,A,w)
  st1,offer=new.recv_m0(forged)
  completed=False
  if st1=='OFFERED':
   final,_,_=attack_complete(new,forged,pid,A,offer)
   completed=(final=='COMPLETED')
  row={'seed':seed,'variant':VARIANTS[0], 'baseline_reject':st0,'variant_offer':st1,
       'unauthorized_allocations':new.counts['alloc']-before,
       'unauthorized_acceptance':int(completed),'passed':int(completed and st1=='OFFERED' and before==0),
       'scope':'functional explicit reused-credential mutant'}
  old.close();new.close();v.close()
  return row

def test_no_kem(seed):
 with tempfile.TemporaryDirectory() as td:
  p=Path(td);root=hashlib.sha3_256(f'ROOT:{seed}'.encode()).digest()
  v=mh.Vehicle(p/'v.db',root=root);r=mh.RSU(p/'r.db',b'RSU-NOK1',B=4)
  w=5000+seed;r.provision(v,w,q=1);desc=v.verify_descriptor(r)
  ctx=v.start(r.rid,w,0);st,m1=r.recv_m0(ctx['m0']);assert st=='OFFERED'
  m2=v.send_m2(ctx,m1,desc);st,m3,actual_key=r.recv_m2(m2);assert st=='COMPLETED'
  vk,_=v.complete(m3);assert vk==actual_key
  # Retire the application session and erase stored endpoint copies in the harness.
  with v.conn:v.conn.execute('UPDATE vehicle SET accepted_key=NULL WHERE id=1')
  public_h=mh.h_full(ctx['m0'],m1,m2)
  # Deliberately insecure key schedule with *no* Z contribution.
  insecure_key=mh.CR.kmac(ctx['A'],public_h,32,mh.LABEL['APP'])
  # After vehicle-root exposure, attacker recomputes the target credential.
  recovered_A=mh.derive(root,r.rid,w,0,0)[1]
  recovered=mh.CR.kmac(recovered_A,public_h,32,mh.LABEL['APP'])
  assert recovered_A==ctx['A']
  row={'seed':seed,'variant':VARIANTS[1],
       'variant_retired_key_recovered':int(hmac.compare_digest(insecure_key,recovered)),
       'same_root_only_guess_matches_original':int(hmac.compare_digest(recovered,actual_key)),
       'passed':int(hmac.compare_digest(insecure_key,recovered)),
       'scope':'constructed KMAC-only mutant; not computational secrecy proof for baseline'}
  r.close();v.close();return row

def send_without_beta(ctx,m1,desc):
 """Exact vehicle derivations with ONLY the beta1 check removed, for ablation."""
 rid,e,B,root,expiry,_,_=desc
 assert m1.rid==rid and m1.e==e and m1.w==ctx['m0'].w
 assert mh.verify_proof(rid,e,m1.j,m1.pk,m1.path,root)
 ct,z=mh.CR.encaps(m1.pk)
 nv=secrets.token_bytes(16)
 h2=mh.prehash(ctx['m0'],m1,ctx['pid'],nv,ct)
 a2=mh.CR.kmac(ctx['A'],h2,32,mh.LABEL['PREAUTH'])
 k=mh.derivemix(ctx['A'],z,h2)
 cv=mh.CR.kmac(k,h2,32,mh.LABEL['CONFV'])
 tv=mh.CR.kmac(cv,h2+a2,32,mh.LABEL['FINV'])
 return mh.M2(ctx['pid'],nv,ct,a2,tv)

def test_no_beta(seed):
 with tempfile.TemporaryDirectory() as td:
  p=Path(td);v=mh.Vehicle(p/'v.db');r=mh.RSU(p/'r.db',b'RSU-BETA',B=8)
  w=6000+seed;r.provision(v,w,q=1);d=v.verify_descriptor(r)
  original=v.start(r.rid,w,0);st,stale=r.recv_m0(original['m0']);assert st=='OFFERED'
  fresh=v.start(r.rid,w,0)
  assert original['m0'].wire()!=fresh['m0'].wire()
  rejected=False
  try:v.send_m2(fresh,stale,d)
  except ValueError as ex:rejected=('beta1' in str(ex))
  assert rejected
  m2=send_without_beta(fresh,stale,d)  # performs ONE real ML-KEM encapsulation
  rsu_status,_,_=r.recv_m2(m2)      # target should still reject mismatched alpha2
  row={'seed':seed,'variant':VARIANTS[2],'baseline_stale_offer_rejected':int(rejected),
       'variant_stale_offer_accepted_for_encapsulation':1,
       'wasted_encapsulations':1,'rsu_final_status':rsu_status,
       'unauthorized_rsu_completion':int(rsu_status=='COMPLETED'),
       'passed':int(rejected and rsu_status!='COMPLETED'),
       'scope':'vehicle stale-offer gate removal; unnecessary encapsulation, not false RSU accept'}
  r.close();v.close();return row

@contextlib.contextmanager
def patch_field(obj,name,value):
 before=getattr(obj,name);setattr(obj,name,value)
 try:yield
 finally:setattr(obj,name,before)

def test_no_alpha0(seed):
 with tempfile.TemporaryDirectory() as td:
  p=Path(td);v=mh.Vehicle(p/'v.db');r=mh.RSU(p/'r.db',b'RSU-GAT0',B=8)
  w=7000+seed;r.provision(v,w,q=1)
  ctx=v.start(r.rid,w,0);bad=replace(ctx['m0'],a0=b'\0'*32)
  st0,_=r.recv_m0(bad);before=r.counts['alloc'];assert st0=='REJECT_ALPHA0' and before==0
  # Gate deletion exactly modeled by making the alpha0 verification expression
  # return the supplied tag; all other receive logic runs unchanged.
  with patch_field(mh,'alpha0',lambda rid,m0,A: m0.a0):
   st1,_=r.recv_m0(bad)
  allocated=r.counts['alloc']-before
  abandoned=r.abandon(ctx['pid'])
  end_states=[x['state'] for x in r.ledger()]
  row={'seed':seed,'variant':VARIANTS[3], 'baseline_status':st0,'mutant_status':st1,
       'invalid_allocations':allocated,'abandoned_indices':end_states.count('abandoned'),
       'passed':int(st1=='OFFERED' and allocated==1 and abandoned),
       'scope':'isolated verifier gate bypass; authenticated pseudonym still known'}
  r.close();v.close();return row

def test_no_alpha2(seed):
 with tempfile.TemporaryDirectory() as td:
  p=Path(td);v=mh.Vehicle(p/'v.db');r=mh.RSU(p/'r.db',b'RSU-GAT2',B=8)
  w=8000+seed;r.provision(v,w,q=1);d=v.verify_descriptor(r)
  ctx=v.start(r.rid,w,0);st,m1=r.recv_m0(ctx['m0']);assert st=='OFFERED'
  m2=v.send_m2(ctx,m1,d);bad=replace(m2,a2=b'\0'*32)
  decaps_before=r.counts['decap']
  st0,_,_=r.recv_m2(bad)
  assert st0=='REJECT_ALPHA2' and r.counts['decap']==decaps_before
  original_kmac=mh.CR.kmac
  def without_preauth_gate(key,data,n,label):
   if label==mh.LABEL['PREAUTH']:return bad.a2
   return original_kmac(key,data,n,label)
  with patch_field(mh.CR,'kmac',without_preauth_gate):
   st1,_,_=r.recv_m2(bad)
  wasted=r.counts['decap']-decaps_before
  state=r.conn.execute('SELECT state FROM kem WHERE j=?',(m1.j,)).fetchone()[0]
  row={'seed':seed,'variant':VARIANTS[4], 'baseline_status':st0,'mutant_status':st1,
       'invalid_decapsulations':wasted,'abandoned_indices':int(state=='abandoned'),
       'unauthorized_completion':int(st1=='COMPLETED'),
       'passed':int(wasted==1 and st1=='REJECT_TAUV' and state=='abandoned'),
       'scope':'isolated predecapsulation gate bypass; confirmation remains active'}
  r.close();v.close();return row

def test_volatile(seed):
 with tempfile.TemporaryDirectory() as td:
  p=Path(td);v=mh.Vehicle(p/'v.db');r=mh.RSU(p/'r.db',b'RSU-VOLT',B=8)
  w=9000+seed;r.provision(v,w,q=2)
  snap=p/'snapshot_before_offer.db'
  con=sqlite3.connect(str(snap));r.conn.backup(con);con.close()
  ctx0=v.start(r.rid,w,0);st,m1=r.recv_m0(ctx0['m0']);assert st=='OFFERED'
  initial_index=m1.j
  rinfo=dict(db=r.db,rid=r.rid,B=r.B,e=r.e,sigkeys=(r.sigpk,r.sigsk))
  r.close()
  # Baseline: reopen genuine durable state and abandon the outstanding offer.
  safe=mh.RSU(initialize=False,**rinfo);num=safe.recover_process()
  ctx1=v.start(safe.rid,w,1)
  safe_status,m1safe=safe.recv_m0(ctx1['m0'])
  baseline_no_reoffer=(num==1 and safe_status=='OFFERED' and m1safe.j!=initial_index)
  safe.close()
  assert baseline_no_reoffer
  # Deliberately FAULTY mutant: restore pre-offer database snapshot (simulated
  # loss of volatile allocation state). Not a naturally occurring SQLite crash.
  for sidecar in ('-wal','-shm'):
   f=Path(str(rinfo['db'])+sidecar)
   if f.exists():f.unlink()
  Path(rinfo['db']).write_bytes(snap.read_bytes())
  bad=mh.RSU(initialize=False,**rinfo)
  alternate=v.start(bad.rid,w,0) # distinct freshly generated M0, same PID
  assert alternate['m0'].wire()!=ctx0['m0'].wire()
  bad_status,newoffer=bad.recv_m0(alternate['m0'])
  recycled=bool(bad_status=='OFFERED' and newoffer.j==initial_index)
  row={'seed':seed,'variant':VARIANTS[5], 'safe_index':m1safe.j,
       'original_offered_index':initial_index,
       'rolled_back_reoffered_index':newoffer.j if newoffer else None,
       'baseline_prevented_reuse':int(baseline_no_reoffer),
       'mutant_reused_index':int(recycled), 'passed':int(recycled),
       'scope':'injected database rollback mutant, NOT real power-loss claim'}
  bad.close();v.close();return row

def test_signature_comparison(seed,B=1024,reps=10):
 with tempfile.TemporaryDirectory() as td:
  p=Path(td);t0=time.perf_counter_ns()
  r=mh.RSU(p/'r.db',b'RSU-SIGN',B=B)
  key_pool_init_ms=(time.perf_counter_ns()-t0)/1e6
  tm,desc=elapsed(r.descriptor)
  _,e,_,root,expiry,_,spk=desc
  A=secrets.token_bytes(32);j=1
  pk=r.conn.execute('SELECT pk FROM kem WHERE j=1').fetchone()[0]
  path=mh.proof(r._levels,0)
  data=[]
  for k in range(reps):
   pid=secrets.token_bytes(16);w=11000+seed*10000+k
   m0=m0_with(pid,A,r.rid,w)
   nr=secrets.token_bytes(16)
   tb=time.perf_counter_ns()
   beta=mh.beta1(A,m0,e,j,nr,pk)
   m1=mh.M1(r.rid,e,w,j,nr,pk,path,beta)
   baseline_offer_ms=(time.perf_counter_ns()-tb)/1e6
   tv=time.perf_counter_ns()
   assert hmac.compare_digest(beta,mh.beta1(A,m0,e,j,nr,pk))
   assert mh.verify_proof(r.rid,e,j,pk,path,root)
   baseline_verify_ms=(time.perf_counter_ns()-tv)/1e6
   # Alternative generates its own beta and signs the request-bound offer.
   # It does not build or retrieve a Merkle path on the online path.
   ts=time.perf_counter_ns()
   beta_alt=mh.beta1(A,m0,e,j,nr,pk)
   signed_input=mh.canon(m0,r.rid,e,w,j,nr,pk,beta_alt) # request-bound
   per_offer_sig=mh.CR.sign(r.sigsk,signed_input)
   alternative_offer_ms=(time.perf_counter_ns()-ts)/1e6
   assert len(per_offer_sig)==mh.SIG_N
   ta=time.perf_counter_ns()
   assert hmac.compare_digest(beta,mh.beta1(A,m0,e,j,nr,pk))
   assert mh.CR.verify(spk,signed_input,per_offer_sig)
   alternative_verify_ms=(time.perf_counter_ns()-ta)/1e6
   base_msg_bytes=len(m1.wire());alt_msg_bytes=base_msg_bytes-len(path)*48+len(per_offer_sig)
   data.append({'seed':seed,'rep':k,'B':B,'variant':VARIANTS[6],
     'pool_generation_ms':key_pool_init_ms,'offline_merkle_descriptor_ms':tm,
     'baseline_offer_auth_ms':baseline_offer_ms,
     'baseline_verify_auth_ms':baseline_verify_ms,
     'alternative_offer_auth_ms':alternative_offer_ms,
     'alternative_verify_auth_ms':alternative_verify_ms,
     'baseline_M1_bytes':base_msg_bytes,'per_offer_signed_M1_bytes':alt_msg_bytes,
     'baseline_total_bytes':72+base_msg_bytes+1184+80,
     'per_offer_total_bytes':72+alt_msg_bytes+1184+80,
     'signature_bytes':len(per_offer_sig),
     'passed':1,
     'scope':'offer-authentication component only; certificate/descriptor cached, not full AKE'} )
  r.close();return data

def test_cold_cache(seed,B=1024):
 with tempfile.TemporaryDirectory() as td:
  p=Path(td);v=mh.Vehicle(p/'v.db');r=mh.RSU(p/'r.db',b'RSU-COLD',B=B)
  desc=r.descriptor();rid,e,n,root,expiry,sig,spk=desc
  t=time.perf_counter_ns()
  sig_ok=mh.CR.verify(spk,mh.canon(rid,e,n,root,expiry),sig)
  verify_ms=(time.perf_counter_ns()-t)/1e6
  assert sig_ok and expiry>time.time()
  # Raw, explicitly defined compact descriptor encoding; cert chain is NOT sent.
  descriptor_wire=(rid+mh.u64(e)+mh.u64(n)+root+mh.u64(expiry)+sig+spk)
  wire_components={'rid':len(rid),'epoch':8,'pool_size':8,'root':len(root),'expiry':8,
                   'signature':len(sig),'signer_public_key_not_certificate':len(spk)}
  assert len(descriptor_wire)==sum(wire_components.values())
  v.verify_descriptor(r)
  w=12000+seed;r.provision(v,w,q=1)
  ctx=v.start(r.rid,w,0);st,m1=r.recv_m0(ctx['m0']);assert st=='OFFERED'
  m2=v.send_m2(ctx,m1,desc);st,m3,key=r.recv_m2(m2);assert st=='COMPLETED'
  v.complete(m3)
  warm=sum(len(z.wire()) for z in (ctx['m0'],m1,m2,m3))
  lower_bound=warm+len(descriptor_wire)
  row={'seed':seed,'variant':VARIANTS[7],'B':B,'warm_cache_bytes':warm,
       'descriptor_bytes_without_certificate_chain':len(descriptor_wire),
       'partial_cold_bytes_lower_bound':lower_bound,
       'local_descriptor_signature_verification_ms':verify_ms,
       'certificate_chain_bytes':None,'directory_framing_bytes':None,
       'network_acquisition_ms':None,'full_cold_cache_bytes':None,
       'passed':int(warm==3064 if B==1024 else warm>0),
       'scope':'partial cold-cache lower bound; trust chain/network not implemented',
       'descriptor_components':json.dumps(wire_components,sort_keys=True)}
  v.close();r.close();return row


def execute_all(seeds,B=1024,signature_reps=10,stop_on_failure=True):
 import pandas as pd
 rows=[];sign_rows=[]
 scalar_jobs=(test_source_credential,test_no_kem,test_no_beta,test_no_alpha0,test_no_alpha2,test_volatile)
 for seed in seeds:
  for fn in scalar_jobs:
   try:
    result=fn(seed);rows.append(result)
    if stop_on_failure:assert result['passed'],f'{fn.__name__} failed seed {seed}: {result}'
   except Exception as e:
    if stop_on_failure:raise
    rows.append({'seed':seed,'variant':VARIANTS[scalar_jobs.index(fn)],'passed':0,'error':repr(e),'scope':'error; not a result'})
  for x in test_signature_comparison(seed,B=B,reps=signature_reps):sign_rows.append(x)
  x=test_cold_cache(seed,B=B)
  if stop_on_failure:assert x['passed'],x
  rows.append(x)
 return pd.DataFrame(rows),pd.DataFrame(sign_rows)


def summarize(raw,signatures,backend):
 import pandas as pd
 out=[]
 for variant in VARIANTS:
  df=signatures if variant==VARIANTS[6] else raw[raw.variant==variant]
  if len(df)==0:
   out.append({'variant':variant,'trials':0,'passed':0,'evidence':'NOT_RUN'});continue
  d={'variant':variant,'trials':int(len(df)),'passed':int(df.passed.sum()),
     'evidence':'NATIVE_TESTED' if backend=='native' else 'DEV_ONLY'}
  cols={'Reusable source-issued credential':['unauthorized_acceptance','unauthorized_allocations'],
        'No ephemeral ML-KEM contribution':['variant_retired_key_recovered','same_root_only_guess_matches_original'],
        'No request-bound beta1':['variant_stale_offer_accepted_for_encapsulation','wasted_encapsulations','unauthorized_rsu_completion'],
        'No alpha0 gate':['invalid_allocations','abandoned_indices'],
        'No alpha2 gate':['invalid_decapsulations','abandoned_indices'],
        'Volatile allocation state':['mutant_reused_index'],
        'Per-offer ML-DSA authentication':['baseline_offer_auth_ms','baseline_verify_auth_ms',
              'alternative_offer_auth_ms','alternative_verify_auth_ms','baseline_total_bytes','per_offer_total_bytes',
              'pool_generation_ms','offline_merkle_descriptor_ms'],
        'Cold-cache operation':['warm_cache_bytes','descriptor_bytes_without_certificate_chain',
              'partial_cold_bytes_lower_bound','local_descriptor_signature_verification_ms']}
  for col in cols[variant]:
   if col in df:
    nums=pd.to_numeric(df[col],errors='coerce').dropna()
    if len(nums):
     d[col+'_sum']=float(nums.sum()) if col in (
       'unauthorized_acceptance','unauthorized_allocations','variant_retired_key_recovered',
       'same_root_only_guess_matches_original','variant_stale_offer_accepted_for_encapsulation',
       'wasted_encapsulations','unauthorized_rsu_completion','invalid_allocations',
       'abandoned_indices','invalid_decapsulations','mutant_reused_index') else None
     d[col+'_mean']=float(nums.mean())
     d[col+'_p95']=float(nums.quantile(0.95))
  out.append(d)
 return pd.DataFrame(out)


def write_table(summary,destination,B=1024):
 """Export strictly observed native outcomes; DEV_ONLY never appears as empirical evidence."""
 by={r['variant']:r for r in summary.to_dict(orient='records')}
 def row(k,detail):
  s=by[k]
  if s['evidence']!='NATIVE_TESTED' or s['trials']==0:return r'\textbf{TBD (native run required)}'
  return detail(s)
 def i(s,k):return int(round(s.get(k+'_sum',0)))
 def f(s,k):return s.get(k+'_mean',float('nan'))
 choices={
 VARIANTS[0]:lambda s:rf"{i(s,'unauthorized_acceptance')}/{s['trials']} unauthorized RSU acceptances in reuse mutant",
 VARIANTS[1]:lambda s:rf"{i(s,'variant_retired_key_recovered')}/{s['trials']} reconstructed after root exposure",
 VARIANTS[2]:lambda s:rf"{i(s,'wasted_encapsulations')}/{s['trials']} wasted encapsulations; 0 RSU accepts",
 VARIANTS[3]:lambda s:rf"{i(s,'invalid_allocations')} invalid allocations; {i(s,'abandoned_indices')} abandoned",
 VARIANTS[4]:lambda s:rf"{i(s,'invalid_decapsulations')} invalid decapsulations; {i(s,'abandoned_indices')} abandoned",
 VARIANTS[5]:lambda s:rf"{i(s,'mutant_reused_index')}/{s['trials']} indices reoffered after injected rollback",
 VARIANTS[6]:lambda s:rf"prep {f(s,'offline_merkle_descriptor_ms'):.1f} ms (Merkle descriptor); offer {f(s,'baseline_offer_auth_ms'):.2f}/{f(s,'alternative_offer_auth_ms'):.2f} ms; {int(f(s,'baseline_total_bytes')):,}/{int(f(s,'per_offer_total_bytes')):,} B",
 VARIANTS[7]:lambda s:rf"{int(f(s,'descriptor_bytes_without_certificate_chain')):,} B descriptor; $\geq${int(f(s,'partial_cold_bytes_lower_bound')):,} B partial cold; full TBD",
 }
 objectives=[
 'Unauthorized target acceptance after source compromise',
 'Reconstruction of retired keys after permitted root exposure',
 'Acceptance of stale offers and unnecessary encapsulations',
 'Allocations and abandoned indices under invalid admission',
 'Decapsulation calls under invalid $M_2$ traffic',
 'Index reuse after injected restart or rollback',
 'Preparation, online processing, and bytes versus Merkle batching',
 'Descriptor-acquisition cost and total exchange bytes',
 ]
 names=[r'Reusable source-issued credential',r'No ephemeral ML-KEM contribution',r'No request-bound $\beta_1$',r'No $\alpha_0$ gate',r'No $\alpha_2$ gate',r'Volatile allocation state',r'Per-offer ML-DSA authentication',r'Cold-cache operation']
 lines=[r'\begin{table*}[!t]',r'\caption{Current-Protocol Component Ablations (Native Restricted Tests)}',r'\label{tab:ablations}',r'\centering\footnotesize',r'\begin{tabular}{@{}p{0.27\textwidth}p{0.50\textwidth}p{0.15\textwidth}@{}}',r'\toprule',r'Variant & Measurement or verification target & Result\\',r'\midrule']
 for k,label,objective in zip(VARIANTS,names,objectives):
  value=row(k,choices[k]);lines.append(label+' & '+objective+' & '+value+r'\\')
 lines += [r'\bottomrule',r'\end{tabular}',r'\par\raggedright\scriptsize Controlled negative variants, not full security proofs. Signature timings cover offer authentication only; cold-cache bytes exclude an actual certificate chain, directory framing, and network transport.',r'\end{table*}']
 Path(destination).write_text('\n'.join(lines)+'\n',encoding='utf-8')
 return str(destination)


def make_figure(sig,destination):
 import matplotlib.pyplot as plt
 from pathlib import Path
 vals=[sig['baseline_offer_auth_ms'].mean(),sig['alternative_offer_auth_ms'].mean(),
       sig['baseline_verify_auth_ms'].mean(),sig['alternative_verify_auth_ms'].mean()]
 fig,ax=plt.subplots(figsize=(8,3.8))
 ax.bar(['Merkle offer','Per-offer signature','Merkle verify','Signature verify'],vals,
        color=['#9bbfd8','#f2bb8c','#9dbca4','#d6b1ce'],edgecolor='#404b5a',linewidth=0.75)
 ax.set_ylabel('Mean authentication-only time (ms)')
 ax.set_title('MobiHeal offer authentication ablation (not full handshake)')
 ax.grid(axis='y',alpha=.25,ls='--');ax.set_axisbelow(True)
 fig.tight_layout();p=Path(destination);p.parent.mkdir(parents=True,exist_ok=True)
 fig.savefig(p.with_suffix('.png'),dpi=220)
 fig.savefig(p.with_suffix('.pdf'),bbox_inches='tight')
 plt.close(fig)
 return str(p.with_suffix('.pdf')),str(p.with_suffix('.png'))

In [ ]:
import mobiheal_ablations as ablations
# Native FULL: 10 distinct seeds, one mutant trial/seed => paper Table XI 10/10.
# Increase ablation_trials_per_seed manually for supplementary repetitions only.
abl_rows=[];sig_rows=[]
for rep in range(CONFIG['ablation_trials_per_seed']):
    raw,sign=ablations.execute_all(SEEDS,B=CONFIG['handover_pool_B'],
                              signature_reps=CONFIG['signature_reps_per_seed'],stop_on_failure=True)
    raw=raw.assign(ablation_repetition=rep)
    sign=sign.assign(ablation_repetition=rep)
    abl_rows.append(raw);sig_rows.append(sign)
ablation_raw=pd.concat(abl_rows,ignore_index=True)
signature_raw=pd.concat(sig_rows,ignore_index=True)
ablation_summary=ablations.summarize(ablation_raw,signature_raw,mh.CR.label)
ablation_raw.to_csv(ROOT/'table_XI_ablations_raw.csv',index=False)
signature_raw.to_csv(ROOT/'signature_comparison_raw.csv',index=False)
ablation_summary.to_csv(ROOT/'table_XI_ablations_summary.csv',index=False)
ablations.write_table(ablation_summary,ROOT/'table_XI_plus_supplement.tex',B=CONFIG['handover_pool_B'])
display(ablation_summary)

## 13. Manuscript Figures 3–6 — generated only from this execution
The plots use this run’s raw datasets. Published figures are **not** redrawn from hardcoded numbers.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
plt.rcParams.update({'font.family':'serif','font.size':9,'axes.spines.top':False,'axes.spines.right':False,
                     'axes.grid':True,'grid.alpha':0.2,'savefig.bbox':'tight'})
def save_fig(filename):
    for extension in ('.png','.pdf'):
        plt.savefig(FIGS/(filename+extension),dpi=240)
    plt.close()
# Figure 3 — five stages, mean and observed p95, NOT bootstrap CI bars.
stage_names=['vehicle_m0_ms','rsu_m0_to_m1_ms','vehicle_m1_to_m2_ms',
             'rsu_m2_to_m3_ms','vehicle_verify_m3_ms']
stage_labels=['Vehicle: M0','RSU: M0→M1','Vehicle: M1→M2','RSU: M2→M3','Vehicle: M3']
summary=local.set_index('metric').loc[stage_names]
fig,ax=plt.subplots(figsize=(7.3,3.25))
x=np.arange(5);width=.38
ax.bar(x-width/2,summary['mean'],width,label='Mean')
ax.bar(x+width/2,summary['empirical_p95'],width,label='p95')
ax.set_xticks(x,stage_labels,rotation=18,ha='right');ax.set_ylabel('Local processing time (ms)');ax.legend();fig.tight_layout();save_fig('Figure_3_local_stage_mean_p95')
# Figure 4 — vehicle, RSU, both combined.
total_names=['vehicle_total_ms','rsu_total_ms','combined_local_ms']
total_labels=['Vehicle total','RSU total','Combined total']
s=local.set_index('metric').loc[total_names];fig,ax=plt.subplots(figsize=(6.5,3))
x=np.arange(3)
ax.bar(x-.2,s['mean'],.4,label='Mean');ax.bar(x+.2,s['empirical_p95'],.4,label='p95')
ax.set_xticks(x,total_labels);ax.set_ylabel('Local processing time (ms)');ax.legend();fig.tight_layout();save_fig('Figure_4_total_local_mean_p95')
# Figure 5 — per-parameter cell p95, averaged across speed/zone cells as in notebook driver.
fig,ax=plt.subplots(figsize=(6.5,3.4))
fig5=network_s.groupby(['loss_prob','one_way_ms'],as_index=False)['p95_completion_ms'].mean()
for delay,part in fig5.groupby('one_way_ms'):
    ax.plot(part.loss_prob*100,part.p95_completion_ms,marker='o',label=f'{delay:g} ms')
ax.set_xlabel('Independent packet-loss probability (%)');ax.set_ylabel('Mean cell-wise p95 handover latency (ms)');ax.legend(title='One-way delay');fig.tight_layout();save_fig('Figure_5_packet_loss_p95')
# Figure 6 — accurate four-message breakdown.
fig,ax=plt.subplots(figsize=(6.2,3.2))
base=np.zeros(len(wire));xs=np.arange(len(wire))
for name in ('M0','M1','M2','M3'):
    ax.bar(xs,wire[name],bottom=base,label=name)
    base+=wire[name].to_numpy()
for i,val in enumerate(base):ax.text(i,val+25,f'{int(val):,}',ha='center',fontsize=8)
ax.set_xticks(xs,wire.B.astype(str));ax.set_xlabel('Ephemeral KEM pool size B');ax.set_ylabel('Online V2I bytes');ax.legend(ncol=4);fig.tight_layout();save_fig('Figure_6_wire_bytes_by_pool')
# Supplementary plot: offer Merkle-vs-signature ablation, not claimed as a manuscript figure.
if not signature_raw.empty:
    ablations.make_figure(signature_raw,FIGS/'Supplement_offer_signature_comparison')
print('Figures written:',*[p.name for p in sorted(FIGS.glob('*.png'))],sep='\n - ')

## 14. Empirical challenge games — restricted attackers, not a computational security proof
We test whether a specific adversary using prior-RSU state (or later vehicle-root exposure but no new ML-KEM secret) can distinguish a new key from a random key. The intentionally invalid freshness control exposes the new target authorization secret **and** new KEM secret. These results are **statistical diagnostics against the explicitly implemented strategies**, not proofs of OPMH, retired-key forward secrecy, or multi-challenge privacy.

In [ ]:
import secrets, hmac
from dataclasses import replace

def one_pcs_trial(seed):
    import tempfile
    with tempfile.TemporaryDirectory() as td:
        root=secrets.token_bytes(32)
        v=mh.Vehicle(Path(td)/'v.db',root=root)
        old=mh.RSU(Path(td)/'old.db',b'OLD-RSU1',B=2)
        old_d=v.verify_descriptor(old);old.provision(v,101,q=1)
        first=v.start(old.rid,101,0);status,m1=old.recv_m0(first['m0'])
        m2=v.send_m2(first,m1,old_d)
        old_sk_saved=old.conn.execute('SELECT sk FROM kem WHERE j=?',(m1.j,)).fetchone()[0]
        old_z=mh.CR.decaps(old_sk_saved,m2.ct)
        status,m3,old_key=old.recv_m2(m2);v.complete(m3)
        old_A=first['A']
        new=mh.RSU(Path(td)/'new.db',b'NEW-RSU2',B=2)
        new_d=v.verify_descriptor(new);new.provision(v,102,q=1)
        second=v.start(new.rid,102,0);status,m1new=new.recv_m0(second['m0'])
        m2new=v.send_m2(second,m1new,new_d)
        new_sk_saved=new.conn.execute('SELECT sk FROM kem WHERE j=?',(m1new.j,)).fetchone()[0]
        status,m3new,new_key=new.recv_m2(m2new)
        v.complete(m3new)
        # Recover public h2/h; transcript only uses paper-admissible fields.
        h2=mh.prehash(second['m0'],m1new,second['pid'],m2new.nv,m2new.ct)
        h=mh.h_full(second['m0'],m1new,m2new)
        # Actual earlier RSU KEM secret was retained only in this test harness
        # before the earlier session completed; no new-target secret is disclosed.
        # The old shared secret is obtained before key erasure, then exposed
        # after the previous RSU completion as a stronger old-state diagnostic.
        exposed_old_z=old_z
        root_derived_A=mh.derive(root,new.rid,102,1,0)[1]
        assert root_derived_A==second['A']
        # Deliberately INVALID freshness control: both target Aa and its
        # un-erased challenge KEM decapsulation secret are exposed by the harness.
        recovered_new_z=mh.CR.decaps(new_sk_saved,m2new.ct)
        predicted_new=mh.CR.kmac(mh.derivemix(second['A'],recovered_new_z,h2),h,32,mh.LABEL['APP'])
        assert predicted_new==new_key
        guesses={
            'old_RSU_state_only':mh.CR.kmac(mh.derivemix(old_A,exposed_old_z,h2),h,32,mh.LABEL['APP']),
            'later_root_exposed_new_KEM_erased':mh.CR.kmac(mh.derivemix(root_derived_A,exposed_old_z,h2),h,32,mh.LABEL['APP']),
            'invalid_target_auth_and_KEM_exposed':predicted_new,
        }
        b=secrets.randbelow(2)
        challenge=new_key if b else secrets.token_bytes(32)
        rows=[]
        for case,prediction in guesses.items():
            guess=1 if hmac.compare_digest(prediction,challenge) else 0
            rows.append({'seed':seed,'case':case,'bit_real':b,'correct':int(b==guess)})
        old.close();new.close();v.close()
        return rows
pcs_rows=[]
for trial in range(CONFIG['pcs_trials']):
    pcs_rows.extend(one_pcs_trial(trial))
pcs_raw=pd.DataFrame(pcs_rows)
pcs_summary=pcs_raw.groupby('case',as_index=False).agg(trials=('correct','size'),correct=('correct','sum'))
pcs_summary['win_rate']=pcs_summary.correct/pcs_summary.trials
pcs_summary['observed_abs_advantage']=abs(pcs_summary.win_rate-.5)
pcs_raw.to_csv(ROOT/'pcs_specific_adversary_trials.csv',index=False)
pcs_summary.to_csv(ROOT/'pcs_specific_adversary_summary.csv',index=False)
display(pcs_summary)
print('NOT A PROOF: guessing success against three hand-coded strategies, not an arbitrary adversary.')

## 15. Paper reference versus measured-run comparison (no substituted measurements)
The `paper_reference.csv` file contains manuscript values for comparison. It must never be passed into the experiment code as measured input. Wire sizes should match exactly; timings and Monte Carlo percentages can differ.

In [ ]:
PAPER_REFERENCE=[
 ('Figure 3','Vehicle M0 mean (ms)',0.234,'time'),
 ('Figure 3','RSU M0→M1 mean (ms)',2.313,'time'),
 ('Figure 3','Vehicle M1→M2 mean (ms)',4.437,'time'),
 ('Figure 3','RSU M2→M3 mean (ms)',2.711,'time'),
 ('Figure 3','Vehicle M3 mean (ms)',3.665,'time'),
 ('Figure 4','Combined local processing mean (ms)',13.360,'time'),
 ('Figure 4','Combined local processing p95 (ms)',16.884,'time'),
 ('Evaluation','RSU commit share (%)',75.3,'time'),
 ('Figure 6','B=64 bytes',2872,'exact_serialized'),
 ('Figure 6','B=256 bytes',2968,'exact_serialized'),
 ('Figure 6','B=1024 bytes',3064,'exact_serialized'),
 ('Figure 6','B=4096 bytes',3160,'exact_serialized'),
 ('Table XII','B=64 preparation mean (ms)',63.825,'time'),
 ('Table XII','B=256 preparation mean (ms)',93.114,'time'),
 ('Table XII','B=1024 preparation mean (ms)',227.677,'time'),
 ('Table X','M2–M3 completion persistence off (ms)',2.991,'time'),
 ('Table X','M2–M3 completion persistence on (ms)',3.453,'time'),
 ('Table X','Application fields (bytes)',124,'exact_logical'),
 ('Table X','Pending report (bytes)',80,'exact_logical'),
 ('Table X','Cached M3 (bytes)',80,'exact_serialized'),
 ('Network','20% loss completion (%)',96.665,'stochastic'),
 ('Network','10% loss completion (%)',99.785,'stochastic'),
]
reference=pd.DataFrame(PAPER_REFERENCE,columns=['paper_location','metric','paper_reported','comparison_type'])
reference.to_csv(ROOT/'paper_reference.csv',index=False)
observed={
 'Vehicle M0 mean (ms)':float(handover.vehicle_m0_ms.mean()),
 'RSU M0→M1 mean (ms)':float(handover.rsu_m0_to_m1_ms.mean()),
 'Vehicle M1→M2 mean (ms)':float(handover.vehicle_m1_to_m2_ms.mean()),
 'RSU M2→M3 mean (ms)':float(handover.rsu_m2_to_m3_ms.mean()),
 'Vehicle M3 mean (ms)':float(handover.vehicle_verify_m3_ms.mean()),
 'Combined local processing mean (ms)':float(handover.combined_local_ms.mean()),
 'Combined local processing p95 (ms)':float(np.percentile(handover.combined_local_ms,95)),
 'RSU commit share (%)':float(commit_share),
 'M2–M3 completion persistence off (ms)':float(persist_raw.storage_off_m2_m3_ms.mean()),
 'M2–M3 completion persistence on (ms)':float(persist_raw.storage_on_m2_m3_ms.mean()),
 'Application fields (bytes)':int(persist_raw.application_fields_logical_bytes.iloc[0]),
 'Pending report (bytes)':int(persist_raw.report_logical_bytes.iloc[0]),
 'Cached M3 (bytes)':int(persist_raw.M3_bytes.iloc[0]),
}
for r in wire.itertuples():observed[f'B={r.B} bytes']=int(r.total)
for r in pool_summary.itertuples():observed[f'B={r.B} preparation mean (ms)']=float(r.preparation_mean_ms)
for r in network_by_loss.itertuples():
    if np.isclose(r.loss_prob,.1):observed['10% loss completion (%)']=float(r.success_rate_pct)
    if np.isclose(r.loss_prob,.2):observed['20% loss completion (%)']=float(r.success_rate_pct)
reference['observed_current_run']=reference.metric.map(observed)
reference['delta_observed_minus_reference']=reference.observed_current_run-reference.paper_reported
reference['comparison_warning']=reference.comparison_type.map({
 'time':'Hardware/storage-dependent: no exact-match guarantee',
 'stochastic':'Independent random simulation: no exact-match guarantee',
 'exact_serialized':'Structural byte count: equality expected',
 'exact_logical':'Logical record size: equality expected'})
reference['publication_eligible']=CRYPTO_BACKEND=='NATIVE'
reference.to_csv(ROOT/'paper_vs_current_run.csv',index=False)
display(reference)

## 16. Claim-to-evidence ledger, model SHA-256, reproducibility archive
Only actual solver output is evidence of a model verdict. Explicitly missing are general computational PCS and retired-key reductions, multi-challenge same-root privacy proofs, physical power-loss tests, complete PKI/CRL validation, and the authenticated B0–B3 network slow path.

In [ ]:
z3_safety=[r for r in z3_results if r.get('kind')=='safety']
z3_reach=[r for r in z3_results if r.get('kind')=='reachability']
z3_mutant=[r for r in z3_results if r.get('kind')=='negative-control']
by_pv={r['objective']:r for r in pv_rows}
def verdict(objective):
    r=by_pv.get(objective)
    return ('NOT_EXECUTED' if not r or r['status']!='EXECUTED'
            else '; '.join(v['result'] for v in r['results']))
ledger=[
 ('Table VII','Past RSU→honest target chronological authorization',f"{recovery_counts['honest_completions']}/{recovery_counts['honest_completion_trials']}",'NATIVE_TRIAL' if mh.CR.native else 'DEV_ONLY'),
 ('PV1–PV3','Transcript authentication + payload non-derivability',verdict('PV1_PV2_PV3_explicit_transcript'),'SYMBOLIC_MODEL_ONLY'),
 ('PV4','Single-challenge observational equivalence',verdict('PV4_single_challenge_privacy'),'SINGLE_CHALLENGE_ONLY'),
 ('PV5','Leaked target credential negative control',verdict('PV5_target_credential_leak_mutant'),'INADMISSIBLE_MUTANT'),
 ('PV6–PV7','Two same-root chronological formal sessions',verdict('PV6_PV7_same_root_two_hops'),'SYMBOLIC_MODEL_ONLY'),
 ('PV8','Identity swap / multiple challenge unlinkability',verdict('PV8_identity_swap_UNESTABLISHED')+'; '+verdict('PV8_linkability_UNESTABLISHED'),'UNESTABLISHED'),
 ('Z1–Z7','Bounded single-use and authenticated guards',str([r['result'] for r in z3_safety]) if z3_safety else 'NOT_EXECUTED','ATOMIC_STORAGE_ASSUMPTION'),
 ('Z8','Bounded reachability',str([r['result'] for r in z3_reach]) if z3_reach else 'NOT_EXECUTED','BOUNDED_MODEL'),
 ('Z9','Crash rollback negative control',str([r['result'] for r in z3_mutant]) if z3_mutant else 'NOT_EXECUTED','FAULT_MUTANT'),
 ('Table X','Atomic wrapped-key + report queue restart',f'{persist_checks.passed.sum()}/{len(persist_checks)}','RESEARCH_IMPLEMENTATION'),
 ('Unproved','General computational one-hop PCS indistinguishability','NOT_PROVED','THEOREM_OPEN'),
 ('Unproved','Retired-key security after later vehicle/MSC disclosure','NOT_PROVED','THEOREM_OPEN'),
 ('Not Implemented','Complete authenticated B0–B3 controller reconciliation','NOT_IMPLEMENTED','CONTROLLER_NETWORK_ABSENT'),
]
ledger_df=pd.DataFrame(ledger,columns=['paper_claim','property','actual_result','evidence_scope'])
ledger_df.to_csv(ROOT/'claim_evidence_ledger.csv',index=False)
display(ledger_df)

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
import csv
end=datetime.now(timezone.utc).isoformat()
PROVENANCE.update({'finished_utc':end,'backend_actual':mh.CR.label,'run_profile':RUN_PROFILE,
 'z3_status':z3_status,'proverif_binary':str(pv_executable) if pv_executable else None,
 'proverif_model_hashes':{r['model']:r['sha256'] for r in pv_rows},
 'limitations':['unverified hardware equivalence','incomplete controller B0-B3',
  'single-challenge formal privacy is not multi-challenge proof',
  'full X.509 chain/revocation not implemented','no hardware key deletion evidence',
  'local MSC delivery is not networked controller communication']})
(ROOT/'environment.json').write_text(json.dumps(PROVENANCE,indent=2,default=str))
RUN_README=f"""# MobiHeal experimental run — {RUN_PROFILE}

Backend: **{mh.CR.label}**. Started: {STARTED}, finished: {end}.

Generated directly from the bundled Jupyter notebook; no published numerical values are
used as measured observations. The paper reference is in `paper_reference.csv`.

## Evidence rules
- `DEV_ONLY` outputs are simulation smoke tests. They are **NOT valid for manuscript performance or PQ security claims**.
- ProVerif statuses and SHA-256 come from the exact embedded model files, NOT prefilled manuscript verdicts.
- Z3 assumes atomic storage, authenticated actions and bounded transition count.
- Table X is a research-level persistence extension, with a local idempotent MSC receiver.
- The packet-loss model is independent erasure Monte Carlo, not live radio.
- Computational security reductions, full B0–B3 network protocol, physical erasure,
  certificate chain and revocation validation are not implemented.
"""
(ROOT/'README_RESULTS.md').write_text(RUN_README)
# Audit all exported datasets and models, copy exact invoked module/model source into archive.
source_names=['mobiheal_core.py','mobiheal_experiments.py','recovery_table_experiment.py',
 'mobiheal_persistence.py','mobiheal_ablations.py','mobiheal_z3_current.py',*PV_TASKS]
source_dir=ROOT/'sources';source_dir.mkdir(exist_ok=True)
for name in source_names:
    if Path(name).exists():shutil.copy2(name,source_dir/name)
files=[p for p in ROOT.rglob('*') if p.is_file() and p.name not in ('SHA256_MANIFEST.json',)]
manifest={p.relative_to(ROOT).as_posix():hashlib.sha256(p.read_bytes()).hexdigest() for p in files}
(ROOT/'SHA256_MANIFEST.json').write_text(json.dumps(manifest,indent=2))
zip_path=Path.cwd()/'MobiHeal_Complete_Results.zip'
with ZipFile(zip_path,'w',compression=ZIP_DEFLATED,compresslevel=6,allowZip64=True) as z:
    for f in ROOT.rglob('*'):
        if f.is_file():z.write(f,arcname=f.relative_to(ROOT.parent))
with ZipFile(zip_path) as z:assert z.testzip() is None
print('Evidence archive:',zip_path)
print('Research status: native=' + str(mh.CR.native) + ', Z3=' + z3_status + ', ProVerif available=' + str(bool(pv_executable)))
print('Total reproducibility artifacts:',len(manifest)+1)

---

### Notes for GitHub and IEEE paper use

1. Commit **this one `.ipynb`** to the repository. Running it writes source modules and raw results to `MobiHeal_Complete_Results/`; optionally commit the generated sources and representative native-run CSVs for independent peer review.
2. To match **trial counts**, run `FULL` on a comparable hardware/SQLite setup. To match **cryptographic semantics**, run `NATIVE`. Neither assures identical wall-clock timings; the environment JSON records provenance.
3. For formal evidence, install **ProVerif 2.05** or opt in to its checksum-verified build (`MOBIHEAL_INSTALL_PROVERIF=1` in a root-capable Colab runtime) and install `z3-solver`. Query logs remain intact for independent inspection.
4. Do **not** call PCS adversary win rates or symbolic proof outputs computational post-quantum security reductions; preserve the paper's exposure and freshness restrictions.
5. Treat `table_X_persistence_*` as a separately measured completion-persistence research extension. It does not silently change the original combined-latency benchmark configuration.

**No empirical numbers are fabricated by this notebook. Missing dependencies are recorded, and unexecuted proof models never become successful queries.**